> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# Entrainment Heads → Agent Memory ·  Behavior Authentication Revision  v2 · FULL

 This edition is for what youve done.  layered_v1_full  Result: Keep the same  Llama-3.1-8B base、bf16、TransformerLens 2.17； Old run read-only, new run saves separately. 

** Execute first.  A  Section: Pure  Python  Offline re-scoring, no need  GPU、 There is no need to install in-depth learning dependency. **  Keep strict primary scores; supplementary diagnosis ignores additional  sequence， Do not deduplicate, supply missing answers, or change provenance. CPU  The audit did not produce new model results. 

** Reexecution  00–15  Section: FULL  Development. ** pilot/train/dev/test=24/120/40/120； Default from Old  read-seed11/23/37.json  Read Fixed Sorting, Check  K=1/2/4/8/16， No rereading side 450 Step. There is no natural error in binding.  clean  Small K Diagnosis of injury, no choice.  H_R， No formal opening.  test。Writer  Use the same experience as saved and verified, using the new version of the example / regenerate the input representation. Because the representation changed, old and new results are a development-stage protocol comparison, not an established  heads  Effect. 

FULL  Internal checks automatically stop expensive invalid branches; they are not  QUICK。 Unpassed is not an error report. Only new side  pilot  The ability to train new candidates and the natural error is complete.  Writer  Behavior selection; no  H_R/H_W  therefore does not fabricate 2×2 Outcome.  test  Default close. 

 According to  [τ-bench](https://arxiv.org/abs/2406.12045)、[BFCL](https://proceedings.mlr.press/v267/patil25a.html)、[AgentBoard](https://arxiv.org/abs/2401.13178)、[LongMemEval](https://arxiv.org/abs/2410.10813)、[AgentDojo](https://arxiv.org/abs/2406.13352)  Report final task success, required execution paths, binding, provenance, and protocol quality separately. Project gates and revisions were chosen after development results; do not claim preregistration or replication of these  benchmark。

 This version has been authenticated locally; not yet running modified 8B GPU Experiments do not pre-empt the effectiveness of interventions. 

## A ·  Old result offline re-scoring (no need)  GPU）

 Check the next two paths first. By default, youre the one.  FULL  run directory; keep new and old directories separate. Run the three code cells in this section to generate audit files; stop here if only inspecting historical results. After the first installation and session restart, rerun the path cell, then continue from dependency validation. 

In [ ]:
from pathlib import Path
import json
PREVIOUS_RUN_DIR = Path('/content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_layered_v1_full')
NEW_RUN_DIR = Path('/content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_behavioral_v2_full')
try:
    from google.colab import drive
except ImportError:
    print(' For local execution, set the two paths to the historical result directory and a new output directory. ')
else:
    drive.mount('/content/drive')
if not PREVIOUS_RUN_DIR.is_dir():
    raise FileNotFoundError(' Historical run directory not found; update  PREVIOUS_RUN_DIR：' + str(PREVIOUS_RUN_DIR))
print(' Read-only old results: ', PREVIOUS_RUN_DIR)
print(' New result saving: ', NEW_RUN_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Read-only old results:  /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_layered_v1_full
 New result saving:  /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_behavioral_v2_full


In [ ]:
_OFFLINE_AUDITOR_SOURCE_SHA256 = '531406c2cb4b6446accb5e07fc282385462fbdf8b51939e65042ce1667168c62'
"""Read-only, standard-library audit of saved pilot/dev generations.

This module imports no experiment code and never executes a model or a tool call.
Sequence-only normalization is a diagnostic, not a replacement for old scores.
Run: python offline_audit.py PREVIOUS_RUN_DIR NEW_AUDIT_DIR
"""

from collections import Counter, defaultdict
from copy import deepcopy
from datetime import datetime, timezone
import argparse
import csv
import hashlib
import json
from pathlib import Path
import re


VERSION = "offline_behavioral_audit_v1"
FIELDS = {"kind", "entity", "attribute", "value", "source_event"}
INPUTS = (
    ("results/pilot-writer-raw.json", "writer", "pilot"),
    ("results/dev-writer-raw.json", "writer", "dev"),
    ("results/dev-read-controlled_decision-raw.json", "actor", "dev"),
)


def _canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def _hash(value):
    return hashlib.sha256(_canonical(value).encode("utf-8")).hexdigest()


def _no_duplicate_keys(pairs):
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValueError("duplicate JSON key: " + key)
        value[key] = item
    return value


def _no_constant(token):
    raise ValueError("nonfinite JSON: " + token)


def _parse(raw):
    """Match the old parser's fence tolerance; do not extract or repair JSON."""
    if not isinstance(raw, str):
        return None, "output_not_text"
    candidate = raw.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate,
                             flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        return json.loads(candidate, object_pairs_hook=_no_duplicate_keys,
                          parse_constant=_no_constant), None
    except (ValueError, TypeError) as exc:
        return None, "invalid_json:" + str(exc)


def _schema(memory):
    issues = []
    if not isinstance(memory, list):
        return [{"type": "wrong_root_type", "expected": "list"}]
    for index, record in enumerate(memory):
        if not isinstance(record, dict):
            issues.append({"record": index, "type": "non_object_record"})
            continue
        missing, extra = FIELDS - set(record), set(record) - FIELDS
        if missing:
            issues.append({"record": index, "type": "missing_fields",
                           "fields": sorted(missing)})
        if extra:
            issues.append({"record": index, "type": "schema_extras",
                           "fields": sorted(extra)})
        if record.get("kind") != "preference":
            issues.append({"record": index, "type": "non_preference_record"})
        if record.get("attribute") != "delivery_city":
            issues.append({"record": index, "type": "wrong_attribute"})
        for key in ("entity", "attribute", "value", "source_event"):
            if key in record and (not isinstance(record[key], str) or not record[key]):
                issues.append({"record": index, "type": "invalid_field_value",
                               "field": key})
    return issues


def _duplicates(memory):
    """Count same entity/attribute records; never remove one or choose a value."""
    groups = defaultdict(list)
    if isinstance(memory, list):
        for index, record in enumerate(memory):
            if (isinstance(record, dict) and record.get("kind") == "preference"
                    and isinstance(record.get("entity"), str)
                    and isinstance(record.get("attribute"), str)):
                groups[(record["entity"], record["attribute"])].append(index)
    return [{"entity": key[0], "attribute": key[1], "record_indices": indices}
            for key, indices in groups.items() if len(indices) > 1]


def _sequence_only(memory):
    normalized = deepcopy(memory)
    removed = []
    if isinstance(normalized, list):
        for index, record in enumerate(normalized):
            if isinstance(record, dict) and "sequence" in record:
                removed.append({"record": index, "value": record.pop("sequence")})
    return normalized, removed


def _is_development(row, expected=None):
    split = row.get("split")
    source = row.get("source_id", "")
    if not isinstance(source, str) or source.startswith("test-") or split == "test":
        return False
    if split not in ("pilot", "dev"):
        return False
    return expected is None or split == expected


class _Reader:
    def __init__(self, root):
        self.root = root
        self.files = {}
        self.warnings = []

    def read(self, relative, role):
        path = (self.root / relative).resolve()
        if not path.is_relative_to(self.root):
            raise ValueError("Input symlink leaves the previous run: " + relative)
        content = path.read_bytes()
        self.files[relative] = {"path": relative, "role": role,
                                "bytes": len(content),
                                "sha256": hashlib.sha256(content).hexdigest()}
        return json.loads(content)

    def unchanged(self):
        for relative, info in self.files.items():
            digest = hashlib.sha256((self.root / relative).read_bytes()).hexdigest()
            if digest != info["sha256"]:
                raise RuntimeError("Source changed during audit: " + relative)


def _rebuild_history(actor):
    """Reproduce frozen_history from SAVED autonomous memory and executed events."""
    if actor.get("evaluation_layer") == "controlled_decision" or actor.get("setup_origin"):
        raise ValueError("Controlled setup cannot supply a Writer experience")
    if not isinstance(actor.get("memory"), list) or not isinstance(actor.get("events"), list):
        raise ValueError("Saved Actor lacks memory or events")
    history = []
    for record in actor["memory"]:
        if not isinstance(record, dict):
            raise ValueError("Saved memory record is not an object")
        event = deepcopy(record)
        event["sequence"] = len(history)
        history.append(event)
    for event in actor["events"]:
        step = event["step"]
        if event.get("action") is not None:
            history.append({"kind": "observed_action", "sequence": len(history),
                            "source_event": f"{actor['source_id']}:action:{step}",
                            "action": deepcopy(event["action"])})
        else:
            history.append({"kind": "observed_output", "sequence": len(history),
                            "source_event": f"{actor['source_id']}:output:{step}",
                            "text": event.get("raw_output", ""),
                            "parse_error": event.get("parse_error")})
        if event.get("observation") is not None:
            history.append({"kind": "tool_observation", "sequence": len(history),
                            "source_event": f"{actor['source_id']}:observation:{step}",
                            "observation": deepcopy(event["observation"])})
    return history


def _evidence_index(reader, writer_rows):
    cases, histories = {}, defaultdict(list)
    relevant = {(row["source_id"], row.get("condition")) for row in writer_rows}
    manifest = reader.root / "data_manifest.json"
    if manifest.is_file():
        data = reader.read("data_manifest.json", "case_evidence").get("data", {})
        # The manifest also contains sealed test INPUTS; never use that split.
        for split in ("pilot", "dev"):
            for case in data.get(split, []):
                if _is_development(case, split):
                    cases[case["source_id"]] = (case, "data_manifest.json")

    def add_actor(actor, path):
        key = (actor.get("source_id"), actor.get("condition"))
        if not _is_development(actor) or key not in relevant:
            return
        try:
            histories[key].append((_rebuild_history(actor), path))
        except (KeyError, TypeError, ValueError) as exc:
            reader.warnings.append({"path": path, "reason": str(exc)})

    directory = reader.root / "items" / "fixed_history_actors"
    if directory.is_dir():
        for path in sorted(directory.glob("*.json")):
            # Do not open test checkpoints or unrelated training data.
            if not any(path.name.startswith(source + "-") for source, _ in relevant):
                continue
            relative = path.relative_to(reader.root).as_posix()
            payload = reader.read(relative, "saved_autonomous_actor_evidence")
            actor = payload.get("result", payload)
            if isinstance(actor, dict):
                add_actor(actor, relative)
    for split in ("pilot", "dev"):
        relative = f"results/{split}-read-end_to_end-raw.json"
        if (reader.root / relative).is_file():
            for actor in reader.read(relative, "saved_autonomous_actor_evidence"):
                if isinstance(actor, dict) and actor.get("intervention") == "none":
                    add_actor(actor, relative)
    return cases, histories


def _verified_evidence(row, cases, histories, source_file):
    case_info = cases.get(row["source_id"])
    if isinstance(row.get("case"), dict):
        case_info = (row["case"], source_file + "#embedded_case")
    if case_info is None:
        return None, "case_unavailable", []
    case, case_path = case_info
    if (not _is_development(case, row["split"])
            or case.get("source_id") != row["source_id"]
            or not isinstance(case.get("target_name"), str)):
        return None, "case_identity_unverified", [case_path]
    result = row.get("writer_result", {})
    expected = row.get("history_hash") or result.get("history_hash")
    if not expected:
        return None, "history_hash_unavailable", [case_path]
    if (row.get("history_hash") and result.get("history_hash")
            and row["history_hash"] != result["history_hash"]):
        return None, "conflicting_history_hashes", [case_path]
    candidates = list(histories.get((row["source_id"], row.get("condition")), []))
    if isinstance(row.get("history"), list):
        candidates.append((row["history"], source_file + "#embedded_history"))
    if not candidates:
        return None, "actual_history_unavailable", [case_path]
    for history, path in candidates:
        if _hash(history) == expected:
            return (case, history), None, [case_path, path]
    return None, "actual_history_hash_mismatch", [case_path] + [p for _, p in candidates]


def _semantic_scores(memory, schema_valid, evidence, unknown_reason):
    """Score only against verified evidence; invalid outputs are not fabricated omissions."""
    if evidence is None:
        return {"status": "unknown", "reason": unknown_reason, "metrics": None}
    if not schema_valid:
        return {"status": "unscorable", "reason": "schema_or_parse_failure", "metrics": None}
    case, history = evidence
    ordered = []
    for index, event in enumerate(history):
        if not isinstance(event, dict):
            return {"status": "unknown", "reason": "malformed_history", "metrics": None}
        if event.get("kind") != "preference":
            continue
        if any(not isinstance(event.get(k), str) for k in FIELDS):
            return {"status": "unknown", "reason": "malformed_preference_evidence", "metrics": None}
        sequence = event.get("sequence", index)
        if not isinstance(sequence, (int, float)):
            return {"status": "unknown", "reason": "invalid_evidence_sequence", "metrics": None}
        ordered.append((sequence, index, event))
    reference = {}
    for _, _, event in sorted(ordered, key=lambda item: (item[0], item[1])):
        reference[(event["entity"], event["attribute"])] = event
    target_key = (case["target_name"], "delivery_city")
    if target_key not in reference:
        return {"status": "unknown", "reason": "target_preference_evidence_unavailable", "metrics": None}
    groups = defaultdict(list)
    for record in memory:
        groups[(record["entity"], record["attribute"])].append(record)
    target = groups.get(target_key, [])
    metrics = {
        "target_correct": len(target) == 1 and target[0]["value"] == reference[target_key]["value"],
        "target_omission": not target,
        "omission": any(key not in groups for key in reference),
        "missing_count": sum(key not in groups for key in reference),
        "duplicate_count": sum(max(0, len(records) - 1) for records in groups.values()),
        "unsupported_count": sum(key not in reference or rec["value"] != reference[key]["value"]
                                 for key, records in groups.items() for rec in records),
        "source_mismatch_count": sum(key in reference and rec["source_event"] != reference[key]["source_event"]
                                     for key, records in groups.items() for rec in records),
        "required_count": len(reference), "record_count": len(memory),
    }
    return {"status": "verified", "reason": None, "metrics": metrics}


def _writer_audit(row, source_file, row_index, cases, histories):
    result = row.get("writer_result", {})
    raw = result.get("raw")
    memory, parse_error = _parse(raw)
    issues = _schema(memory) if parse_error is None else []
    duplicates = _duplicates(memory)
    normalized, removed = _sequence_only(memory)
    normalized_issues = _schema(normalized) if parse_error is None else []
    stop = result.get("stop_reason")
    flags = {
        "json_syntax": parse_error is not None,
        "schema_extras": any(i["type"] == "schema_extras" for i in issues),
        "missing_fields": any(i["type"] == "missing_fields" for i in issues),
        "metadata": isinstance(memory, list) and any(isinstance(r, dict) and r.get("kind") == "metadata" for r in memory),
        "duplicate": bool(duplicates),
        "truncated": stop in ("max_new_tokens", "length", "max_tokens"),
    }
    strict_valid = parse_error is None and not issues
    normalized_valid = parse_error is None and not normalized_issues
    if strict_valid:
        failure = "none"
    elif flags["truncated"] and flags["json_syntax"]:
        failure = "truncated"
    else:
        failure = next((name for name in ("json_syntax", "missing_fields", "schema_extras", "metadata")
                        if flags[name]), "other_schema_failure")
    evidence, unknown, evidence_paths = _verified_evidence(row, cases, histories, source_file)
    return {
        "source_file": source_file, "source_row_index": row_index,
        "source_id": row["source_id"], "split": row["split"],
        "condition": row.get("condition"), "intervention": row.get("intervention"),
        "original_record": deepcopy(row),
        "raw_sha256": hashlib.sha256((raw if isinstance(raw, str) else _canonical(raw)).encode()).hexdigest(),
        "stop_reason": stop, "flags": flags, "primary_failure_class": failure,
        "strict_parse_error": parse_error, "strict_schema_issues": issues,
        "strict_schema_valid_recomputed": strict_valid,
        "strict_valid_matches_saved": strict_valid == result.get("valid", row.get("valid")),
        "duplicate_groups": duplicates,
        "sequence_only_normalization": {
            "removed_fields": removed, "memory": normalized if parse_error is None else None,
            "schema_valid": normalized_valid, "schema_issues": normalized_issues,
            "operation": "delete_top_level_record_sequence_only",
            "semantic_scores": _semantic_scores(normalized, normalized_valid, evidence, unknown),
        },
        "strict_semantic_scores": _semantic_scores(memory, strict_valid, evidence, unknown),
        "evidence": {"status": "verified" if evidence else "unknown",
                     "reason": unknown, "paths": evidence_paths,
                     "history_hash": _hash(evidence[1]) if evidence else None},
    }


def _json_candidates(raw):
    """Nonoverlapping JSON containers in text; diagnostic spans ONLY, never actions."""
    if not isinstance(raw, str):
        return []
    decoder = json.JSONDecoder(object_pairs_hook=_no_duplicate_keys, parse_constant=_no_constant)
    spans, index = [], 0
    while index < len(raw):
        if raw[index] not in "[{":
            index += 1
            continue
        try:
            value, end = decoder.raw_decode(raw, index)
        except (ValueError, TypeError):
            index += 1
            continue
        spans.append({"start": index, "end": end,
                      "type": "object" if isinstance(value, dict) else "array"})
        index = end
    return spans


def _actor_audit(row, source_file, row_index):
    diagnostics = []
    for index, event in enumerate(row.get("events", [])):
        raw = event.get("raw_output")
        parsed, error = _parse(raw)
        candidates = _json_candidates(raw)
        diagnostics.append({
            "event_index": index, "step": event.get("step"),
            "stop_reason": event.get("stop_reason"), "saved_parse_error": event.get("parse_error"),
            "strict_whole_output_is_object": error is None and isinstance(parsed, dict),
            "strict_parse_error": error, "complete_json_candidate_count": len(candidates),
            "complete_json_object_candidate_count": sum(s["type"] == "object" for s in candidates),
            "candidate_spans": candidates,
            "interpretation": "text_diagnostic_only_no_tool_execution_or_success_rescoring",
        })
    return {"source_file": source_file, "source_row_index": row_index,
            "source_id": row["source_id"], "split": row["split"],
            "condition": row.get("condition"), "intervention": row.get("intervention"),
            "original_record": deepcopy(row), "event_diagnostics": diagnostics,
            "execution_rescored": False}


def _write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def _write_csv(path, rows):
    columns = list(rows[0]) if rows else ["source_file", "source_row_index", "source_id"]
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: _canonical(value) if isinstance(value, (dict, list)) else value
                             for key, value in row.items()})


def _group_summary(writer_rows, actor_rows):
    groups = {}
    for row in writer_rows:
        key = (row["split"], row["condition"], row["intervention"])
        group = groups.setdefault(key, {"split": key[0], "condition": key[1], "intervention": key[2],
                                       "n_rows": 0, "saved_valid_count": 0,
                                       "primary_failure_classes": Counter(), "overlapping_flags": Counter(),
                                       "normalized_schema_valid_count": 0, "semantic_status": Counter()})
        group["n_rows"] += 1
        group["saved_valid_count"] += bool(row["original_record"].get("valid"))
        group["primary_failure_classes"][row["primary_failure_class"]] += 1
        group["overlapping_flags"].update(name for name, flag in row["flags"].items() if flag)
        group["normalized_schema_valid_count"] += row["sequence_only_normalization"]["schema_valid"]
        group["semantic_status"][row["sequence_only_normalization"]["semantic_scores"]["status"]] += 1
    actors = {}
    for row in actor_rows:
        key = (row["split"], row["condition"], row["intervention"])
        group = actors.setdefault(key, {"split": key[0], "condition": key[1], "intervention": key[2],
                                       "n_rows": 0, "saved_format_failure_count": 0,
                                       "stop_reasons": Counter(), "candidate_counts": Counter()})
        group["n_rows"] += 1
        group["saved_format_failure_count"] += bool(row["original_record"].get("format_failure"))
        for event in row["event_diagnostics"]:
            group["stop_reasons"][str(event["stop_reason"])] += 1
            group["candidate_counts"][str(event["complete_json_candidate_count"])] += 1
    return list(groups.values()), list(actors.values())


def _writer_metric_summary(rows):
    groups = defaultdict(list)
    for row in rows:
        groups[(row["split"], row["condition"], row["intervention"])].append(row)
    result = []
    for (split, condition, arm), members in sorted(groups.items()):
        statuses = Counter(r["sequence_only_normalization"]["semantic_scores"]["status"] for r in members)
        scores = [r["sequence_only_normalization"]["semantic_scores"]["metrics"] for r in members
                  if r["sequence_only_normalization"]["semantic_scores"]["status"] == "verified"]
        saved = [r["original_record"].get("target_correct") for r in members]
        saved_known = [v for v in saved if isinstance(v, bool)]
        correct = sum(bool(m["target_correct"]) for m in scores)
        complete = sum(bool(m["target_correct"]) and all(m[k] == 0 for k in
                       ("missing_count", "duplicate_count", "unsupported_count", "source_mismatch_count")) for m in scores)
        result.append({"split": split, "condition": condition, "intervention": arm,
                       "n_all": len(members), "n_semantic_verified": statuses["verified"],
                       "n_semantic_unknown": statuses["unknown"], "n_semantic_unscorable": statuses["unscorable"],
                       "n_saved_target_scores": len(saved_known),
                       "saved_target_correct_fraction": sum(saved_known) / len(saved_known) if saved_known else None,
                       "normalized_schema_valid_count": sum(r["sequence_only_normalization"]["schema_valid"] for r in members),
                       "normalized_verified_target_correct_count": correct,
                       "normalized_verified_correct_valid_fraction_all_rows": correct / len(members),
                       "normalized_target_accuracy_among_scorable": correct / len(scores) if scores else None,
                       "normalized_verified_complete_memory_correct_count": complete,
                       "normalized_verified_complete_memory_fraction_all_rows": complete / len(members),
                       "interpretation": "auxiliary_only; all_rows_fraction_counts_verified_successes; unknown_is_not_a_semantic_error"})
    return result


def _auditor_source_hash():
    embedded = globals().get("_OFFLINE_AUDITOR_SOURCE_SHA256")
    if embedded is not None:
        if not isinstance(embedded, str) or not re.fullmatch(r"[0-9a-f]{64}", embedded):
            raise ValueError("Invalid embedded audit source hash")
        return embedded
    source = globals().get("__file__")
    if source is None:
        raise RuntimeError("Embedded audit source hash is missing; rerun the complete offline definitions cell")
    return hashlib.sha256(Path(source).read_bytes()).hexdigest()


def _resume_audit(previous, output):
    summary_path = output / "summary.json"
    if not summary_path.is_file():
        raise ValueError("Use a new or empty audit output directory")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if (summary.get("previous_run_dir") != str(previous) or summary.get("audit_version") != VERSION
            or summary.get("provenance", {}).get("auditor_sha256") != _auditor_source_hash()):
        raise ValueError("Audit source or previous directory changed; use a new audit output directory")
    for item in summary["provenance"]["inputs"]:
        path = previous / item["path"]
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Previous audit input changed: " + str(path))
    if any((previous / path).exists() for path in summary.get("missing_inputs", [])):
        raise ValueError("Previously missing inputs are now present; use a new audit output directory")
    for name, expected in summary.get("artifact_sha256", {}).items():
        path = output / name
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError("Audit artifact changed or is incomplete: " + str(path))
    if not summary.get("artifact_sha256"):
        raise ValueError("Audit lacks artifact integrity metadata; use a new output directory")
    return summary


def audit_previous_run(previous_run_dir, output_dir):
    """Audit saved pilot/dev files into a NEW, disjoint directory; return summary.

    Missing inputs are reported. Test results/checkpoints are never selected.
    Original records and scores are retained unchanged in both JSON and CSV.
    Only actual, hash-matching Writer histories permit new semantic scoring.
    """
    previous = Path(previous_run_dir).expanduser().resolve()
    output = Path(output_dir).expanduser().resolve()
    if not previous.is_dir():
        raise ValueError("previous_run_dir must be an existing directory")
    if output.is_relative_to(previous) or previous.is_relative_to(output):
        raise ValueError("The audit output and previous run must be disjoint directories")
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        return _resume_audit(previous, output)
    reader = _Reader(previous)
    loaded, missing, skipped = [], [], []
    for relative, kind, split in INPUTS:
        if not (previous / relative).is_file():
            missing.append(relative)
            continue
        rows = reader.read(relative, "saved_" + kind + "_results")
        if not isinstance(rows, list):
            raise ValueError(relative + " must contain a JSON list of rows")
        for index, row in enumerate(rows):
            if not isinstance(row, dict) or not _is_development(row, split):
                skipped.append({"path": relative, "row_index": index,
                                "reason": "not_expected_pilot_or_dev_split"})
                continue
            loaded.append((kind, relative, index, row))
    cases, histories = _evidence_index(reader, [row for kind, _, _, row in loaded if kind == "writer"])
    writer_rows, actor_rows = [], []
    for kind, relative, index, row in loaded:
        if kind == "writer":
            writer_rows.append(_writer_audit(row, relative, index, cases, histories))
        else:
            actor_rows.append(_actor_audit(row, relative, index))
    reader.unchanged()
    writer_groups, actor_groups = _group_summary(writer_rows, actor_rows)
    summary = {
        "audit_version": VERSION, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "previous_run_dir": str(previous), "output_dir": str(output),
        "writer_rows": len(writer_rows), "actor_rows": len(actor_rows),
        "missing_inputs": missing, "skipped_rows": skipped, "warnings": reader.warnings,
        "writer_groups": writer_groups, "actor_groups": actor_groups,
        "writer_metric_summary": _writer_metric_summary(writer_rows),
        "provenance": {"inputs": list(reader.files.values()),
                       "auditor_sha256": _auditor_source_hash(),
                       "input_files_verified_unchanged": True,
                       "allowed_splits": ["pilot", "dev"], "test_results_used": False},
        "interpretation": [
            "Saved strict scores remain unchanged; diagnostics are not replacement outcomes.",
            "Writer flags overlap; primary_failure_classes are mutually exclusive.",
            "Normalization only removes record-level sequence; duplicates and factual values remain.",
            "Unknown semantics are not counted as correct or incorrect.",
            "Actor JSON candidates are text diagnostics, never executed actions or recovered successes.",
        ],
        "artifacts": {"writer_json": "writer-audit.json", "writer_csv": "writer-audit.csv",
                      "actor_json": "actor-audit.json", "actor_csv": "actor-audit.csv",
                      "writer_metric_summary": "writer-metric-summary.csv",
                      "summary": "summary.json"},
    }
    output.mkdir(parents=True, exist_ok=True)
    for stem, rows in (("writer-audit", writer_rows), ("actor-audit", actor_rows)):
        _write_json(output / (stem + ".json"), rows)
        _write_csv(output / (stem + ".csv"), rows)
    _write_csv(output / "writer-metric-summary.csv", summary["writer_metric_summary"])
    summary["artifact_sha256"] = {name: hashlib.sha256((output / name).read_bytes()).hexdigest()
                                  for name in summary["artifacts"].values() if name != "summary.json"}
    _write_json(output / "summary.json", summary)
    return summary

In [ ]:
OFFLINE_SUMMARY = audit_previous_run(PREVIOUS_RUN_DIR, NEW_RUN_DIR / 'offline_audit')
print('Writer Number of articles :', OFFLINE_SUMMARY['writer_rows'], 'Actor Number of articles :', OFFLINE_SUMMARY['actor_rows'])
print(' Missing input :', OFFLINE_SUMMARY['missing_inputs'])
for row in OFFLINE_SUMMARY['writer_metric_summary']:
    print(row['split'], row['condition'], row['intervention'],
          ' Total =', row['n_all'], ' Auxiliary is legitimate and the target is correct. =', row['normalized_verified_target_correct_count'],
          ' Auxiliary to the whole fact / The source is correct. =', row['normalized_verified_complete_memory_correct_count'],
          ' Non-scoring =', row['n_semantic_unscorable'], ' The evidence is unknown. =', row['n_semantic_unknown'])
print(' Offline re-scoring has been saved; this step is not loaded. ')

Writer Number of articles : 528 Actor Number of articles : 1680
 Missing input : []
dev correct_only H_R_cross_role  Total = 40  Auxiliary is legitimate and the target is correct. = 13  Auxiliary to the whole fact / The source is correct. = 13  Non-scoring = 27  The evidence is unknown. = 0
dev correct_only none  Total = 40  Auxiliary is legitimate and the target is correct. = 17  Auxiliary to the whole fact / The source is correct. = 16  Non-scoring = 20  The evidence is unknown. = 0
dev distractor H_R_cross_role  Total = 40  Auxiliary is legitimate and the target is correct. = 6  Auxiliary to the whole fact / The source is correct. = 5  Non-scoring = 23  The evidence is unknown. = 0
dev distractor none  Total = 40  Auxiliary is legitimate and the target is correct. = 40  Auxiliary to the whole fact / The source is correct. = 40  Non-scoring = 0  The evidence is unknown. = 0
dev distractor_value_swap H_R_cross_role  Total = 40  Auxiliary is legitimate and the target is correct. = 6  A

## 00 ·  Dependence 

 Keep the old book all the majors  imports。 To avoid the old version,  TransformerLens  Disrupted by new dependence, fixed.  `transformers`、`datasets`、`pyarrow`  e.g. compatible combinations. Python 3.13  Use  NumPy 2.1.3；Python 3.10–3.12  Use  NumPy 1.26.4。 Install with the first code cell once. After success, restart the runtime and rerun  A  , and then continue from the dependent authentication block. Torch  Reuse an existing runtime satisfying  `>=2.6`   CUDA  Version. 

In [ ]:
#  First use: Run this block and restart after installation  Colab  Run when. 
import sys
import subprocess

python_version = sys.version_info[:2]
print(f"Python: {sys.version.split()[0]}\n Interpreter : {sys.executable}", flush=True)
if not ((3, 10) <= python_version <= (3, 13)):
    raise RuntimeError(
        f" current  Python {sys.version.split()[0]}  Not applicable to Ben  notebook  A fixed dependency combination. "
        " Please switch to  Python 3.10–3.13   Colab  Try again when running. "
    )

numpy_version = "2.1.3" if python_version == (3, 13) else "1.26.4"
print(f" Select when running this one  NumPy {numpy_version}", flush=True)

packages = [
    "transformer_lens==2.17.0",
    "transformers==4.57.1",
    "huggingface_hub==0.36.0",
    f"numpy=={numpy_version}",
    "pandas==2.2.3",
    "plotly==6.1.2",
    "scipy==1.15.3",
    "scikit-learn==1.6.1",
    "datasets==3.6.0",
    "pyarrow==20.0.0",
    "tqdm",
]
print(" Start installation of dependency; maintain a full log to check. ", flush=True)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        " Reliance on installation failed. Please look above  pip  First in the log  ERROR，"
        " Reruns the block after repair; for the time being, do not run the subsequent block. "
    ) from exc

print("\n Install command successfully executed. Please be on  Colab  . When running again in the menu, "
      " Reexecut A path cell in this section, then continue from dependency validation without reinstalling. ", flush=True)

Python: 3.13.16
 Interpreter : /usr/bin/python3
 Select when running this one  NumPy 2.1.3
 Start installation of dependency; maintain a full log to check. 

 Install command successfully executed. Please be on  Colab  Restart running in menu, reexecut A path cell in this section, then continue from dependency validation without reinstalling. 


In [ ]:
#  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
import sys
import importlib.metadata as metadata
import importlib.util

print("Python:", sys.version.split()[0])
print(" Interpreter :", sys.executable)
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

expected_versions = {
    "transformer-lens": "2.17.0", "transformers": "4.57.1",
    "huggingface-hub": "0.36.0",
    "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
    "pandas": "2.2.3",
    "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
    "datasets": "3.6.0", "pyarrow": "20.0.0",
}
problems = []
for package, expected in expected_versions.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
    if installed != expected:
        problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

for package in ("torch", "tqdm"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        problems.append(f"{package}:  not installed ")

try:
    torch_version = metadata.version("torch")
    torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
    if torch_major_minor < (2, 6):
        problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
except metadata.PackageNotFoundError:
    pass  #  The record is missing. 
except ValueError:
    problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

if importlib.util.find_spec("transformer_lens") is None:
    problems.append(" current  Python  Cannot find  transformer_lens  module ")
if problems:
    raise RuntimeError(
        " Reliance check failed: \n- " + "\n- ".join(problems)
        + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
          " Go on from this block. "
    )
print(" Reliance on a version check pass; continues to run the third code block. ")

Python: 3.13.16
 Interpreter : /usr/bin/python3
transformer-lens: 2.17.0 ( Yes.  2.17.0)
transformers: 4.57.1 ( Yes.  4.57.1)
huggingface-hub: 0.36.0 ( Yes.  0.36.0)
numpy: 2.1.3 ( Yes.  2.1.3)
pandas: 2.2.3 ( Yes.  2.2.3)
plotly: 6.1.2 ( Yes.  6.1.2)
scipy: 1.15.3 ( Yes.  1.15.3)
scikit-learn: 1.6.1 ( Yes.  1.6.1)
datasets: 3.6.0 ( Yes.  3.6.0)
pyarrow: 20.0.0 ( Yes.  20.0.0)
torch: 2.11.0+cu130
tqdm: 4.67.3
 Reliance on a version check pass; continues to run the third code block. 


In [ ]:
#  Formal Import: Keep the whole package if there is still an error report here  traceback  In the past few years, many of the worlds leading players have been able to identify specific conflicts. 
import gc, json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, utils
from huggingface_hub import notebook_login, get_token, model_info
import importlib.metadata as metadata
from IPython.display import display

PACKAGE_NAMES = ["torch", "transformer-lens", "transformers", "huggingface-hub", "numpy", "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly"]
PACKAGE_VERSIONS = {name: metadata.version(name) for name in PACKAGE_NAMES}
print(PACKAGE_VERSIONS)
print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print(" Reliance on import success. ")

{'torch': '2.11.0+cu130', 'transformer-lens': '2.17.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0', 'numpy': '2.1.3', 'pandas': '2.2.3', 'datasets': '3.6.0', 'pyarrow': '20.0.0', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'plotly': '6.1.2'}
Python: 3.13.16
CUDA available: True
 Reliance on import success. 


## 01 · FULL  Configure with Independent Directory 

 Path by  A  Section. Reservations  FULL  Samples and fixes  seed=11； Read sidebacks in old sorting. K=0 Baseline with each small K Use the same batch  source。 The default behavior-preservation rule permits neither reduced clean-task success nor increased format failures. This is a development rule, not a statistical-significance guarantee. 

 Old  Actor  Output is only model version, dependent /Python、 Reuse only after checking task, generation-parameter, and code compatibility; otherwise recompute  Actor baseline， The old frozen experience can still be used.  Writer  . Change the configuration / It has to be changed when it is realized.  NEW_RUN_DIR。

In [ ]:
QUICK_MODE = False
MODEL_NAME = "meta-llama/Llama-3.1-8B"  #  Same as old book: base， No, it is not.  Instruct
MODEL_REVISION = None  #  first parsing and locking  commit SHA； Continue reading records in the same directory 
REQUESTED_DTYPE = "bfloat16"  #  Same as the old book; no support is given without a clear statement of error and no secret cutting of precision 
MOUNT_DRIVE = True
RUN_NAME = NEW_RUN_DIR.name
RUN_DEVELOPMENT = True  # FULL  development; no manual release of each stage of development is required 
RUN_ONLINE = False             #  Additional expenses; exploratory analysis independent of  2×2
FREEZE_AND_RUN_TEST = False    # FULL  Freezing protocol only works when the development is complete 

CFG = {
    "model_name": MODEL_NAME, "dtype": REQUESTED_DTYPE, "quick_mode": QUICK_MODE,
    "actor_prompt_version": "complete_demos_v4", "data_design": "crossed_2x2x2_v4",
    "evaluation_version": "behavioral_v2_full",
    "writer_prompt_version": "behavioral_v2_sequence_free_examples_v1",
    "selection_version": "all_source_complete_behavior_v1",
    "compatible_previous_implementation": "c8ecd333da8c71c6c9cb58c68caf6ef0abcccd901d463519b1843c400eec266e",
    "min_natural_read_errors": 2, "compute_snapshot_margin": False, "show_progress": True,
    "evaluation_references": ["arxiv:2406.12045", "pmlr:v267/patil25a", "arxiv:2401.13178",
                              "arxiv:2410.10813", "arxiv:2406.13352"],
    "min_controlled_success": 0.80, "min_writer_clean_correct": 0.80,
    "data_seed": 1729, "n_pilot": 8 if QUICK_MODE else 24,
    "n_train": 16 if QUICK_MODE else 120, "n_dev": 8 if QUICK_MODE else 40,
    "n_test": 24 if QUICK_MODE else 120,
    "search_seeds": [11] if QUICK_MODE else [11, 23, 37],
    "random_seeds": [101] if QUICK_MODE else [101, 202, 303],
    "search_steps": 40 if QUICK_MODE else 450,
    "batch_size": 1, "lr": 0.08, "sparsity": 0.025,
    "temp_start": 1.0, "temp_end": 0.25, "clean_weight": 0.1,
    "top_k": [1, 2, 4, 8, 16],
    "max_clean_margin_drop": 0.25, "save_every": 5,
    "hook_kind": "z", "use_cache": True, "context_window": 4096, "max_prompt_tokens": 3072,
    "actor_max_new_tokens": 128, "writer_max_new_tokens": 384,
    "max_actions": 6, "followup_tasks": 3, "online_horizons": [1, 3, 5],
    "min_pilot_success": 0.80, "max_pilot_format_failure": 0.20,
    "min_natural_writer_errors": 2, "min_writer_valid": 0.80,
    "bootstrap_reps": 2000,
    "city_candidates": ["Paris", "Tokyo", "Rome", "London", "Berlin", "Boston", "Chicago", "Madrid", "Moscow", "Seattle", "Toronto", "Vienna", "Dublin", "Sydney", "Austin", "Denver"],
}
if QUICK_MODE:
    raise RuntimeError(" This revision is fixed  FULL； Do not treat historical  FULL  Data Mix  QUICK。")
RUN_DIR = NEW_RUN_DIR
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("New outputs:", RUN_DIR)
print("FULL sizes:", {k:v for k,v in CFG.items() if k.startswith("n_")})

New outputs: /content/drive/MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_behavioral_v2_full
FULL sizes: {'n_pilot': 24, 'n_train': 120, 'n_dev': 40, 'n_test': 120}


## 02 ·  Core achievement 

 Next definition.  cells  You can fold. Default is in  `hook_z`  Up to zero.  query-head  Output, Reduce Old Books  `hook_result`  Middle billing; back.  smoke  Checks the consistency of the two interfaces. Search only updates  mask， Model parameters are always frozen. 

 All  forward  Im using a single line.  padding  Enter, Candidature  token  Must be satisfied.  `encode(prefix + value) == encode(prefix) + [one_token]`。 Free Generate Every New  KV cache， Intervention coverage  prefill  And every step.  decode。

In [ ]:
# Core infrastructure: exact-prefix token probes, full generation, and search.
import contextlib
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import time
import warnings
import numpy as np
import torch
import torch.nn.functional as F


def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def checkpoint_item(stage, key, payload, fn):
    """Only reuse an item produced by exactly this code/model/config/payload."""
    fingerprint = stable_hash({"implementation": IMPLEMENTATION_HASH,
                               "run": RUN_FINGERPRINT, "payload": payload})
    safe_key = re.sub(r"[^A-Za-z0-9_.-]", "_", str(key))[:150]
    path = RUN_DIR / "items" / stage / (safe_key + ".json")
    if path.exists():
        record = json.loads(path.read_text())
        if record["fingerprint"] != fingerprint:
            raise RuntimeError(f"Stale checkpoint: {path}. Use a new RUN_NAME; do not mix experiments.")
        return record["result"]
    result = fn()
    atomic_json(path, {"fingerprint": fingerprint, "payload_hash": stable_hash(payload), "result": result})
    return result


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_heads(heads, n_layers, n_heads):
    result = sorted(set((int(layer), int(head)) for layer, head in heads))
    if any(not (0 <= l < n_layers and 0 <= h < n_heads) for l, h in result):
        raise ValueError("Head coordinates out of bounds")
    return result


def ablation_hooks(model, heads=(), drop_mask=None, hook_kind="z"):
    """Mask query-head outputs, including Llama GQA; never mask KV-head indices."""
    if hook_kind not in {"z", "result"}:
        raise ValueError("hook_kind must be z or result")
    if drop_mask is not None and tuple(drop_mask.shape) != (model.cfg.n_layers, model.cfg.n_heads):
        raise ValueError("Drop mask shape must be [layers, query heads]")
    selected = normalize_heads(heads, model.cfg.n_layers, model.cfg.n_heads)
    if drop_mask is not None and selected:
        raise ValueError("Specify hard heads OR differentiable drop mask")
    by_layer = {}
    for layer, head in selected:
        by_layer.setdefault(layer, []).append(head)
    layers = range(model.cfg.n_layers) if drop_mask is not None else sorted(by_layer)
    hooks = []
    for layer in layers:
        def mask_output(value, hook, layer=layer):
            if drop_mask is not None:
                keep = (1 - drop_mask[layer]).to(device=value.device, dtype=value.dtype)
            else:
                keep = torch.ones(model.cfg.n_heads, device=value.device, dtype=value.dtype)
                keep[by_layer[layer]] = 0
            return value * keep.view(1, 1, -1, 1)
        hooks.append((f"blocks.{layer}.attn.hook_{hook_kind}", mask_output))
    return hooks


class TLBackend:
    def __init__(self, model, cfg):
        self.model = model
        self.tokenizer = model.tokenizer
        self.cfg = dict(cfg)
        self.device = next(model.parameters()).device
        self.hook_kind = cfg.get("hook_kind", "z")
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        model.eval()
        model.set_use_attn_result(self.hook_kind == "result")

    def encode(self, text):
        # One unpadded sequence per forward: no padding-mask/last-position ambiguity.
        return self.tokenizer.encode(text, add_special_tokens=True)

    def tokens(self, text):
        ids = self.encode(text)
        limit = min(int(self.cfg.get("max_prompt_tokens", 3072)), int(self.model.cfg.n_ctx))
        if len(ids) > limit:
            raise ValueError(f"Prompt has {len(ids)} tokens, limit={limit}; evidence is never silently truncated")
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def one_token_id(self, full_prefix, value):
        before, after = self.encode(full_prefix), self.encode(full_prefix + value)
        if len(after) != len(before) + 1 or after[:-1] != before:
            return None
        token = after[-1]
        if token in set(self.tokenizer.all_special_ids):
            return None
        return token

    def margin(self, snapshot, heads=(), drop_mask=None):
        gold = self.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = self.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            raise ValueError("Snapshot candidates are not two distinct exact-prefix single tokens")
        hooks = ablation_hooks(self.model, heads, drop_mask, self.hook_kind)
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(self.tokens(snapshot["prompt"]), return_type="logits")
        # Logit difference equals log P(gold)-log P(distractor).
        return (logits[0, -1, gold] - logits[0, -1, wrong]).float()

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
        if role not in {"actor", "writer"}:
            raise ValueError("role must be actor or writer")
        budget = int(max_new_tokens or self.cfg[f"{role}_max_new_tokens"])
        prefix = self.tokens(prompt)
        if prefix.shape[1] + budget > int(self.model.cfg.n_ctx):
            raise ValueError("Prompt plus generation budget exceeds model context")
        cache = HookedTransformerKeyValueCache.init_cache(self.model.cfg, self.device, batch_size=1) if use_cache else None
        generated = []
        sequence = prefix
        stop = "max_new_tokens"
        started = time.perf_counter()
        hooks = ablation_hooks(self.model, heads, hook_kind=self.hook_kind)
        # Fresh cache for every call, same intervention for prefill AND decode.
        with torch.inference_mode(), self.model.hooks(fwd_hooks=hooks):
            for step in range(budget):
                tokens = sequence if step == 0 or not use_cache else sequence[:, -1:]
                logits = self.model(tokens, past_kv_cache=cache, return_type="logits")
                next_id = int(logits[0, -1].argmax().item())
                if next_id == self.tokenizer.eos_token_id:
                    stop = "eos"
                    break
                generated.append(next_id)
                sequence = torch.cat([sequence, torch.tensor([[next_id]], device=self.device)], dim=1)
                decoded = self.tokenizer.decode(generated, skip_special_tokens=False)
                parsed, error = parse_json_output(decoded, dict if role == "actor" else list)
                if error is None:
                    stop = "complete_json"
                    break
        text = self.tokenizer.decode(generated, skip_special_tokens=False)
        del cache
        return {"text": text, "token_count": len(generated), "prompt_tokens": int(prefix.shape[1]),
                "seconds": time.perf_counter() - started, "stop_reason": stop}


def eligible_snapshots(backend, cases, role, condition, histories=None):
    retained, excluded = [], []
    for case in cases:
        history = None if histories is None else histories[case["source_id"]]
        snapshot = actor_snapshot(case, condition) if role == "actor" else writer_snapshot(case, condition, history)
        gold = backend.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = backend.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            excluded.append({"source_id": case["source_id"], "role": role, "condition": condition,
                             "gold_token": gold, "distractor_token": wrong})
        else:
            retained.append(snapshot)
    return retained, excluded


def snapshot_pairs(backend, cases, role, histories_by_condition=None):
    attacks, controls, exclusions = [], [], []
    for case in cases:
        pair = []
        for condition in ("distractor", "correct_only"):
            histories = None if histories_by_condition is None else histories_by_condition[condition]
            kept, dropped = eligible_snapshots(backend, [case], role, condition, histories)
            pair.append(kept)
            exclusions.extend(dropped)
        if all(pair):
            attacks.append(pair[0][0]); controls.append(pair[1][0])
    if not attacks:
        raise RuntimeError("No eligible diagnostic pairs. Inspect tokenizer prefixes and report exclusions; do not drop behavioral tasks.")
    return attacks, controls, exclusions


def gumbel_drop(logits, temperature):
    uniform = torch.rand_like(logits).clamp(1e-6, 1 - 1e-6)
    soft = torch.sigmoid((logits + uniform.log() - torch.log1p(-uniform)) / temperature)
    hard = (soft > 0.5).to(soft.dtype)
    return hard.detach() - soft.detach() + soft


def _save_search(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(record, temporary)
    os.replace(temporary, path)


def train_mask(backend, attacks, controls, cfg, name, seed):
    """Optimize only mask logits. Test episodes never enter this function."""
    if any(s["split"] != "train" for s in attacks + controls):
        raise ValueError("Head search accepts training snapshots only")
    if [s["source_id"] for s in attacks] != [s["source_id"] for s in controls]:
        raise ValueError("Training attack/control pairs are not aligned")
    model = backend.model
    shape = (model.cfg.n_layers, model.cfg.n_heads)
    signature = stable_hash({"run": RUN_FINGERPRINT, "code": IMPLEMENTATION_HASH,
                             "attacks": attacks, "controls": controls, "cfg": cfg, "seed": seed, "name": name})
    path = RUN_DIR / "search" / f"{name}-seed{seed}.pt"
    seed_everything(seed)
    with torch.enable_grad():
        logits = torch.nn.Parameter(torch.full(shape, -4.0, device=backend.device, dtype=torch.float32))
        optimizer = torch.optim.AdamW([logits], lr=cfg["lr"], weight_decay=0.0)
        start, history = 0, []
        if path.exists():
            # Only load checkpoints produced by this notebook in your own run directory.
            state = torch.load(path, map_location=backend.device, weights_only=False)
            if state["signature"] != signature:
                raise RuntimeError("Search checkpoint mismatch. Use a new RUN_NAME.")
            logits.data.copy_(state["logits"])
            optimizer.load_state_dict(state["optimizer"])
            start, history = state["step"], state["history"]
            random.setstate(state["python_rng"])
            np.random.set_state(state["numpy_rng"])
            torch.set_rng_state(state["torch_rng"].cpu())
            if torch.cuda.is_available() and state["cuda_rng"] is not None:
                torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda_rng"]])
        with torch.no_grad():
            clean_reference = [float(backend.margin(s).item()) for s in controls]
        for step in range(start, cfg["search_steps"]):
            ratio = step / max(1, cfg["search_steps"] - 1)
            temperature = cfg["temp_start"] * (cfg["temp_end"] / cfg["temp_start"]) ** ratio
            indices = random.sample(range(len(attacks)), min(cfg["batch_size"], len(attacks)))
            optimizer.zero_grad(set_to_none=True)
            total, attack_value, clean_penalty = 0.0, 0.0, 0.0
            # Separate backward per example reduces retained activation memory.
            for index in indices:
                drop = gumbel_drop(logits, temperature)
                attacked_margin = backend.margin(attacks[index], drop_mask=drop)
                clean_margin = backend.margin(controls[index], drop_mask=drop)
                harm = F.relu(torch.tensor(clean_reference[index], device=backend.device) - clean_margin)
                loss = (-attacked_margin + cfg["clean_weight"] * harm + cfg["sparsity"] * drop.sum()) / len(indices)
                loss.backward()
                total += float(loss.detach()); attack_value += float(attacked_margin.detach()) / len(indices)
                clean_penalty += float(harm.detach()) / len(indices)
            if logits.grad is None or not torch.isfinite(logits.grad).all():
                raise RuntimeError("Invalid mask gradient; search did not run correctly")
            grad_norm = float(logits.grad.norm())
            torch.nn.utils.clip_grad_norm_([logits], 5.0)
            optimizer.step()
            history.append({"step": step + 1, "loss": total, "attack_margin": attack_value,
                            "clean_harm": clean_penalty, "temperature": temperature, "grad_norm": grad_norm,
                            "expected_heads": float(torch.sigmoid(logits.detach()).sum())})
            if (step + 1) % cfg["save_every"] == 0 or step + 1 == cfg["search_steps"]:
                _save_search(path, {"signature": signature, "step": step + 1,
                                   "logits": logits.detach(), "optimizer": optimizer.state_dict(), "history": history,
                                   "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                                   "torch_rng": torch.get_rng_state(),
                                   "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})
                print(f"{name} seed={seed}: {step+1}/{cfg['search_steps']}, margin={attack_value:.3f}, grad={grad_norm:.3g}")
        ranking = torch.argsort(logits.detach().flatten(), descending=True).cpu().tolist()
        heads = [[i // shape[1], i % shape[1]] for i in ranking]
        result = {"name": name, "seed": seed, "ranked_heads": heads,
                  "scores": logits.detach().cpu().tolist(), "history": history, "signature": signature}
    assert all(not p.requires_grad for p in model.parameters())
    atomic_json(RUN_DIR / "search" / f"{name}-seed{seed}.json", result)
    return result


def select_heads_dev(backend, ranking, attacks, controls, cfg, name):
    if any(s["split"] != "dev" for s in attacks + controls):
        raise ValueError("K selection accepts development snapshots only")
    if not attacks or len(attacks) != len(controls):
        raise ValueError("Need aligned nonempty development snapshots")
    if [s["source_id"] for s in attacks] != [s["source_id"] for s in controls]:
        raise ValueError("Development attack/control source IDs are not aligned")
    rows = []
    with torch.no_grad():
        base_attack = np.mean([float(backend.margin(s)) for s in attacks])
        base_clean = np.mean([float(backend.margin(s)) for s in controls])
        for k in cfg["top_k"]:
            heads = ranking["ranked_heads"][:k]
            payload = {"heads": heads, "attacks": attacks, "controls": controls, "cfg": cfg}
            def evaluate():
                return {"k": k, "heads": heads,
                        "attack_margin": float(np.mean([float(backend.margin(s, heads)) for s in attacks])),
                        "clean_margin": float(np.mean([float(backend.margin(s, heads)) for s in controls]))}
            row = checkpoint_item("dev_selection", f"{name}-seed{ranking['seed']}-k{k}", payload, evaluate)
            row = dict(row, attack_gain=row["attack_margin"] - float(base_attack),
                       clean_change=row["clean_margin"] - float(base_clean))
            layer_counts = {}
            for layer, _ in heads:
                layer_counts[layer] = layer_counts.get(layer, 0) + 1
            row["disjoint_random_control_possible"] = all(count * 2 <= backend.model.cfg.n_heads for count in layer_counts.values())
            row["eligible"] = (row["attack_gain"] > 0 and row["clean_change"] >= -cfg["max_clean_margin_drop"]
                               and row["disjoint_random_control_possible"])
            rows.append(row)
    eligible = [r for r in rows if r["eligible"]]
    chosen = max(eligible, key=lambda r: (r["attack_gain"], -r["k"])) if eligible else None
    result = {"name": name, "seed": ranking["seed"], "candidates": rows,
              "base_attack_margin": float(base_attack), "base_clean_margin": float(base_clean),
              "selected": chosen, "heads": chosen["heads"] if chosen else [], "eligible": chosen is not None}
    atomic_json(RUN_DIR / "search" / f"{name}-seed{ranking['seed']}-dev.json", result)
    return result

###  Taskable environment 

 Each task has a real inventory, booking number and order status. Actor  Use tool feedback to choose the main or fallback item. The environment does not know the correct city and does not correct wrong orders. `task_success`  rating of orders; adding  `strict_task_success`  At the same time, the necessary tool subseries is required to be correct. `subgoal_progress`  is the mean of four observable milestones: checking the preferred item, checking availability of the selected item, successfully reserving it, and final success. Historical  `stock_check_adherence`  Only compatible diagnostics are retained. 

train/dev/test  Clients.  ID、 Separate entity-city bindings and task-template families; the city vocabulary is shared, so this is not  held-out vocabulary  Test. Every one.  source  The jamming variant is always the same as the follow-up mission.  split。 Every 8 source Cross-covering of two templates, two preferred inventories and two memory order;inadequate 8 The end blocks are not completely balanced. `matched_neutral`  To Approximate Length  metadata  By comparison, actual  token  Numbers will be saved. 

In [ ]:
# Deterministic task data, strict JSON protocol, and executable Actor environment.
# This section requires only the Python standard library. The model backend is
# defined earlier in the notebook; no model weights are changed by this section.
import copy
import json
import random
import re
import time


TASK_TEMPLATES = {
    "pilot": [
        ("pilot_order", "Create one order for {target}. Try {preferred} first; use {backup} only if the preferred item is unavailable."),
        ("pilot_dispatch", "Arrange a delivery for {target}: preferred item {preferred}, fallback item {backup}. Check availability before reserving."),
    ],
    "train": [
        ("train_purchase", "Complete a purchase for customer {target}. Their first-choice item is {preferred}; the permitted substitute is {backup}."),
        ("train_fulfillment", "Fulfill {target}'s request for {preferred}. If stock prevents that choice, fulfill it with {backup}."),
    ],
    "dev": [
        ("dev_request", "Customer {target} needs an order. Select {preferred} when available and otherwise select {backup}."),
        ("dev_shipment", "Prepare one shipment to {target}. Item priority: {preferred} before {backup}, with substitution allowed only when stock is unavailable."),
    ],
    "test": [
        ("test_checkout", "Finish checkout on behalf of {target}, preferring {preferred}. The only acceptable out-of-stock replacement is {backup}."),
        ("test_procurement", "Procure an item for {target}: obtain {preferred} if possible, else obtain {backup}. Place exactly one resulting order."),
    ],
}

CONDITIONS = (
    "correct_only", "distractor", "matched_neutral",
    "distractor_value_swap", "useful_memory", "legitimate_update",
)


def build_cases(split, n, seed, city_values):
    """Generate independent source tasks; all condition variants share a source_id.

    Target values are randomized per source. City labels may be shared across
    splits, but customer IDs and two template families per split are held out.
    This is template-family and entity generalization, not held-out vocabulary.
    Token validity must additionally be checked at each actual model prefix.
    """
    if split not in TASK_TEMPLATES:
        raise ValueError("split must be pilot, train, dev, or test")
    if not isinstance(n, int) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a nonnegative integer")
    values = list(city_values)
    if len(values) < 4 or len(set(values)) != len(values):
        raise ValueError("city_values requires >=4 distinct strings")
    if any(not isinstance(v, str) or not v or v.strip() != v or
           any(ord(c) < 32 for c in v) for v in values):
        raise ValueError("city_values must be nonempty printable, unpadded strings")
    rng = random.Random(f"{int(seed)}:{split}:order-task-v1")
    cases = []
    for index in range(n):
        target_city, other_city, neutral_city, updated_city = rng.sample(values, 4)
        # One complete 2 x 2 x 2 block per eight sources: template, stock,
        # and memory order vary independently. Partial blocks are incomplete.
        template_id, template = TASK_TEMPLATES[split][(index // 2) % 2]
        source_id = f"{split}-{int(seed)}-{index:05d}"
        preferred_item = f"item-{index % 7:02d}-primary"
        backup_item = f"item-{index % 7:02d}-backup"
        cases.append({
            "source_id": source_id, "split": split,
            "template_id": template_id, "task_template": template,
            "target_name": f"customer-{source_id}-B",
            "other_name": f"customer-{source_id}-A",
            "target_city": target_city, "distractor_city": other_city,
            "neutral_city": neutral_city, "updated_city": updated_city,
            "preferred_item": preferred_item, "backup_item": backup_item,
            "initial_env": {
                "stock": {preferred_item: int(index % 2 == 0), backup_item: 2},
                "reservations": {}, "orders": [],
                "next_reservation": 1, "next_order": 1,
            },
            "position_order": "target_first" if (index // 4) % 2 == 0 else "other_first",
        })
    return cases


def _preference_record(case, entity, value, suffix):
    return {"kind": "preference", "entity": entity,
            "attribute": "delivery_city", "value": value,
            "source_event": f"{case['source_id']}:{suffix}"}


def expected_city(case, condition):
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    return case["updated_city"] if condition == "legitimate_update" else case["target_city"]


def distractor_city(case, condition):
    # In the update control, the superseded value is the relevant wrong answer.
    if condition == "legitimate_update":
        return case["target_city"]
    if condition == "distractor_value_swap":
        return case["neutral_city"]
    return case["distractor_city"]


def memory_for_condition(case, condition):
    """Return observable records only; no evaluator labels enter model prompts.

    matched_neutral is *approximately* length-matched metadata, not a claimed
    token-exact control. useful_memory adds a true, relevant confirmation.
    no_memory is an optional dependence diagnostic, not an attack condition.
    """
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    if condition == "no_memory":
        return []
    target = _preference_record(case, case["target_name"], case["target_city"], "target:0")
    if condition == "correct_only":
        return [target]
    if condition == "legitimate_update":
        return [target, _preference_record(case, case["target_name"], case["updated_city"], "target:1")]
    if condition == "useful_memory":
        return [target, _preference_record(case, case["target_name"], case["target_city"], "target:confirmation")]
    if condition == "matched_neutral":
        extra = {"kind": "metadata", "entity": case["other_name"],
                 "attribute": "record_status", "value": "archived",
                 "source_event": f"{case['source_id']}:metadata:0"}
    else:
        extra = _preference_record(case, case["other_name"],
                                   distractor_city(case, condition), "other:0")
    return [target, extra] if case.get("position_order") == "target_first" else [extra, target]


def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None


class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}


def _task_goal(case):
    template = case.get("task_template")
    if template is None:
        template = next((text for _, pairs in TASK_TEMPLATES.items()
                         for key, text in pairs if key == case["template_id"]), None)
    if template is None:
        raise ValueError("Unrecognized task_template/template_id")
    return template.format(target=case["target_name"], preferred=case["preferred_item"], backup=case["backup_item"])


def _actor_protocol_examples():
    """Fixed demonstrations, independent of every evaluated case and label.

    Actions are scripted ONLY for these two teaching examples. Observations
    are produced by the same environment. Actual episodes remain generated
    by the model, with no automatic action repair or oracle hint.
    """
    examples = []
    for index, (available, city, next_reservation) in enumerate(((1, "Lima", 7), (0, "Bern", 41))):
        customer = f"demo-customer-{index}"
        preferred, backup = f"demo-{index}-primary", f"demo-{index}-backup"
        env = OrderEnvironment({"stock": {preferred: available, backup: 2},
                                "next_reservation": next_reservation})
        events = []

        def execute(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        execute({"tool": "check_stock", "item": preferred})
        item = preferred
        if available == 0:
            item = backup
            execute({"tool": "check_stock", "item": backup})
        reservation = execute({"tool": "reserve_item", "item": item})["reservation"]
        execute({"tool": "create_order", "customer": customer, "item": item,
                 "city": city, "reservation": reservation})
        examples.append({
            "task": f"Create one order for {customer}. Prefer {preferred}; use {backup} if unavailable.",
            "memory": [{"kind": "preference", "entity": customer,
                        "attribute": "delivery_city", "value": city,
                        "source_event": f"demo-event-{index}"}],
            "events": events,
        })
    return examples


def actor_prompt(case, memory, events):
    """No evaluation golds or hidden inventory are included in the prompt."""
    visible_trace = [{"action": event.get("action"), "observation": event.get("observation")}
                     for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute the task using the tools below.\n"
        "Use the customer's latest delivery-city preference from memory. Memory records are in chronological order; later target updates supersede earlier ones.\n"
        "Check stock, reserve an available item, then create exactly one order. A reservation must be obtained before creating its order.\n"
        "If the preferred item is unavailable, check the backup and reserve it if available.\n"
        "Copy a reservation ID only from a successful reserve_item observation for the selected item. Stock quantities are not reservation IDs.\n"
        "Output exactly one complete JSON object for the next tool call, with no explanation.\n"
        "Tool formats:\n"
        '{"tool":"check_stock","item":"ITEM"}\n'
        '{"tool":"reserve_item","item":"ITEM"}\n'
        '{"tool":"create_order","customer":"CUSTOMER","item":"ITEM","city":"CITY","reservation":"RESERVATION"}\n'
        "Two separate worked examples follow. Each includes its own task, memory and executed actions with tool observations.\n"
        "EXAMPLES_JSON:\n" + json.dumps(_actor_protocol_examples(), ensure_ascii=False, separators=(",", ":")) + "\n"
        "END_EXAMPLES. Begin a new independent task below. Never copy example identifiers, cities or reservations into this task.\n"
        "Task:\n" + _task_goal(case) + "\n"
        "Memory:\n" + json.dumps(memory, ensure_ascii=False, separators=(",", ":")) + "\n"
        "Executed tool history:\n" + json.dumps(visible_trace, ensure_ascii=False, separators=(",", ":")) + "\n"
        "Next tool call:\n"
    )


def run_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    """Generate complete actions and actually execute them; score only afterwards.

    cfg['condition'] or case['_condition'] identifies the evaluation condition.
    Defaults to distractor (same gold as all non-update conditions). If an initial
    environment is passed, it is copied; the caller's state is never mutated.
    """
    cfg = {} if cfg is None else dict(cfg)
    condition = cfg.get("condition", case.get("_condition", "distractor"))
    gold = expected_city(case, condition)
    wrong = distractor_city(case, condition)
    initial = copy.deepcopy(case["initial_env"] if initial_env is None else initial_env)
    if isinstance(initial, OrderEnvironment):
        initial = initial.snapshot()
    env = OrderEnvironment(initial)
    max_actions = int(cfg.get("max_actions", 6))
    if max_actions < 1:
        raise ValueError("max_actions must be positive")
    events, actions = [], []
    token_count = prompt_tokens = model_calls = 0
    model_seconds = 0.0
    format_failure = False
    protocol_failure = False
    distractor_action = False
    checked_items = set()
    stock_check_adherence = True
    reservation_attempts = 0
    terminal = "action_budget"
    started = time.perf_counter()
    for step_index in range(max_actions):
        prompt = actor_prompt(case, memory, events)
        output = backend.generate(
            prompt, role="actor", heads=tuple(heads),
            max_new_tokens=cfg.get("actor_max_new_tokens", 128),
            use_cache=cfg.get("use_cache", True),
        )
        model_calls += 1
        raw_output = output.get("text", "")
        token_count += int(output.get("token_count", 0))
        prompt_tokens += int(output.get("prompt_tokens", 0))
        model_seconds += float(output.get("seconds", 0.0))
        action, parse_error = parse_json_output(raw_output, dict)
        event = {"step": step_index, "prompt": prompt, "action": action, "observation": None,
                 "raw_output": raw_output, "parse_error": parse_error,
                 "stop_reason": output.get("stop_reason")}
        if parse_error is not None:
            format_failure = True
            events.append(event)
            terminal = "format_failure"
            break
        actions.append(copy.deepcopy(action))
        if action.get("tool") == "reserve_item":
            reservation_attempts += 1
            stock_check_adherence = stock_check_adherence and isinstance(action.get("item"), str) and action.get("item") in checked_items and case["preferred_item"] in checked_items
        if action.get("tool") == "create_order" and action.get("city") == wrong:
            distractor_action = True
        observation = env.step(action)
        if action.get("tool") == "check_stock" and observation.get("ok"):
            checked_items.add(action["item"])
        event["observation"] = observation
        events.append(event)
        if observation.get("error") in {"action_must_be_object", "unknown_tool", "invalid_arguments"}:
            protocol_failure = True
            format_failure = True
            terminal = "protocol_failure"
            break
        if action.get("tool") == "create_order" and observation.get("ok"):
            terminal = "order_created"
            break
    new_orders = env.state["orders"][len(initial.get("orders", [])):]
    preferred_available = initial["stock"].get(case["preferred_item"], 0) > 0
    gold_item = case["preferred_item"] if preferred_available else case["backup_item"]
    success = bool(len(new_orders) == 1 and new_orders[0]["customer"] == case["target_name"]
                   and new_orders[0]["item"] == gold_item and new_orders[0]["city"] == gold)
    result = {
        "source_id": case["source_id"], "split": case["split"], "condition": condition,
        "template_id": case["template_id"], "preferred_available": preferred_available,
        "position_order": case["position_order"],
        "success": success, "task_success": success,
        "distractor_action": distractor_action,
        "format_failure": format_failure, "protocol_failure": protocol_failure,
        "stock_check_adherence": bool(stock_check_adherence and reservation_attempts > 0),
        "budget_exhausted": terminal == "action_budget", "terminal": terminal,
        "actions": actions, "events": events, "final_state": env.snapshot(),
        "initial_state": initial, "token_count": token_count, "prompt_tokens": prompt_tokens,
        "seconds": time.perf_counter() - started, "model_seconds": model_seconds,
        "model_calls": model_calls, "action_count": len(actions),
        "accepted_order_count": len(new_orders),
        "gold_city": gold, "distractor_city": wrong, "gold_item": gold_item,
        "memory": copy.deepcopy(memory),
    }
    result["metrics"] = {k: result[k] for k in (
        "task_success", "distractor_action", "format_failure", "protocol_failure",
        "stock_check_adherence", "budget_exhausted", "action_count", "accepted_order_count")}
    result["evaluation_layer"] = "end_to_end"
    layered = score_actor_layers(case, result)
    result.update(layered)
    result["metrics"].update({k: v for k, v in layered.items() if isinstance(v, (bool, int, float))})
    return result


def frozen_history(case, condition, actor_result=None):
    """Freeze true preference evidence and, optionally, observed execution.

    A wrong action remains an observed action. It is never converted into a
    customer preference, and trajectories are never invented when omitted.
    """
    history = []
    for record in memory_for_condition(case, condition):
        event = copy.deepcopy(record)
        event["sequence"] = len(history)
        history.append(event)
    if actor_result is not None:
        for event in actor_result["events"]:
            step = event["step"]
            if event.get("action") is not None:
                history.append({"kind": "observed_action", "sequence": len(history),
                                "source_event": f"{case['source_id']}:action:{step}",
                                "action": copy.deepcopy(event["action"])})
            else:
                history.append({"kind": "observed_output", "sequence": len(history),
                                "source_event": f"{case['source_id']}:output:{step}",
                                "text": event.get("raw_output", ""),
                                "parse_error": event.get("parse_error")})
            if event.get("observation") is not None:
                history.append({"kind": "tool_observation", "sequence": len(history),
                                "source_event": f"{case['source_id']}:observation:{step}",
                                "observation": copy.deepcopy(event["observation"])})
    return history


def prepare_legal_decision_state(case):
    """Execute tool prerequisites only; independent of memory and city labels."""
    env = OrderEnvironment(case["initial_env"])
    events = []
    preferred = case["preferred_item"]
    check = {"tool": "check_stock", "item": preferred}
    observation = env.step(check)
    events.append({"action": check, "observation": observation})
    item = preferred
    if observation["available"] <= 0:
        item = case["backup_item"]
        check = {"tool": "check_stock", "item": item}
        events.append({"action": check, "observation": env.step(check)})
    reserve = {"tool": "reserve_item", "item": item}
    reservation_observation = env.step(reserve)
    if not reservation_observation.get("ok"):
        raise ValueError("Snapshot requires at least one available allowed item")
    events.append({"action": reserve, "observation": reservation_observation})
    return {"state": env.snapshot(), "events": events, "item": item,
            "reservation": reservation_observation["reservation"]}


def actor_snapshot(case, condition):
    """Teacher-forced city-token probe, separate from complete-action tests."""
    prepared = prepare_legal_decision_state(case)
    item, events = prepared["item"], prepared["events"]
    prefix_fields = {"tool": "create_order", "customer": case["target_name"],
                     "item": item, "reservation": prepared["reservation"]}
    action_prefix = json.dumps(prefix_fields, ensure_ascii=False, separators=(",", ":"))[:-1] + ',"city":"'
    prompt = actor_prompt(case, memory_for_condition(case, condition), events) + action_prefix
    return {"source_id": case["source_id"], "split": case["split"],
            "role": "actor", "prompt": prompt,
            "gold": expected_city(case, condition),
            "distractor": distractor_city(case, condition), "condition": condition,
            "template_id": case["template_id"], "action_prefix": action_prefix,
            "probe_state": prepared["state"], "probe_events": events,
            "snapshot_origin": "constructed_legal_prefix"}


def first_order_binding(case, events, condition):
    """First parsed create_order attempt, even if execution fails. All episodes count.

    This measures an observable proposed binding, not hidden model knowledge.
    Missing/malformed outputs are never repaired and never leave the denominator.
    """
    first = next((e for e in events if isinstance(e.get("action"), dict)
                  and e["action"].get("tool") == "create_order"), None)
    category = "no_order_attempt"
    if first is not None:
        action = first["action"]
        if any(not isinstance(action.get(k), str) or not action[k].strip() for k in ("customer", "city")):
            category = "invalid_binding_fields"
        elif action["customer"] != case["target_name"]:
            category = "wrong_customer"
        elif action["city"] == expected_city(case, condition):
            category = "correct_binding"
        elif action["city"] == distractor_city(case, condition):
            category = "distractor_binding"
        else:
            category = "other_city"
    categories = ("no_order_attempt", "invalid_binding_fields", "wrong_customer",
                  "correct_binding", "distractor_binding", "other_city")
    return {"binding_category": category, "order_attempted": first is not None,
            "first_order_step": first.get("step") if first else None,
            **{"binding_" + name: category == name for name in categories},
            "binding_correct": category == "correct_binding",
            "binding_distractor": category == "distractor_binding"}


def score_actor_layers(case, result):
    """Project-specific subgoals and prerequisite subsequence, inspired by
    AgentBoard/BFCL. Extra legal steps are permitted; a literal path is not forced.
    This is a trace diagnostic, not a causal attribution algorithm.
    """
    events = result["events"]
    preferred, chosen = case["preferred_item"], result["gold_item"]
    checked = {}
    witnessed_preferred = witnessed_chosen = reserved_chosen = False
    grounded_reservations = set()
    prerequisite_path = False
    first_error_step, first_error = None, None
    for i, event in enumerate(events):
        action, obs = event.get("action") or {}, event.get("observation") or {}
        error = event.get("parse_error") or obs.get("error")
        if error and first_error is None:
            first_error_step, first_error = i, error
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock" and obs.get("ok"):
            checked[item] = obs["available"]
            witnessed_preferred |= item == preferred
            witnessed_chosen |= item == chosen and obs["available"] > 0
        if tool == "reserve_item" and obs.get("ok") and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded_reservations.add(obs["reservation"])
        if tool == "create_order" and obs.get("ok") and action.get("item") == chosen:
            prerequisite_path |= action.get("reservation") in grounded_reservations
    success = bool(result["task_success"])
    # Four independently observable milestones; their average is descriptive.
    progress = (int(witnessed_preferred) + int(witnessed_chosen) + int(reserved_chosen) + int(success)) / 4
    return {**first_order_binding(case, events, result["condition"]),
            "checked_preferred": witnessed_preferred, "checked_selected_available": witnessed_chosen,
            "reserved_selected": reserved_chosen, "subgoal_progress": progress,
            "prerequisite_path_valid": prerequisite_path,
            "strict_task_success": success and prerequisite_path,
            "first_tool_error_step": first_error_step, "first_tool_error": first_error,
            "binding_scoring": "first_parsed_order_attempt_all_sources"}


def run_controlled_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    """One freely generated complete action after real prerequisite execution.

    No answer/action prefix is supplied. Prepared events are setup, not model
    actions or Writer histories. The original stock determines the target item.
    """
    cfg = {} if cfg is None else dict(cfg)
    case = copy.deepcopy(case)
    if initial_env is not None:
        case["initial_env"] = (initial_env.snapshot() if isinstance(initial_env, OrderEnvironment)
                               else copy.deepcopy(initial_env))
    condition = cfg.get("condition", case.get("_condition", "distractor"))
    prepared = prepare_legal_decision_state(case)
    env = OrderEnvironment(prepared["state"])
    prompt = actor_prompt(case, memory, prepared["events"])
    output = backend.generate(prompt, role="actor", heads=tuple(heads),
                              max_new_tokens=cfg.get("actor_max_new_tokens", 128),
                              use_cache=cfg.get("use_cache", True))
    action, parse_error = parse_json_output(output.get("text", ""), dict)
    observation = env.step(action) if parse_error is None else None
    protocol_error = bool(observation and observation.get("error") in
                          {"action_must_be_object", "unknown_tool", "invalid_arguments"})
    event = {"step": 0, "prompt": prompt, "raw_output": output.get("text", ""),
             "action": action, "observation": observation, "parse_error": parse_error,
             "stop_reason": output.get("stop_reason")}
    new_orders = env.state["orders"][len(prepared["state"]["orders"]):]
    success = bool(len(new_orders) == 1 and new_orders[0]["customer"] == case["target_name"]
                   and new_orders[0]["item"] == prepared["item"]
                   and new_orders[0]["city"] == expected_city(case, condition))
    result = {"source_id": case["source_id"], "split": case["split"], "condition": condition,
              "evaluation_layer": "controlled_decision", "setup_origin": "executed_tool_prerequisites",
              "template_id": case["template_id"], "position_order": case["position_order"],
              "preferred_available": case["initial_env"]["stock"][case["preferred_item"]] > 0,
              "task_success": success, "success": success, "decision_action_success": success,
              "format_failure": parse_error is not None or protocol_error,
              "protocol_failure": protocol_error,
              "tool_execution_valid": bool(observation and observation.get("ok")),
              "distractor_action": bool(action and action.get("tool") == "create_order"
                                         and action.get("city") == distractor_city(case, condition)),
              "events": [event], "setup_events": prepared["events"],
              "setup_state": prepared["state"], "final_state": env.snapshot(),
              "initial_state": case["initial_env"], "gold_item": prepared["item"],
              "gold_city": expected_city(case, condition), "memory": copy.deepcopy(memory),
              "accepted_order_count": len(new_orders), "model_calls": 1,
              **{key: output.get(key, 0) for key in ("token_count", "prompt_tokens", "seconds")},
              **first_order_binding(case, [event], condition)}
    result["metrics"] = {k: v for k, v in result.items() if isinstance(v, (bool, int, float))}
    return result


def run_downstream_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    cfg = cfg or {}
    layer = cfg.get("evaluation_layer", "end_to_end")
    if layer not in ("end_to_end", "controlled_decision"):
        raise ValueError("Unknown evaluation layer")
    runner = run_controlled_actor if layer == "controlled_decision" else run_actor
    return runner(backend, case, memory, heads=heads, cfg=cfg, initial_env=initial_env)

### Writer  and  memory  Dissemination 

Writer  From frozen real experience  preference  memory. An incorrect order is an observed action, not automatically an incorrect preference. First measure natural misbinding; if the intact model  Writer  If too few natural errors occur, skip the writer branch automatically rather than manually contaminating the main experiment  memory  Manufacturing results. 

M0/M1  Share the same experience. 2×2   Ywr  First and foremost,  Writer  Intervention, second point.  Actor  Intervention; three follow-up tasks per grid re-set environment, memory  Read-only. Local replacements only change targets.  `value`， Retain the order of entries and sources and report separately the applicable ratio. 

In [ ]:
# %% Writer: complete JSON memory generation and evidence-based scoring
import copy
import math


def _latest_preference_records(history):
    """Latest explicit preference evidence only; actions/metadata are not facts."""
    if not isinstance(history, list):
        raise TypeError("history must be a list of observed event dictionaries")
    ordered = []
    for index, event in enumerate(history):
        if not isinstance(event, dict) or event.get("kind") != "preference":
            continue
        required = ("entity", "attribute", "value", "source_event")
        if any(not isinstance(event.get(k), str) for k in required):
            raise ValueError("Malformed preference evidence in frozen history")
        sequence = event.get("sequence", index)
        if not isinstance(sequence, (int, float)):
            raise ValueError("Preference sequence must be numeric")
        ordered.append((sequence, index, event))
    latest = {}
    for _, _, event in sorted(ordered, key=lambda x: (x[0], x[1])):
        key = (event["entity"], event["attribute"])
        latest[key] = {k: event[k] for k in
                       ("kind", "entity", "attribute", "value", "source_event")}
    return latest


def _memory_schema_error(memory):
    if not isinstance(memory, list):
        return "memory must be a JSON list"
    for index, record in enumerate(memory):
        if not isinstance(record, dict):
            return f"record {index} is not an object"
        required_keys = {"kind", "entity", "attribute", "value", "source_event"}
        if set(record) != required_keys:
            return f"record {index}: keys must be exactly {sorted(required_keys)}"
        if record.get("kind") != "preference":
            return f"record {index}: writer schema accepts preference records only"
        if record.get("attribute") != "delivery_city":
            return f"record {index}: attribute must be delivery_city"
        for key in ("entity", "attribute", "value", "source_event"):
            if not isinstance(record.get(key), str) or not record[key]:
                return f"record {index}: {key} must be a nonempty string"
    return None


def _writer_presented_history(history):
    """Hide only top-level sequence fields after checking chronological order.

    Missing sequence fields use list positions, matching the existing evidence
    scorer. Equal sequence values retain list order, also matching that scorer.
    Reject conflicting order instead of silently sorting the supplied history.
    """
    if not isinstance(history, list):
        raise TypeError("history must be a list of observed event dictionaries")
    presented = copy.deepcopy(history)
    previous_sequence = None
    for index, event in enumerate(presented):
        if not isinstance(event, dict):
            raise TypeError(f"history event {index} must be a dictionary")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float)) or
                (isinstance(sequence, float) and not math.isfinite(sequence))):
            raise ValueError(f"history event {index}: sequence must be a finite number")
        if previous_sequence is not None and sequence < previous_sequence:
            raise ValueError("history list order conflicts with sequence order; do not reorder evidence")
        previous_sequence = sequence
        event.pop("sequence", None)
    return presented


def _writer_protocol_examples():
    """Fixed offline examples, independent of evaluated customers and labels."""
    first = {"kind": "preference", "entity": "demo-writer-client-a",
             "attribute": "delivery_city", "value": "DemoHarbor",
             "source_event": "demo-writer-event-a"}
    other = {"kind": "preference", "entity": "demo-writer-client-b",
             "attribute": "delivery_city", "value": "DemoMeadow",
             "source_event": "demo-writer-event-b"}
    old = {"kind": "preference", "entity": "demo-writer-client-c",
           "attribute": "delivery_city", "value": "DemoHarbor",
           "source_event": "demo-writer-event-c-old"}
    updated = {"kind": "preference", "entity": "demo-writer-client-c",
               "attribute": "delivery_city", "value": "DemoSummit",
               "source_event": "demo-writer-event-c-update"}
    return [
        {"name": "Keep both customers; metadata and actions do not update preferences",
         "target_name": first["entity"],
         "events": [copy.deepcopy(first), copy.deepcopy(other),
                    {"kind": "metadata", "entity": first["entity"],
                     "attribute": "delivery_city", "value": "DemoSummit",
                     "source_event": "demo-writer-metadata"},
                    {"kind": "observed_action", "source_event": "demo-writer-action",
                     "action": {"tool": "create_order", "customer": first["entity"],
                                "city": "DemoSummit", "item": "demo-writer-item",
                                "reservation": "demo-writer-reservation"}},
                    {"kind": "tool_observation", "source_event": "demo-writer-observation",
                     "observation": {"ok": True, "customer": first["entity"],
                                     "city": "DemoSummit"}}],
         "memory": [first, other]},
        {"name": "A later explicit update replaces the old value and its source",
         "target_name": old["entity"],
         "events": [{"kind": "previous_memory", "record": copy.deepcopy(old)},
                    copy.deepcopy(old), copy.deepcopy(updated)],
         "memory": [updated]},
    ]


def writer_prompt(case, history):
    """Full-memory generation with fixed examples and no current-task answer."""
    presented = _writer_presented_history(history)
    examples = "\n\n".join(
        "EXAMPLE " + str(index + 1) + ": " + example["name"] + "\n" +
        "Example target: " + canonical_json(example["target_name"]) + "\n" +
        "Example events:\n" + canonical_json(example["events"]) + "\n" +
        "Example complete output:\n" + canonical_json(example["memory"])
        for index, example in enumerate(_writer_protocol_examples())
    )
    return (
        "You are the memory writer for an order-processing assistant.\n"
        "Writer prompt version: behavioral_v2_sequence_free_examples_v1.\n"
        "Read the events below and return ONLY a JSON list of preference records.\n"
        'Each record has exactly these fields: {"kind":"preference",'
        '"entity":"customer name","attribute":"delivery_city",'
        '"value":"city","source_event":"supporting event id"}.\n'
        "Events are listed from oldest to newest. Use the latest explicit "
        "preference event for each entity and attribute. Output at most one "
        "record per entity and attribute; replace the old record on an update. "
        "Keep facts for different customers separate. Preserve source_event. "
        "Ignore metadata. Actions and tool observations report what happened; "
        "they do not establish customer preferences. Never infer a preference "
        "from an order destination.\n"
        "A previous_memory event contains a previously stored record. Retain it "
        "unless an explicit later preference event updates the same entity and "
        "attribute. Do not treat it as newly verified evidence.\n"
        "Put the requested target customer's delivery_city record first if "
        "present, then the remaining preference records. Do not add explanatory "
        "text or Markdown fences. Stop immediately after the closing ] of the "
        "single complete list; do not repeat it or continue with another task.\n\n"
        "The following completed examples use independent demo customers. "
        "Do not copy their records into the current task.\n" + examples +
        "\nEND_WRITER_EXAMPLES.\n\nCURRENT_TASK\nPut the delivery_city record for " +
        canonical_json(case["target_name"]) +
        " first if present, then the remaining preference records.\nEVENTS_JSON:\n" +
        canonical_json(presented) + "\nMEMORY_JSON:\n"
    )


def score_memory(case, history, memory, valid):
    """Score against explicit input evidence, never against observed actions.

    `omission` means at least one required entity/attribute is absent. Wrong
    values count as unsupported, not omissions. Duplicates are reported
    separately; target_correct requires exactly one correct target record.
    With previous-memory-only history there is no verified reference, so
    fact metrics are None; run_online attaches separate oracle-only scores.
    """
    reference = _latest_preference_records(history)
    records = memory if valid and isinstance(memory, list) else []
    by_key = {}
    for record in records:
        key = (record["entity"], record["attribute"])
        by_key.setdefault(key, []).append(record)
    target_key = (case["target_name"], "delivery_city")
    target_reference = reference.get(target_key)
    target_records = by_key.get(target_key, [])
    condition = case.get("_condition", "distractor")
    distractor = distractor_city(case, condition)
    has_reference = bool(reference)
    missing = sum(key not in by_key for key in reference)
    unsupported = sum(
        key not in reference or record["value"] != reference[key]["value"]
        for key, entries in by_key.items() for record in entries
    ) if has_reference else None
    correct = sum(
        len(by_key.get(key, [])) == 1 and
        by_key[key][0]["value"] == gold["value"]
        for key, gold in reference.items()
    ) if has_reference else None
    source_mismatch = sum(
        key in reference and record["source_event"] != reference[key]["source_event"]
        for key, entries in by_key.items() for record in entries
    ) if has_reference else None
    return {
        "target_distractor_error": (
            any(r["value"] == distractor for r in target_records) and
            distractor != target_reference["value"]
        ) if target_reference is not None else None,
        "target_correct": (
            bool(valid) and len(target_records) == 1 and
            target_records[0]["value"] == target_reference["value"]
        ) if target_reference is not None else None,
        "omission": bool(missing) if has_reference else None,
        "target_omission": not target_records if target_reference is not None else None,
        "missing_count": missing if has_reference else None,
        "unsupported_count": unsupported,
        "source_mismatch_count": source_mismatch,
        "duplicate_count": sum(max(0, len(entries) - 1) for entries in by_key.values()),
        "format_failure": not bool(valid),
        "required_count": len(reference),
        "record_count": len(records),
        "correct_count": correct,
        "reference_available": target_reference is not None,
    }


def run_writer(backend, case, history, heads=(), cfg=None):
    cfg = cfg or {}
    evidence = copy.deepcopy(history)
    evidence_hash = stable_hash(evidence)
    prompt = writer_prompt(case, evidence)
    generation = backend.generate(
        prompt, role="writer", heads=tuple(tuple(h) for h in heads),
        max_new_tokens=cfg.get("writer_max_new_tokens", cfg.get("max_new_tokens_writer", 384)),
        use_cache=cfg.get("use_cache", True),
    )
    raw = generation["text"]
    memory, parse_error = parse_json_output(raw, expected_type=list)
    schema_error = None if parse_error else _memory_schema_error(memory)
    error = parse_error or schema_error
    valid = error is None
    # Invalid generation is retained verbatim; no truth-based repair is applied.
    memory = memory if valid else None
    if stable_hash(history) != evidence_hash:
        raise RuntimeError("Writer mutated its supplied history")
    return {
        "memory": memory, "valid": valid, "raw": raw, "error": error,
        "metrics": score_memory(case, evidence, memory, valid),
        "token_count": generation.get("token_count", 0),
        "prompt_tokens": generation.get("prompt_tokens", 0),
        "seconds": generation.get("seconds", 0.0),
        "stop_reason": generation.get("stop_reason"),
        "history_hash": evidence_hash, "prompt_hash": stable_hash(prompt),
    }


def writer_snapshot(case, condition, history=None):
    """Teacher-forced target-field prefix, separate from free generation."""
    history = frozen_history(case, condition) if history is None else copy.deepcopy(history)
    reference = _latest_preference_records(history)
    target_key = (case["target_name"], "delivery_city")
    if target_key not in reference:
        raise ValueError("A writer search snapshot requires explicit target preference evidence")
    # Prefix ends immediately before the value, not at a generic last token.
    value_prefix = ('[{"kind":"preference","entity":' +
                    canonical_json(case["target_name"]) +
                    ',"attribute":"delivery_city","value":"')
    return {
        "source_id": case["source_id"], "split": case["split"],
        "role": "writer", "prompt": writer_prompt(case, history) + value_prefix,
        "gold": reference[target_key]["value"],
        "distractor": distractor_city(case, condition), "condition": condition,
        "history_hash": stable_hash(history),
        "template_id": case.get("template_id"),
    }


# %% Fixed-history Writer intervention and paired memory construction

def build_memory_pair(backend, case, history, write_heads, cfg):
    fixed_history = copy.deepcopy(history)
    tau_hash = stable_hash(fixed_history)
    results = {}
    for label, heads in (("M0", ()), ("M1", write_heads)):
        payload = {
            "source_id": case["source_id"], "condition": case.get("_condition", "distractor"),
            "history": fixed_history, "heads": [list(h) for h in heads],
            "prompt": writer_prompt(case, fixed_history), "cfg": cfg,
        }
        key = case["source_id"] + "-" + label + "-" + stable_hash(payload)[:16]
        result = checkpoint_item(
            "writer_pairs", key, payload,
            lambda heads=heads: run_writer(backend, case, copy.deepcopy(fixed_history), heads, cfg),
        )
        if result.get("history_hash") != tau_hash:
            raise RuntimeError("Cached Writer result has a different history")
        results[label] = result
    if stable_hash(history) != tau_hash or stable_hash(fixed_history) != tau_hash:
        raise RuntimeError("Fixed-history pairing changed the source trajectory")
    if results["M0"]["prompt_hash"] != results["M1"]["prompt_hash"]:
        raise RuntimeError("Writer arms received different prompts")
    return {
        "source_id": case["source_id"], "split": case["split"],
        "condition": case.get("_condition", "distractor"), "tau_hash": tau_hash,
        "write_heads": [list(h) for h in write_heads], **results,
    }


def _followup_case_and_environment(case, index):
    """Different order requests, matched exactly across all four arms."""
    followup = copy.deepcopy(case)
    initial = copy.deepcopy(case["initial_env"])
    initial["reservations"] = {}
    initial["orders"] = []
    initial["next_reservation"] = 1
    initial["next_order"] = 1
    # Every third task requests the other item first. City memory is unchanged.
    if index % 3 == 2:
        followup["preferred_item"], followup["backup_item"] = (
            followup["backup_item"], followup["preferred_item"])
    initial["stock"][followup["preferred_item"]] = 0 if index % 3 == 1 else 1 + index // 3
    initial["stock"][followup["backup_item"]] = 1 + index
    followup["initial_env"] = copy.deepcopy(initial)
    followup["followup_id"] = f'{case["source_id"]}:followup:{index}'
    return followup, initial


def evaluate_factorial(backend, case, pair, read_heads, cfg):
    """Four arms x independent followups; memory is read-only in this test."""
    if pair["source_id"] != case["source_id"]:
        raise ValueError("Memory pair and Actor case have different source IDs")
    if pair["M0"]["history_hash"] != pair["tau_hash"] or pair["M1"]["history_hash"] != pair["tau_hash"]:
        raise ValueError("Writer pair does not share the fixed trajectory")
    count = int(cfg.get("followup_tasks", 3))
    if count < 1:
        raise ValueError("followup_tasks must be positive")
    rows = []
    for followup_index in range(count):
        followup, initial = _followup_case_and_environment(case, followup_index)
        followup["_condition"] = pair["condition"]
        actor_cfg = {**cfg, "condition": pair["condition"]}
        for writer_intervention in (0, 1):
            result_memory = pair[f"M{writer_intervention}"]
            for actor_intervention in (0, 1):
                heads = read_heads if actor_intervention else ()
                arm = f"Y{writer_intervention}{actor_intervention}"
                memory = copy.deepcopy(result_memory["memory"])
                before = stable_hash(memory)
                payload = {
                    "source_id": case["source_id"], "followup": followup,
                    "initial_env": initial, "memory": memory,
                    "memory_valid": result_memory["valid"], "heads": [list(h) for h in heads],
                    "tau_hash": pair["tau_hash"], "cfg": cfg,
                }
                key = f'{case["source_id"]}-{arm}-{followup_index}-' + stable_hash(payload)[:16]
                if result_memory["valid"]:
                    actor_result = checkpoint_item(
                        "factorial", key, payload,
                        lambda: run_downstream_actor(backend, copy.deepcopy(followup), copy.deepcopy(memory),
                                          heads=heads, cfg=actor_cfg, initial_env=copy.deepcopy(initial)),
                    )
                    success = bool(actor_result.get("task_success", actor_result.get("success", False)))
                    failure_reason = None
                else:
                    # Parsing failures are full-pipeline failures, not dropped samples.
                    actor_result, success, failure_reason = None, False, "invalid_writer_memory"
                if stable_hash(memory) != before:
                    raise RuntimeError("Factorial evaluation modified its read-only memory")
                rows.append({
                    "source_id": case["source_id"], "split": case["split"],
                    "followup_id": followup["followup_id"], "followup_index": followup_index,
                    "arm": arm, "evaluation_layer": cfg.get("evaluation_layer", "end_to_end"),
                    "writer_intervention": writer_intervention,
                    "actor_intervention": actor_intervention,
                    "success": success, "task_success": success,
                    "binding_correct": bool(actor_result and actor_result.get("binding_correct", False)),
                    "binding_distractor": bool(actor_result and actor_result.get("binding_distractor", False)),
                    "distractor_action": bool(actor_result and actor_result.get("distractor_action", False)),
                    "format_failure": bool(not result_memory["valid"] or
                                           (actor_result and actor_result.get("format_failure", False))),
                    "memory_valid": result_memory["valid"], "memory_hash": before,
                    "environment_hash": stable_hash(initial), "tau_hash": pair["tau_hash"],
                    "failure_reason": failure_reason, "actor_result": actor_result,
                })
    return rows


# %% Local factual-value swaps: conditional diagnostic, all other fields fixed

def local_entry_variants(case, pair):
    """Swap only target.value. Preserve order, provenance, and all other bytes.

    Eligibility is conditioned on the observed M0/M1 values; report its rate.
    This diagnostic must not replace the unconditional whole-memory test.
    """
    natural = {label: copy.deepcopy(pair[label]["memory"]) for label in ("M0", "M1")}
    output = {"source_id": case["source_id"], "eligible": False,
              "reason": None, "variants": natural,
              "intervention_fields": ["value"], "provenance_held_fixed": True}
    if not all(pair[label]["valid"] for label in ("M0", "M1")):
        output["reason"] = "invalid_memory"
        return output
    indices = {}
    for label, memory in natural.items():
        matches = [i for i, r in enumerate(memory)
                   if r.get("kind") == "preference" and r.get("entity") == case["target_name"]
                   and r.get("attribute") == "delivery_city"]
        if len(matches) != 1:
            output["reason"] = f"{label}_target_missing_or_duplicate"
            return output
        indices[label] = matches[0]
    condition = pair.get("condition", case.get("_condition", "distractor"))
    base_value = natural["M0"][indices["M0"]]["value"]
    donor_value = natural["M1"][indices["M1"]]["value"]
    if base_value != distractor_city(case, condition) or base_value == expected_city(case, condition):
        output["reason"] = "M0_is_not_a_distractor_error"
        return output
    if donor_value != expected_city(case, condition):
        output["reason"] = "M1_is_not_a_correct_donor"
        return output
    corrected, corrupted = copy.deepcopy(natural["M0"]), copy.deepcopy(natural["M1"])
    corrected[indices["M0"]]["value"] = donor_value
    corrupted[indices["M1"]]["value"] = base_value
    output["variants"].update(M0_corrected=corrected, M1_corrupted=corrupted)
    output.update(eligible=True, reason=None, target_indices=indices,
                  token_check_required=True)
    return output


def evaluate_local_variants(backend, case, pair, read_heads=(), cfg=None):
    cfg = cfg or {}
    variants = local_entry_variants(case, pair)
    if not variants["eligible"]:
        return {**variants, "rows": []}
    rows = []
    for index in range(int(cfg.get("followup_tasks", 3))):
        followup, initial = _followup_case_and_environment(case, index)
        followup["_condition"] = pair["condition"]
        actor_cfg = {**cfg, "condition": pair["condition"]}
        for name, memory in variants["variants"].items():
            payload = {"case": followup, "initial_env": initial, "memory": memory,
                       "heads": [list(h) for h in read_heads], "cfg": cfg}
            key = f'{case["source_id"]}-{name}-{index}-' + stable_hash(payload)[:16]
            result = checkpoint_item(
                "local_swaps", key, payload,
                lambda: run_actor(backend, copy.deepcopy(followup), copy.deepcopy(memory),
                                  heads=read_heads, cfg=actor_cfg, initial_env=copy.deepcopy(initial)),
            )
            rows.append({"source_id": case["source_id"], "variant": name,
                         "followup_id": followup["followup_id"], "followup_index": index,
                         "task_success": bool(result.get("task_success", result.get("success", False))),
                         "actor_result": result, "environment_hash": stable_hash(initial)})
    return {**variants, "rows": rows}


# %% Optional online updating: separate from the read-only factorial endpoint

def run_online(backend, case, initial_memory, read_heads=(), write_heads=(), horizon=3, cfg=None):
    """Sequential Actor -> Writer updates with no injected ground truth.

    Orders are independent episodes; memory carries across them. Only the
    previous stored records plus actually observed actions/feedback enter the
    next Writer prompt. Oracle preference events are used for scoring only.
    """
    cfg = cfg or {}
    if horizon not in (1, 3, 5):
        raise ValueError("The prespecified exploratory horizons are 1, 3, and 5")
    error = _memory_schema_error(initial_memory)
    if error:
        return {"source_id": case["source_id"], "horizon": horizon, "steps": [],
                "failed": True, "failure_reason": "invalid_initial_memory: " + error}
    condition = case.get("_condition", "distractor")
    actor_cfg = {**cfg, "condition": condition}
    evaluator_reference = frozen_history(case, condition)
    current = copy.deepcopy(initial_memory)
    steps = []
    for index in range(horizon):
        followup, initial = _followup_case_and_environment(case, index)
        input_memory = copy.deepcopy(current)
        actor_result = run_actor(backend, followup, copy.deepcopy(current), heads=read_heads,
                                 cfg=actor_cfg, initial_env=copy.deepcopy(initial))
        # Full frozen_history would reintroduce truth. Include only real trace
        # events after filtering out the original preference/metadata records.
        actual_trace = [event for event in frozen_history(followup, condition, actor_result)
                        if event.get("kind") in ("observed_action", "tool_observation")]
        history = [{"kind": "previous_memory", "record": copy.deepcopy(record),
                    "sequence": j} for j, record in enumerate(current)]
        for event in actual_trace:
            event = copy.deepcopy(event)
            event["sequence"] = len(history)
            history.append(event)
        writer_result = run_writer(backend, followup, history, heads=write_heads, cfg=cfg)
        external_scores = score_memory(followup, evaluator_reference,
                                       writer_result["memory"], writer_result["valid"])
        steps.append({"source_id": case["source_id"], "step": index + 1,
                      "input_memory": input_memory, "actor_result": actor_result,
                      "writer_history": history, "writer_result": writer_result,
                      "external_reference_metrics": external_scores})
        if not writer_result["valid"]:
            return {"source_id": case["source_id"], "horizon": horizon, "steps": steps,
                    "failed": True, "failure_reason": "invalid_writer_memory", "completed_steps": index + 1}
        current = copy.deepcopy(writer_result["memory"])
    return {"source_id": case["source_id"], "horizon": horizon, "steps": steps,
            "failed": False, "failure_reason": None, "completed_steps": horizon,
            "final_memory": current}

In [ ]:
# %% Writer evaluation stages: actual fixed experiences, unconditional source sets

def collect_fixed_histories(backend, cases, conditions, cfg):
    """Run an untreated Actor once per case/condition, then freeze its experience.

    Failed, malformed, and successful episodes are all retained. Every Writer
    arm later receives exactly this recorded history; no Actor is rerun under
    Writer interventions. Checkpoints retain the full Actor result separately.
    """
    histories = {condition: {} for condition in conditions}
    for condition in conditions:
        for original_case in cases:
            case = copy.deepcopy(original_case)
            case["_condition"] = condition
            actor_cfg = {**cfg, "condition": condition}
            memory = memory_for_condition(case, condition)
            payload = {"case": case, "memory": memory, "heads": [],
                       "cfg": actor_cfg, "purpose": "fixed_writer_history"}
            key = f'{case["source_id"]}-{condition}-' + stable_hash(payload)[:16]
            actor_result = checkpoint_item(
                "fixed_history_actors", key, payload,
                lambda: run_actor(backend, copy.deepcopy(case), copy.deepcopy(memory),
                                  heads=(), cfg=actor_cfg,
                                  initial_env=copy.deepcopy(case["initial_env"])),
            )
            histories[condition][case["source_id"]] = frozen_history(case, condition, actor_result)
    return histories


def evaluate_writer(backend, cases, histories, head_sets, cfg,
                    conditions=("correct_only", "distractor", "matched_neutral",
                                "distractor_value_swap", "useful_memory", "legitimate_update")):
    """All planned source cases x conditions x named interventions.

    head_sets is a dict {label: [(layer, head), ...]}; include an untreated
    baseline explicitly. Format failures stay in the output denominator.
    Metrics are flattened for source-level paired summaries, while raw text,
    parse errors, tokens, and full memory remain in writer_result.
    """
    if not isinstance(head_sets, dict) or not head_sets:
        raise ValueError("head_sets must be a nonempty dict of named interventions")
    rows = []
    for condition in conditions:
        if condition not in histories:
            raise KeyError(f"No frozen histories for condition {condition}")
        for original_case in cases:
            case = copy.deepcopy(original_case)
            case["_condition"] = condition
            source_id = case["source_id"]
            if source_id not in histories[condition]:
                raise KeyError(f"Missing frozen history: {condition}/{source_id}")
            history = copy.deepcopy(histories[condition][source_id])
            history_hash = stable_hash(history)
            writer_cfg = {**cfg, "condition": condition}
            for label, heads in head_sets.items():
                payload = {"case": case, "history": history,
                           "heads": [list(h) for h in heads], "cfg": writer_cfg,
                           "prompt": writer_prompt(case, history)}
                key = f"{source_id}-{condition}-{label}-" + stable_hash(payload)[:16]
                result = checkpoint_item(
                    "writer_evaluation", key, payload,
                    lambda: run_writer(backend, copy.deepcopy(case), copy.deepcopy(history),
                                       heads=heads, cfg=writer_cfg),
                )
                if result["history_hash"] != history_hash:
                    raise RuntimeError("Writer evaluation did not use the frozen history")
                rows.append({
                    "source_id": source_id, "split": case["split"],
                    "condition": condition, "intervention": str(label),
                    **result["metrics"], "valid": result["valid"],
                    "token_count": result.get("token_count", 0),
                    "seconds": result.get("seconds", 0.0),
                    "history_hash": history_hash, "writer_result": result,
                })
            if stable_hash(history) != history_hash:
                raise RuntimeError("Writer evaluation modified frozen history")
    return rows


def evaluate_memory_pipeline(backend, cases, histories, read_heads, write_heads, cfg):
    """Unconditional whole-memory endpoint plus a labeled local diagnostic.

    Returns pairs as a source-ID dictionary, factorial as flat rows over every
    case, and local as per-source dictionaries including ineligible cases.
    Local eligible-only results never filter the primary factorial sample.
    """
    condition = "distractor"
    if condition not in histories:
        raise KeyError("Memory pipeline requires distractor frozen histories")
    pairs, factorial, controlled_factorial, local = {}, [], [], []
    for original_case in cases:
        case = copy.deepcopy(original_case)
        case["_condition"] = condition
        source_id = case["source_id"]
        if source_id not in histories[condition]:
            raise KeyError(f"Missing distractor history for {source_id}")
        history = copy.deepcopy(histories[condition][source_id])
        pair_cfg = {**cfg, "condition": condition}
        pair = build_memory_pair(backend, case, history, write_heads, pair_cfg)
        pairs[source_id] = pair
        factorial.extend(evaluate_factorial(backend, case, pair, read_heads, dict(pair_cfg, evaluation_layer="end_to_end")))
        controlled_factorial.extend(evaluate_factorial(backend, case, pair, read_heads,
                                                        dict(pair_cfg, evaluation_layer="controlled_decision")))
        # Untreated downstream Actor isolates the local memory edit itself.
        local.append(evaluate_local_variants(backend, case, pair, read_heads=(), cfg=pair_cfg))
    return {"pairs": pairs, "factorial": factorial, "controlled_factorial": controlled_factorial, "local": local}

###  Assessment and statistics 

 Main analysis of retention of all planning cases;  token  Unqualified only skips  margin。 Random matching each layer  head  Count and select  heads  Do not overlap. If you cannot construct this random comparison, the  K  Do not enter the development pool. 

bootstrap  The independent unit is...  source episode， Well start with the same thing.  source  Inner aggregation of follow-up tasks and matching; cannot be used  3   followups  Search or more  seed  As a stand-alone sample. Intersection is descriptive.  percentile bootstrap； Formal inference is that sufficient samples are required to match presets. 

In [ ]:
# Read-side evaluation and source-clustered descriptive/paired uncertainty.
import collections
import math
import random
import numpy as np


def _scalar_float(value):
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()
    if hasattr(value, "item"):
        value = value.item()
    result = float(value)
    if not math.isfinite(result):
        raise ValueError("Nonfinite measurement; preserve the failing case and diagnose it")
    return result


def _evaluate_read_fresh(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    """Execute every planned source × condition × intervention, with checkpoints.

    head_sets maps human-readable labels to (layer, head) pairs. Actual full
    behavior is never filtered by token eligibility or initial attack success.
    Fixed-prefix margin is a secondary, explicitly eligible-only diagnostic.
    cfg is expected to contain model/revision/dtype/decoding identity so the
    checkpoint payload changes when the run configuration changes.
    """
    if not hasattr(head_sets, "items") or not head_sets:
        raise ValueError("head_sets must be a nonempty mapping label -> head pairs")
    rows = []
    for case in cases:
        for condition in conditions:
            memory = memory_for_condition(case, condition)
            snapshot = actor_snapshot(case, condition)
            gold_id = backend.one_token_id(snapshot["prompt"], snapshot["gold"])
            distractor_id = backend.one_token_id(snapshot["prompt"], snapshot["distractor"])
            eligible = gold_id is not None and distractor_id is not None and gold_id != distractor_id
            for intervention, selected in head_sets.items():
                heads = [list(pair) for pair in selected]
                condition_cfg = dict(cfg, condition=condition)
                payload = {"case": case, "memory": memory, "heads": heads,
                           "cfg": condition_cfg, "intervention": intervention,
                           "snapshot": snapshot, "version": "read_eval_v1"}

                def compute(case=case, memory=memory, heads=heads,
                            condition_cfg=condition_cfg, snapshot=snapshot,
                            intervention=intervention, eligible=eligible,
                            gold_id=gold_id, distractor_id=distractor_id):
                    result = run_actor(backend, case, memory, heads=heads, cfg=condition_cfg)
                    result.update({"intervention": intervention, "heads": heads,
                                   "single_token_eligible": eligible,
                                   "gold_token_id": gold_id, "distractor_token_id": distractor_id,
                                   "snapshot_margin": _scalar_float(backend.margin(snapshot, heads=heads)) if eligible and cfg.get("compute_snapshot_margin", True) else None,
                                   "snapshot_origin": snapshot["snapshot_origin"]})
                    return result

                key = f"{case['source_id']}-{condition}-{intervention}"
                rows.append(checkpoint_item("read_eval", key, payload, compute))
    return rows


def same_layer_random_heads(selected, n_layers, n_heads, seed):
    """Match each selected layer's head count, excluding ALL selected heads.

    Raise if the requested disjoint control cannot be formed. Never silently
    relax exclusion or layer matching. Repeat with distinct recorded seeds.
    """
    selected = [tuple(pair) for pair in selected]
    if len(set(selected)) != len(selected):
        raise ValueError("selected contains duplicate heads")
    if n_layers < 1 or n_heads < 1:
        raise ValueError("n_layers and n_heads must be positive")
    for layer, head in selected:
        if type(layer) is not int or type(head) is not int or not (0 <= layer < n_layers and 0 <= head < n_heads):
            raise ValueError(f"Invalid selected head {(layer, head)}")
    by_layer = collections.defaultdict(set)
    for layer, head in selected:
        by_layer[layer].add(head)
    rng = random.Random(int(seed))
    control = []
    for layer in sorted(by_layer):
        available = [head for head in range(n_heads) if head not in by_layer[layer]]
        count = len(by_layer[layer])
        if len(available) < count:
            raise ValueError(f"Layer {layer}: {count} selected heads but only {len(available)} nonselected heads; no disjoint layer-matched control")
        control.extend((layer, head) for head in rng.sample(available, count))
    return sorted(control)


def _evaluate_controlled_read_fresh(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    """All-source complete-action evaluation. Never uses the token probe prefix."""
    if not head_sets:
        raise ValueError("A nonempty head-set mapping is required")
    rows = []
    for case in cases:
        for condition in conditions:
            memory = memory_for_condition(case, condition)
            condition_cfg = dict(cfg, condition=condition, evaluation_layer="controlled_decision")
            for label, heads in head_sets.items():
                payload = {"case": case, "memory": memory, "heads": [list(h) for h in heads],
                           "cfg": condition_cfg, "intervention": label,
                           "version": "controlled_complete_action_v1"}
                key = f"{case['source_id']}-{condition}-{label}"
                result = checkpoint_item("controlled_read", key, payload,
                    lambda: run_controlled_actor(backend, case, memory, heads=heads, cfg=condition_cfg))
                rows.append({**result, "intervention": label, "heads": [list(h) for h in heads]})
    return rows


def layered_read_effects(end_to_end, controlled, arm="H_R", n_boot=2000):
    """Do not pool controlled decisions and autonomous trajectories."""
    effects = []
    for layer, rows, metrics in (
        ("end_to_end", end_to_end, ("task_success", "strict_task_success", "binding_correct", "binding_distractor")),
        ("controlled_decision", controlled, ("decision_action_success", "binding_correct", "binding_distractor")),
    ):
        for condition in sorted({r["condition"] for r in rows}):
            for metric in metrics:
                effects.append(dict(evaluation_layer=layer, condition=condition,
                    **paired_effect(rows, "intervention", "none", arm, metric=metric,
                                    filters={"condition": condition}, n_boot=n_boot)))
    return effects


def actor_diagnostics(rows):
    """Descriptive process failures; never remove episodes or alter scoring."""
    diagnostics = []
    for row in rows:
        errors = [event["observation"].get("error") for event in row["events"]
                  if isinstance(event.get("observation"), dict)
                  and not event["observation"].get("ok", False)]
        reserved = sum(event.get("action", {}).get("tool") == "reserve_item"
                       and bool((event.get("observation") or {}).get("ok"))
                       for event in row["events"] if isinstance(event.get("action"), dict))
        diagnostics.append({
            key: row[key] for key in ("source_id", "condition", "intervention", "template_id",
                                     "preferred_available", "position_order", "task_success",
                                     "format_failure", "terminal", "action_count", "budget_exhausted")
        })
        diagnostics[-1].update(tool_error_count=len(errors),
                               out_of_stock_count=errors.count("out_of_stock"),
                               invalid_reservation_count=errors.count("invalid_reservation"),
                               successful_reservations=reserved,
                               accepted_order_count=row["accepted_order_count"],
                               gold_city=row["gold_city"], gold_item=row["gold_item"])
        diagnostics[-1].update({key: row.get(key) for key in (
            "binding_category", "binding_correct", "binding_distractor", "order_attempted",
            "subgoal_progress", "prerequisite_path_valid", "strict_task_success",
            "first_tool_error_step", "first_tool_error")})
    return diagnostics


def _passes_filters(row, filters):
    if filters is None:
        return True
    if callable(filters):
        return bool(filters(row))
    for key, expected in filters.items():
        actual = row.get(key)
        if isinstance(expected, (list, tuple, set, frozenset)):
            if actual not in expected:
                return False
        elif actual != expected:
            return False
    return True


def _get_metric(row, metric):
    if metric in row:
        return row[metric]
    return row.get("metrics", {}).get(metric)


def _source_bootstrap(values, seed, n_boot):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("Bootstrap requires a nonempty finite 1-D source-level array")
    if not isinstance(n_boot, int) or n_boot < 1:
        raise ValueError("n_boot must be a positive integer")
    rng = np.random.default_rng(int(seed))
    # Chunk so large source counts do not allocate n_boot × n_sources at once.
    estimates = []
    for start in range(0, n_boot, 256):
        count = min(256, n_boot - start)
        indices = rng.integers(0, len(values), size=(count, len(values)))
        estimates.extend(values[indices].mean(axis=1).tolist())
    lower, upper = np.quantile(np.asarray(estimates), [0.025, 0.975])
    return {"estimate": float(values.mean()), "ci_low": float(lower), "ci_high": float(upper),
            "n_sources": int(len(values)), "n_boot": n_boot,
            "ci_method": "percentile_source_bootstrap_95pct",
            "small_sample_warning": bool(len(values) < 20)}


def paired_effect(rows, arm_col, arm0, arm1, metric="task_success", filters=None, seed=11, n_boot=2000):
    """Mean(arm1 − arm0), bootstrapping source cases after within-source means.

    Repeated future tasks, fields, or seeds never become independent units here.
    A missing arm/source/metric is an error, not silent complete-case filtering.
    Equal repeat counts are required within each paired source; if followup_id
    exists, its set must also match. Use filters to select the planned contrast.
    """
    if arm0 == arm1:
        raise ValueError("arm0 and arm1 must differ")
    grouped = {arm0: collections.defaultdict(list), arm1: collections.defaultdict(list)}
    for row in rows:
        arm = row.get(arm_col)
        if arm not in grouped or not _passes_filters(row, filters):
            continue
        if "source_id" not in row:
            raise ValueError("Each observation requires source_id")
        value = _get_metric(row, metric)
        if value is None:
            raise ValueError(f"Missing {metric} for {row['source_id']} / {arm}")
        grouped[arm][row["source_id"]].append((row, _scalar_float(value)))
    sources0, sources1 = set(grouped[arm0]), set(grouped[arm1])
    if not sources0 or sources0 != sources1:
        raise ValueError(f"Unpaired or empty sources: arm0={len(sources0)}, arm1={len(sources1)}, missing_in_arm1={sorted(sources0-sources1)[:5]}, missing_in_arm0={sorted(sources1-sources0)[:5]}")
    differences, means0, means1 = [], [], []
    count_rows = 0
    for source in sorted(sources0):
        entries0, entries1 = grouped[arm0][source], grouped[arm1][source]
        if len(entries0) != len(entries1):
            raise ValueError(f"Unmatched repeat counts for source {source}: {len(entries0)} vs {len(entries1)}")
        if any("followup_id" in row for row, _ in entries0 + entries1):
            ids0 = collections.Counter(row.get("followup_id") for row, _ in entries0)
            ids1 = collections.Counter(row.get("followup_id") for row, _ in entries1)
            if ids0 != ids1:
                raise ValueError(f"Unpaired followup_id values for source {source}")
        mean0 = sum(v for _, v in entries0) / len(entries0)
        mean1 = sum(v for _, v in entries1) / len(entries1)
        means0.append(mean0)
        means1.append(mean1)
        differences.append(mean1 - mean0)
        count_rows += len(entries0) + len(entries1)
    result = _source_bootstrap(differences, seed, n_boot)
    result.update({"arm_col": arm_col, "arm0": arm0, "arm1": arm1, "metric": metric,
                   "direction": "arm1_minus_arm0", "n_rows": count_rows,
                   "arm0_mean": float(np.mean(means0)), "arm1_mean": float(np.mean(means1)),
                   "n_unpaired_sources": 0, "seed": int(seed)})
    return result


def bootstrap_summary(rows, group_cols, metrics, n_boot=2000):
    """Descriptive group means with a source-clustered 95% percentile interval.

    Returns one row per group × metric. Missing optional metrics (e.g. ineligible
    snapshot margins) are counted explicitly. For confirmatory arm comparisons,
    use paired_effect; these separate intervals do not estimate a difference.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    if isinstance(metrics, str):
        metrics = [metrics]
    groups = collections.defaultdict(list)
    for row in rows:
        if "source_id" not in row:
            raise ValueError("Each observation requires source_id")
        key = tuple(row.get(col) for col in group_cols)
        groups[key].append(row)
    summaries = []
    for group_index, (key, members) in enumerate(sorted(groups.items(), key=lambda item: repr(item[0]))):
        for metric_index, metric in enumerate(metrics):
            per_source = collections.defaultdict(list)
            n_missing = 0
            for row in members:
                value = _get_metric(row, metric)
                if value is None:
                    n_missing += 1
                else:
                    per_source[row["source_id"]].append(_scalar_float(value))
            summary = dict(zip(group_cols, key))
            summary.update({"metric": metric, "n_rows": len(members),
                            "n_missing_rows": n_missing, "n_valid_rows": len(members) - n_missing,
                            "n_all_sources": len({row["source_id"] for row in members})})
            values = [sum(values) / len(values) for _, values in sorted(per_source.items())]
            if values:
                summary.update(_source_bootstrap(values, 11 + 1009 * group_index + metric_index, n_boot))
            else:
                summary.update({"estimate": None, "ci_low": None, "ci_high": None,
                                "n_sources": 0, "n_boot": n_boot, "ci_method": "no_valid_values"})
            summaries.append(summary)
    return summaries

# Standalone evaluators remain available before the explicit reuse wrappers load.
evaluate_read = _evaluate_read_fresh
evaluate_controlled_read = _evaluate_controlled_read_fresh

In [ ]:
# Run gates, final protocol, and serializable result exports.
import inspect
import csv


def _implementation_source(obj):
    # Interactive notebook classes lack a normal module file. Their method
    # functions do retain IPython linecache source and are stable on reruns.
    if inspect.isclass(obj):
        return {"name": obj.__name__, "doc": obj.__doc__,
                "bases": [base.__qualname__ for base in obj.__bases__],
                "methods": {name: inspect.getsource(method) for name, method in vars(obj).items()
                            if inspect.isfunction(method)}}
    return inspect.getsource(obj)


def live_implementation_hash():
    return stable_hash({"definitions": {name: _implementation_source(globals()[name]) for name in IMPLEMENTATION_NAMES},
                        "task_templates": TASK_TEMPLATES, "conditions": CONDITIONS})


def verify_live_run():
    current_hash = live_implementation_hash()
    current_run = stable_hash({"config": CFG, "model": MODEL_IDENTITY, "implementation": current_hash})
    if current_hash != IMPLEMENTATION_HASH or current_run != RUN_FINGERPRINT:
        raise RuntimeError("Live definitions/constants/config differ from the run manifest. Start a new run rather than mixing checkpoints.")


def capability_gate(rows, cfg):
    clean = [r for r in rows if r["condition"] == "correct_only" and r["intervention"] == "none"]
    if not clean:
        raise ValueError("Capability pilot requires untreated correct_only cases")
    success = float(np.mean([r["task_success"] for r in clean]))
    format_failure = float(np.mean([r["format_failure"] for r in clean]))
    return {"eligible": success >= cfg["min_pilot_success"] and format_failure <= cfg["max_pilot_format_failure"],
            "n_sources": len(clean), "clean_success": success, "format_failure": format_failure,
            "min_success": cfg["min_pilot_success"], "max_format_failure": cfg["max_pilot_format_failure"]}


def natural_writer_gate(rows, cfg):
    clean = [r for r in rows if r["condition"] == "correct_only" and r["intervention"] == "none"]
    attacks = [r for r in rows if r["condition"] == "distractor" and r["intervention"] == "none"]
    if not clean or not attacks:
        raise ValueError("Writer pilot requires paired clean/distractor conditions")
    errors = sum(bool(r["target_distractor_error"]) for r in attacks)
    valid = float(np.mean([not r["format_failure"] for r in clean]))
    correct = float(np.mean([r["target_correct"] for r in clean]))
    return {"eligible": errors >= cfg["min_natural_writer_errors"] and valid >= cfg["min_writer_valid"] and correct >= cfg.get("min_writer_clean_correct", cfg["min_pilot_success"]),
            "natural_distractor_errors": errors, "n_sources": len(attacks),
            "clean_valid_fraction": valid, "clean_target_correct": correct,
            "min_errors": cfg["min_natural_writer_errors"],
            "interpretation": "feasibility_gate_only_not_a_significance_test"}


def controlled_capability_gate(rows, cfg):
    if any(r.get("evaluation_layer") != "controlled_decision" for r in rows):
        raise ValueError("Controlled capability requires controlled-decision rows")
    result = capability_gate(rows, dict(cfg, min_pilot_success=cfg["min_controlled_success"]))
    return dict(result, evaluation_layer="controlled_decision", metric="decision_action_success",
                interpretation="project_feasibility_threshold_not_benchmark_standard")


def _atomic_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = sorted({key for row in rows for key in row})
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: canonical_json(value) if isinstance(value, (list, dict, tuple)) else value
                             for key, value in row.items()})
    os.replace(temporary, path)


def export_rows(name, rows, group_cols, metrics):
    atomic_json(RUN_DIR / "results" / f"{name}-raw.json", rows)
    summary = bootstrap_summary(rows, group_cols, metrics, n_boot=CFG["bootstrap_reps"])
    _atomic_csv(RUN_DIR / "results" / f"{name}-summary.csv", summary)
    atomic_json(RUN_DIR / "results" / f"{name}-summary.json", summary)
    return summary


def export_layered_read(name, end_to_end, controlled):
    end_metrics = ["task_success", "strict_task_success", "binding_correct", "binding_distractor",
                   "order_attempted", "format_failure", "prerequisite_path_valid", "subgoal_progress", "snapshot_margin"]
    decision_metrics = ["decision_action_success", "binding_correct", "binding_distractor",
                        "order_attempted", "tool_execution_valid", "format_failure"]
    output = []
    for layer, rows, metrics in (("end_to_end", end_to_end, end_metrics),
                                 ("controlled_decision", controlled, decision_metrics)):
        output.extend(export_rows(name + "-" + layer, rows, ["evaluation_layer", "condition", "intervention"], metrics))
        # Crossed factors remain explicit. Do not discard failures within a stratum.
        stratified = bootstrap_summary(rows, ["evaluation_layer", "condition", "intervention",
                          "template_id", "preferred_available", "position_order"], metrics,
                          n_boot=CFG["bootstrap_reps"])
        _atomic_csv(RUN_DIR / "results" / f"{name}-{layer}-strata.csv", stratified)
        categories = []
        for group in sorted({(r["condition"], r["intervention"]) for r in rows}):
            members = [r for r in rows if (r["condition"], r["intervention"]) == group]
            counts = collections.Counter(r["binding_category"] for r in members)
            for category in ("no_order_attempt", "invalid_binding_fields", "wrong_customer",
                             "correct_binding", "distractor_binding", "other_city"):
                counts.setdefault(category, 0)
            categories.extend({"evaluation_layer": layer, "condition": group[0], "intervention": group[1],
                               "category": key, "count": value, "denominator": len(members),
                               "fraction": value / len(members)} for key, value in sorted(counts.items()))
        _atomic_csv(RUN_DIR / "results" / f"{name}-{layer}-binding-categories.csv", categories)
    return output


def factorial_contrasts(rows):
    if not rows:
        return []
    if len({r.get("evaluation_layer", "end_to_end") for r in rows}) != 1:
        raise ValueError("Filter factorial rows by evaluation_layer before computing contrasts")
    definitions = [("writer_effect_actor_untreated", "Y00", "Y10"),
                   ("reader_effect_baseline_memory", "Y00", "Y01"),
                   ("combined_effect", "Y00", "Y11"),
                   ("reader_effect_intervened_memory", "Y10", "Y11")]
    effects = [dict(contrast=name, **paired_effect(rows, "arm", a, b, n_boot=CFG["bootstrap_reps"]))
               for name, a, b in definitions]
    # Interaction = (Y11-Y10) - (Y01-Y00), computed within the same source.
    by_source = {}
    for row in rows:
        by_source.setdefault(row["source_id"], {}).setdefault(row["arm"], []).append(float(row["task_success"]))
    interaction = []
    for source, arms in sorted(by_source.items()):
        if set(arms) != {"Y00", "Y01", "Y10", "Y11"}:
            raise ValueError(f"Incomplete factorial for {source}")
        means = {key: np.mean(values) for key, values in arms.items()}
        interaction.append(means["Y11"] - means["Y10"] - means["Y01"] + means["Y00"])
    effects.append(dict(contrast="writer_reader_interaction", **_source_bootstrap(interaction, 11, CFG["bootstrap_reps"])))
    return effects


def freeze_protocol(read_selection, write_selection, read_controls, write_controls, data_hashes, writer_gate):
    verify_live_run()
    if QUICK_MODE:
        raise RuntimeError("Final test is disabled in QUICK_MODE. Use a new full-run directory and finish development first.")
    protocol = {"schema": "agent_entrainment_behavioral_v2", "run_fingerprint": RUN_FINGERPRINT,
                "implementation_hash": IMPLEMENTATION_HASH, "model": MODEL_IDENTITY,
                "config": CFG, "data_hashes": data_hashes,
                "primary_search_seed": CFG["search_seeds"][0],
                "read_heads": read_selection["heads"] if read_selection else [],
                "read_selection_method": read_selection.get("selection_method") if read_selection else None,
                "write_selection_method": write_selection.get("selection_method") if write_selection else None,
                "read_phenomenon_gate": read_selection.get("gate") if read_selection else None,
                "write_heads": write_selection["heads"] if write_selection else [],
                "read_controls": read_controls, "write_controls": write_controls,
                "writer_branch": bool(writer_gate and writer_gate["eligible"] and write_selection and write_selection["eligible"]),
                "writer_gate": writer_gate,
                "primary_endpoints": {
                    "read": {"layer": "controlled_decision", "condition": "distractor", "metric": "binding_correct", "contrast": "H_R minus none", "population": "all test sources, including malformed/no-order outputs"},
                    "write": {"condition": "distractor", "metric": "target_correct", "contrast": "H_W minus none", "population": "all test sources; only if writer branch prespecified"},
                    "propagation": {"layer": "controlled_decision", "condition": "distractor", "metric": "task_success", "contrast": "Y10 minus Y00", "population": "all test sources; only if both head sets available"}},
                "secondary_endpoints": ["end_to_end.task_success", "end_to_end.strict_task_success",
                    "controlled_decision.decision_action_success", "binding_distractor", "subgoal_progress",
                    "format_failure", "omission", "useful_memory", "legitimate_update", "random_head_controls",
                    "end_to_end.factorial", "Y11 minus Y00", "factorial_interaction"],
                "interpretation": "controlled effects establish behavior at a fixed decision state; autonomous gains require end_to_end evidence",
                "literature": CFG.get("evaluation_references", []),
                "uncertainty_unit": "source_episode", "decoding": "greedy_complete_json",
                "test_rule": "no retuning or case exclusion after inspecting test outcomes"}
    # Deep JSON snapshot: no aliases to mutable CFG/heads and no tuple/list
    # mismatch when the same frozen protocol is loaded after a disconnect.
    protocol = json.loads(canonical_json(protocol))
    path = RUN_DIR / "frozen_protocol.json"
    if path.exists() and json.loads(path.read_text()) != protocol:
        raise RuntimeError("Frozen protocol differs from current settings/heads. Do not overwrite a used test protocol.")
    atomic_json(path, protocol)
    return protocol


def verify_protocol(protocol, data):
    verify_live_run()
    if protocol["run_fingerprint"] != RUN_FINGERPRINT or protocol["implementation_hash"] != IMPLEMENTATION_HASH:
        raise RuntimeError("Protocol/model/config/code mismatch")
    if protocol["data_hashes"] != {split: stable_hash(cases) for split, cases in data.items()}:
        raise RuntimeError("Data changed after protocol freeze")


def runtime_smoke(backend):
    """Small actual-model interface checks; no scientific outcome is assumed."""
    model = backend.model
    probe = backend.tokens('A short interface check: the city is')
    head = [(min(1, model.cfg.n_layers - 1), 0)]
    original_kind = backend.hook_kind
    try:
        with torch.no_grad():
            model.set_use_attn_result(False)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="z")):
                z_logits = model(probe, return_type="logits")[0, -1].float()
            model.set_use_attn_result(True)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="result")):
                result_logits = model(probe, return_type="logits")[0, -1].float()
        max_difference = float((z_logits - result_logits).abs().max())
        if not torch.allclose(z_logits, result_logits, atol=0.25, rtol=0.02):
            raise RuntimeError(f"z/result ablation disagreement (max={max_difference}); inspect precision before search")
        model.set_use_attn_result(original_kind == "result")
        with torch.enable_grad():
            mask_logits = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=backend.device, requires_grad=True)
            drop = torch.sigmoid(mask_logits)
            with model.hooks(fwd_hooks=ablation_hooks(model, drop_mask=drop, hook_kind=original_kind)):
                score = model(probe, return_type="logits")[0, -1, :16].float().sum()
            score.backward()
            gradient_norm = float(mask_logits.grad.norm())
            if not math.isfinite(gradient_norm) or gradient_norm == 0:
                raise RuntimeError("Mask gradient is zero or nonfinite")
        # Different cache paths may expose tiny dtype differences. A mismatch
        # is a failed software check, never evidence of an intervention effect.
        generation_prompt = 'Complete one JSON object.\n{"check":'
        cache_agreement = None
        if backend.cfg.get("use_cache", True):
            cached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=True)
            uncached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
            cache_agreement = cached["text"] == uncached["text"]
            if not cache_agreement:
                raise RuntimeError("Cached/uncached greedy text differs. Inspect precision or start an explicitly uncached run (use_cache=False); do not silently continue.")
        else:
            backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
        return {"passed": True, "z_result_max_abs_difference": max_difference,
                "mask_gradient_norm": gradient_norm, "cache_text_agreement": cache_agreement,
                "cache_test_skipped": not backend.cfg.get("use_cache", True),
                "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None}
    finally:
        backend.hook_kind = original_kind
        model.set_use_attn_result(original_kind == "result")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
"""Explicit, read-only imports from the completed v1 development run.

Old records are inputs with provenance, never new inference or copied checkpoints.
"""
import copy
import hashlib
import json
import re
from pathlib import Path
from tqdm.auto import tqdm


def _read_hashed_json(path, provenance):
    path = Path(path)
    content = path.read_bytes()
    provenance[str(path)] = hashlib.sha256(content).hexdigest()
    return json.loads(content)


def _baseline_comparable_config(cfg):
    keys = ('model_name', 'dtype', 'actor_prompt_version', 'actor_max_new_tokens',
            'max_actions', 'context_window', 'max_prompt_tokens', 'use_cache', 'hook_kind')
    return {key: copy.deepcopy(cfg.get(key)) for key in keys}


def _previous_model_mismatch_reasons(previous_model):
    current = globals().get('MODEL_IDENTITY')
    if current is None:
        return []
    keys = ('name', 'revision', 'dtype') + tuple(key for key in ('packages', 'python') if key in current)
    return [f'current model {key} differs' for key in keys
            if current.get(key) != previous_model.get(key)]


def _validate_previous_data(data, cfg):
    splits = ('pilot', 'train', 'dev', 'test')
    if not isinstance(data, dict) or set(data) != set(splits):
        raise ValueError('Previous data must contain exactly pilot/train/dev/test')
    seen_sources, entity_splits, template_splits = set(), {}, {}
    for split in splits:
        cases = data[split]
        if not isinstance(cases, list) or len(cases) != cfg[f'n_{split}']:
            raise ValueError(f'FULL source count differs for {split}')
        for case in cases:
            if not isinstance(case, dict) or case.get('split') != split:
                raise ValueError(f'Previous case is assigned to the wrong split: {split}')
            source = case.get('source_id')
            if not isinstance(source, str) or not source or source in seen_sources:
                raise ValueError('Previous source IDs must be nonempty and unique across all splits')
            seen_sources.add(source)
            for field in ('target_name', 'other_name'):
                entity = case.get(field)
                if not isinstance(entity, str) or not entity:
                    raise ValueError(f'Previous case has an invalid {field}')
                if entity in entity_splits and entity_splits[entity] != split:
                    raise ValueError('Previous customer identities overlap across splits')
                entity_splits[entity] = split
            template = case.get('template_id')
            if not isinstance(template, str) or not template:
                raise ValueError('Previous case has an invalid template_id')
            if template in template_splits and template_splits[template] != split:
                raise ValueError('Previous template identities overlap across splits')
            template_splits[template] = split


def _verified_previous_cases(previous, split):
    cases = previous['data'][split]
    if stable_hash(cases) != previous['metadata']['data_hashes'][split]:
        raise ValueError(f'Previous {split} data changed after verification')
    return {case['source_id']: case for case in cases}


def verified_previous_rankings(previous=None):
    """Return copied candidates only after checking the imported ranking hashes."""
    previous = globals().get('PREVIOUS_INPUTS') if previous is None else previous
    if previous is None:
        raise ValueError('No previous rankings were prepared')
    rankings = previous['rankings']
    expected = previous['metadata']['ranking_hashes']
    seeds = [str(ranking['seed']) for ranking in rankings]
    if len(seeds) != len(set(seeds)) or set(seeds) != set(expected):
        raise ValueError('Previous ranking set changed after verification')
    for ranking in rankings:
        if stable_hash(ranking) != expected[str(ranking['seed'])]:
            raise ValueError('Previous ranking changed after verification')
    return copy.deepcopy(rankings)


def prepare_previous_run(previous_dir, new_dir, cfg, package_versions, python_version):
    previous_dir, new_dir = Path(previous_dir).resolve(), Path(new_dir).resolve()
    if previous_dir == new_dir or previous_dir in new_dir.parents or new_dir in previous_dir.parents:
        raise ValueError('Old and new run directories must be disjoint siblings.')
    provenance = {}
    manifest = _read_hashed_json(previous_dir / 'manifest.json', provenance)
    data_manifest = _read_hashed_json(previous_dir / 'data_manifest.json', provenance)
    data = data_manifest['data']
    _validate_previous_data(data, cfg)
    if any(stable_hash(data[s]) != data_manifest['hashes'][s] for s in data):
        raise ValueError('Previous data manifest hash mismatch')
    for key in ('model_name', 'dtype', 'hook_kind', 'data_design', 'data_seed'):
        if manifest['config'].get(key) != cfg.get(key):
            raise ValueError(f'Cannot reuse previous rankings/data: {key} differs')
    rankings = []
    for seed in cfg['search_seeds']:
        ranking = _read_hashed_json(previous_dir / 'search' / f'read-seed{seed}.json', provenance)
        heads = ranking['ranked_heads']
        if ranking.get('seed') != seed or ranking.get('name') != 'read':
            raise ValueError('Ranking identity mismatch')
        if len(heads) != 1024 or len({tuple(h) for h in heads}) != 1024:
            raise ValueError('Expected a complete, unique 32x32 Llama-3.1-8B ranking')
        if any(len(h) != 2 or any(type(x) is not int or x < 0 or x >= 32 for x in h) for h in heads):
            raise ValueError('Invalid old head coordinates')
        rankings.append(ranking)
    baseline = {}
    old_writer = {}
    for split in ('pilot', 'dev'):
        writer_path = previous_dir / 'results' / f'{split}-writer-raw.json'
        old_writer[split] = _read_hashed_json(writer_path, provenance) if writer_path.exists() else []
        for layer in ('end_to_end', 'controlled_decision'):
            path = previous_dir / 'results' / f'{split}-read-{layer}-raw.json'
            if not path.exists():
                raise FileNotFoundError(f'Previous FULL results missing: {path}')
            rows = _read_hashed_json(path, provenance)
            index = {}
            for row in rows:
                if row['intervention'] != 'none':
                    continue
                key = (row['source_id'], row['condition'])
                if (key in index or row['split'] != split or row.get('evaluation_layer') != layer or
                        row.get('heads')):
                    raise ValueError('Duplicate or mismatched previous baseline')
                index[key] = row
            conditions = ('correct_only', 'distractor') if split == 'pilot' else CONDITIONS
            expected = {(c['source_id'], cond) for c in data[split] for cond in conditions}
            if set(index) != expected:
                raise ValueError(f'Incomplete previous {split}/{layer} baseline')
            baseline[(split, layer)] = index
    histories = {}
    for split in ('pilot', 'dev'):
        histories[split] = {cond: {} for cond in (('correct_only', 'distractor') if split == 'pilot' else CONDITIONS)}
        expected_hashes = {}
        for row in old_writer[split]:
            if row['intervention'] != 'none':
                continue
            key = (row['source_id'], row['condition'])
            if key in expected_hashes or row.get('split') != split:
                raise ValueError('Duplicate or mismatched previous Writer history')
            if row.get('writer_result', {}).get('history_hash', row['history_hash']) != row['history_hash']:
                raise ValueError('Previous Writer history hashes disagree')
            expected_hashes[key] = row['history_hash']
        for case in data[split]:
            for condition in histories[split]:
                key = (case['source_id'], condition)
                actor_row = baseline[(split, 'end_to_end')][key]
                history = frozen_history(case, condition, actor_row)
                expected = expected_hashes.get(key)
                if expected is None:
                    raise ValueError(f'No prior Writer history hash for {key}')
                if stable_hash(history) != expected:
                    # Fixed-history Actor recordings may differ from the read evaluation.
                    candidates = sorted((previous_dir / 'items' / 'fixed_history_actors').glob(
                        f"{case['source_id']}-{condition}-*.json"))
                    matches = []
                    for path in candidates:
                        record = _read_hashed_json(path, provenance)
                        candidate = frozen_history(case, condition, record['result'])
                        if stable_hash(candidate) == expected:
                            matches.append(candidate)
                    if not matches:
                        raise ValueError(f'Frozen history cannot be verified for {key}; do not regenerate it silently')
                    history = matches[0]
                histories[split][condition][case['source_id']] = history
    baseline_config = _baseline_comparable_config(manifest['config'])
    reuse_reasons = [f'{key} changed' for key, value in _baseline_comparable_config(cfg).items()
                    if value != baseline_config[key]]
    if manifest.get('implementation') != cfg['compatible_previous_implementation']:
        reuse_reasons.append('unrecognized previous Actor implementation')
    if manifest['model'].get('packages') != package_versions:
        reuse_reasons.append('package versions differ')
    if manifest['model'].get('python') != python_version:
        reuse_reasons.append('Python version differs')
    reuse_reasons.extend(_previous_model_mismatch_reasons(manifest['model']))
    metadata = {'previous_dir': str(previous_dir), 'previous_run_fingerprint': manifest['fingerprint'],
                'previous_model': copy.deepcopy(manifest['model']), 'previous_implementation': manifest['implementation'],
                'files_sha256': provenance, 'data_hashes': copy.deepcopy(data_manifest['hashes']),
                'baseline_config': baseline_config,
                'baseline_hashes': {split: {layer: {cond: {sid: stable_hash(row)
                    for (sid, condition), row in baseline[(split, layer)].items() if condition == cond}
                    for cond in histories[split]} for layer in ('end_to_end', 'controlled_decision')}
                    for split in histories},
                'ranking_hashes': {str(ranking['seed']): stable_hash(ranking) for ranking in rankings},
                'frozen_history_hashes': {split: {cond: {sid: stable_hash(h) for sid,h in cases.items()}
                                                    for cond,cases in groups.items()} for split,groups in histories.items()},
                'reuse_actor_baselines': not reuse_reasons, 'baseline_reuse_disabled_reasons': reuse_reasons,
                'ranking_role': 'fixed imported candidates; no new mask training',
                'test_outputs_read': False}
    return {'metadata': metadata, 'data': data, 'rankings': rankings,
            'baseline': baseline, 'histories': histories}


def _verified_previous_baseline(case, condition, layer, cfg):
    previous = globals().get('PREVIOUS_INPUTS')
    if not previous or not previous['metadata']['reuse_actor_baselines'] or case['split'] == 'test':
        return None
    metadata = previous['metadata']
    reasons = [f'current {key} differs' for key, value in _baseline_comparable_config(cfg).items()
               if value != metadata['baseline_config'][key]]
    reasons.extend(_previous_model_mismatch_reasons(metadata['previous_model']))
    if reasons:
        raise RuntimeError('Previous Actor baseline identity changed after preparation: ' +
                           '; '.join(reasons) + '. Reinitialize with a new RUN_NAME; '
                           'do not change the active run configuration or model.')
    originals = _verified_previous_cases(previous, case['split'])
    if case != originals.get(case['source_id']):
        raise ValueError('Cannot import baseline for a modified case')
    key = (case['source_id'], condition)
    row = previous['baseline'].get((case['split'], layer), {}).get(key)
    if row is None:
        return None
    expected = metadata['baseline_hashes'][case['split']][layer][condition][case['source_id']]
    if stable_hash(row) != expected:
        raise ValueError('Previous baseline changed after verification')
    row = copy.deepcopy(row)
    row['data_origin'] = 'previous_run_verified_baseline'
    row['origin_run_fingerprint'] = previous['metadata']['previous_run_fingerprint']
    return row


def _evaluate_with_reuse(fresh, layer, backend, cases, head_sets, cfg, conditions):
    if not head_sets:
        raise ValueError('head_sets must not be empty')
    if not globals().get('PREVIOUS_INPUTS') and globals().get('RUN_DIR') is None:
        return fresh(backend, cases, head_sets, cfg, conditions=conditions)
    jobs = [(case, condition, label, heads) for case in cases for condition in conditions
            for label, heads in head_sets.items()]
    # Cached and imported jobs have their own timer; inference ETA excludes them.
    stage = 'read_eval' if layer == 'end_to_end' else 'controlled_read'
    results, cached, pending = [None] * len(jobs), [], []
    for i, (case, condition, label, heads) in enumerate(jobs):
        old = _verified_previous_baseline(case, condition, layer, cfg) if label == 'none' and not heads else None
        if old is not None:
            results[i] = old
            continue
        filename = re.sub(r'[^A-Za-z0-9_.-]', '_', f"{case['source_id']}-{condition}-{label}")[:150] + '.json'
        run_dir = globals().get('RUN_DIR')
        (cached if run_dir is not None and (Path(run_dir) / 'items' / stage / filename).exists()
         else pending).append(i)
    print(f'{layer}: imported={sum(r is not None for r in results)}, cached={len(cached)}, pending={len(pending)}', flush=True)
    for indices, title in ((cached, ' Validate breakpoints '), (pending, ' New calculation ')):
        for i in tqdm(indices, desc=f'{layer} · {title}', unit=' Group ', dynamic_ncols=True,
                      bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [ Used  {elapsed}， Projected surplus  {remaining}]',
                      disable=not cfg.get('show_progress', True)):
            case, condition, label, heads = jobs[i]
            rows = fresh(backend, [case], {label: heads}, cfg, conditions=(condition,))
            if len(rows) != 1:
                raise RuntimeError('Expected exactly one complete result')
            results[i] = rows[0]
    return results


def evaluate_read(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    return _evaluate_with_reuse(_evaluate_read_fresh, 'end_to_end', backend, cases, head_sets, cfg, conditions)


def evaluate_controlled_read(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    return _evaluate_with_reuse(_evaluate_controlled_read_fresh, 'controlled_decision', backend, cases, head_sets, cfg, conditions)


def load_or_collect_histories(backend, cases, conditions, cfg):
    previous = globals().get('PREVIOUS_INPUTS')
    if not cases:
        raise ValueError('No cases')
    splits = {c['split'] for c in cases}
    if len(splits) != 1:
        raise ValueError('Do not mix history splits')
    split = next(iter(splits))
    if previous and split in previous['histories']:
        originals = _verified_previous_cases(previous, split)
        if any(c != originals.get(c['source_id']) for c in cases):
            raise ValueError('Cannot reuse frozen experiences for modified cases')
        result = {}
        for condition in conditions:
            result[condition] = {}
            for case in cases:
                source = case['source_id']
                history = previous['histories'][split][condition][source]
                expected = previous['metadata']['frozen_history_hashes'][split][condition][source]
                if stable_hash(history) != expected:
                    raise ValueError('Previous frozen history changed after verification')
                result[condition][source] = copy.deepcopy(history)
        return result
    return collect_fixed_histories(backend, cases, conditions, cfg)

In [ ]:
# Behavioral feasibility and head selection. Embedded into the notebook globals.
import math


def _behavioral_number(row, metric):
    value = row.get(metric)
    if value is None or not isinstance(value, (bool, int, float)) or not math.isfinite(float(value)):
        raise ValueError(f"Missing/nonfinite behavioral metric {metric}: {row.get('source_id')}")
    if metric.endswith("_count"):
        if value < 0 or int(value) != value:
            raise ValueError(f"Invalid behavioral count {metric}")
    elif value not in (0, 1):
        raise ValueError(f"Behavioral indicator {metric} must be 0 or 1")
    return float(value)


def _behavioral_index(rows, source_ids, conditions, intervention, metrics, layer=None):
    """Fail closed on missing, duplicate or unexpected observations; never filter failures."""
    expected = {(source, condition) for source in source_ids for condition in conditions}
    indexed = {}
    for row in rows:
        key = (row.get("source_id"), row.get("condition"))
        if key not in expected or row.get("intervention") != intervention:
            raise ValueError(f"Unexpected behavioral observation: {key}/{row.get('intervention')}")
        if key in indexed:
            raise ValueError(f"Duplicate behavioral observation: {key}")
        if layer is not None and row.get("evaluation_layer") != layer:
            raise ValueError(f"Wrong evaluation layer: {key}")
        for metric in metrics:
            _behavioral_number(row, metric)
        if "binding_correct" in metrics and "binding_distractor" in metrics:
            if row["binding_correct"] and row["binding_distractor"]:
                raise ValueError(f"Contradictory binding indicators: {key}")
        indexed[key] = row
    if set(indexed) != expected:
        raise ValueError(f"Missing planned behavioral observations: {sorted(expected - set(indexed))}")
    return indexed


def _behavioral_means(indexed, condition, metrics):
    rows = [row for (_, cond), row in indexed.items() if cond == condition]
    if not rows:
        raise ValueError("Cannot summarize an empty behavioral condition")
    return {name: sum(_behavioral_number(row, metric) for row in rows) / len(rows)
            for metric, name in metrics.items()}


def read_phenomenon_gate(rows, cfg):
    """Paired untreated pilot: clean correct actions must become genuine distractor bindings.

    A malformed output or missing order never counts as a natural binding error.
    n_pilot (when supplied) also detects sources absent from both conditions.
    """
    conditions = ("correct_only", "distractor")
    baseline = [r for r in rows if r.get("intervention") == "none" and r.get("condition") in conditions]
    source_ids = sorted({r.get("source_id") for r in baseline if isinstance(r.get("source_id"), str)})
    if not source_ids or ("n_pilot" in cfg and len(source_ids) != cfg["n_pilot"]):
        raise ValueError("Pilot baseline does not contain every planned source")
    metrics = ("decision_action_success", "binding_correct", "binding_distractor", "format_failure")
    indexed = _behavioral_index(baseline, source_ids, conditions, "none", metrics, "controlled_decision")
    clean = [indexed[source, "correct_only"] for source in source_ids]
    attacked = [indexed[source, "distractor"] for source in source_ids]
    success = sum(r["decision_action_success"] for r in clean) / len(clean)
    format_failure = sum(r["format_failure"] for r in clean) / len(clean)
    errors = sum(bool(c["decision_action_success"] and c["binding_correct"] and not c["format_failure"]
                      and a["binding_distractor"] and not a["format_failure"])
                 for c, a in zip(clean, attacked))
    minimum = int(cfg.get("min_natural_read_errors", 2))
    if minimum < 1:
        raise ValueError("min_natural_read_errors must be at least one")
    min_success = cfg.get("min_controlled_success", 0.8)
    max_format_failure = cfg.get("max_pilot_format_failure", 0.2)
    reasons = []
    if success < min_success:
        reasons.append("clean_controlled_capability_below_threshold")
    if format_failure > max_format_failure:
        reasons.append("clean_format_failure_above_threshold")
    if errors < minimum:
        reasons.append("insufficient_paired_natural_binding_errors")
    return {"eligible": not reasons, "reasons": reasons, "n_sources": len(source_ids),
            "natural_read_errors": errors, "paired_binding_regressions": errors,
            "min_errors": minimum, "clean_success": success, "format_failure": format_failure,
            "min_success": min_success, "max_format_failure": max_format_failure,
            "distractor_binding_error": sum(r["binding_distractor"] and not r["format_failure"] for r in attacked) / len(attacked),
            "evaluation_layer": "controlled_decision", "gate_version": "paired_read_phenomenon_v1",
            "interpretation": "feasibility_gate_only_not_a_significance_test"}


def _behavioral_cases(cases, cfg):
    ids = [case.get("source_id") for case in cases]
    if not ids or any(not isinstance(source, str) or not source for source in ids) or len(set(ids)) != len(ids):
        raise ValueError("Behavioral selection requires nonempty unique planned sources")
    if any(case.get("split") != "dev" for case in cases):
        raise ValueError("Behavioral selection accepts development cases only")
    if "n_dev" in cfg and len(cases) != cfg["n_dev"]:
        raise ValueError("Behavioral selection must use all planned development cases")
    return ids


def _behavioral_candidates(backend, ranking, metric_names):
    ranked = ranking["ranked_heads"]
    n_layers, n_heads = backend.model.cfg.n_layers, backend.model.cfg.n_heads
    seen = set()
    for head in ranked:
        if len(head) != 2 or any(type(x) is not int for x in head):
            raise ValueError("Ranking contains invalid head coordinates")
        layer, index = head
        if not (0 <= layer < n_layers and 0 <= index < n_heads) or tuple(head) in seen:
            raise ValueError("Ranking contains duplicate or out-of-bounds heads")
        seen.add(tuple(head))
    candidates = []
    # This schedule is fixed, not chosen after inspecting development outcomes.
    for k in (1, 2, 4, 8, 16):
        heads = [list(h) for h in ranked[:k]]
        counts = {}
        for layer, _ in heads:
            counts[layer] = counts.get(layer, 0) + 1
        possible = len(heads) == k and all(count * 2 <= n_heads for count in counts.values())
        reasons = []
        if len(heads) != k:
            reasons.append("ranking_shorter_than_k")
        elif not possible:
            reasons.append("disjoint_layer_matched_random_control_unavailable")
        candidates.append({"k": k, "heads": heads, "eligible": False, "reasons": reasons,
                           "disjoint_random_control_possible": possible, "evaluated_stages": [],
                           **{name: None for name in metric_names},
                           **{name + "_change": None for name in metric_names}})
    return candidates


def _behavioral_record(candidate, prefix, means, baseline):
    for name, value in means.items():
        candidate[prefix + name] = value
        candidate[prefix + name + "_change"] = value - baseline[name]


def _behavioral_preserve(candidate, prefix, higher, lower):
    """Strict aggregate preservation over the same complete, paired source set."""
    for metric in higher:
        if candidate[prefix + metric + "_change"] < -1e-12:
            candidate["reasons"].append(prefix + metric + "_decreased")
    for metric in lower:
        if candidate[prefix + metric + "_change"] > 1e-12:
            candidate["reasons"].append(prefix + metric + "_increased")


def _behavioral_finish(name, ranking, candidates, baseline, gate, gain_key):
    eligible = [row for row in candidates if row["eligible"]]
    selected = max(eligible, key=lambda row: (row[gain_key], -row["k"])) if eligible else None
    result = {"name": name, "seed": ranking["seed"], "candidates": candidates,
              "selected": selected, "heads": selected["heads"] if selected else [],
              "eligible": selected is not None, "baseline": baseline, "gate": gate,
              "n_sources": candidates[0]["n_sources"],
              "selection_method": "all_source_complete_behavior_v1",
              "reasons": [] if selected else ["no_candidate_passed_behavioral_constraints"],
              "primary_seed_policy": "caller_keeps_predeclared_primary_seed_no_best_of_seeds"}
    atomic_json(RUN_DIR / "search" / f"{name}-seed{ranking['seed']}-behavioral-dev.json", result)
    return result


def select_read_heads_behavioral(backend, ranking, cases, cfg, phenomenon_gate):
    """Evaluate fixed small K on all dev sources; token margins never select heads.

    Clean controlled damage short-circuits expensive downstream evaluations.
    A failed pilot phenomenon gate still permits diagnosis, never a selected arm.
    End-to-end baseline is computed only if some candidate passes clean control.
    """
    source_ids = _behavioral_cases(cases, cfg)
    conditions = ("correct_only", "distractor")
    controlled_metrics = {"decision_action_success": "success", "binding_correct": "binding_correct",
                          "binding_distractor": "binding_distractor", "format_failure": "format_failure"}
    end_metrics = {"task_success": "success", "strict_task_success": "strict_success",
                   "format_failure": "format_failure"}
    names = [f"{condition}_{stage}_{metric}" for condition in ("clean", "distractor")
             for stage, metrics in (("controlled", controlled_metrics), ("end_to_end", end_metrics))
             for metric in metrics.values()]
    candidates = _behavioral_candidates(backend, ranking, names)
    for row in candidates:
        row["n_sources"] = len(source_ids)
    baseline_rows = evaluate_controlled_read(backend, cases, {"none": []}, cfg, conditions=conditions)
    indexed = _behavioral_index(baseline_rows, source_ids, conditions, "none", controlled_metrics, "controlled_decision")
    baseline = {"clean_controlled": _behavioral_means(indexed, "correct_only", controlled_metrics),
                "distractor_controlled": _behavioral_means(indexed, "distractor", controlled_metrics),
                "clean_end_to_end": None, "distractor_end_to_end": None}
    for row in candidates:
        if row["reasons"]:
            continue
        label = f"behavioral_read_seed{ranking['seed']}_k{row['k']}"
        raw = evaluate_controlled_read(backend, cases, {label: row["heads"]}, cfg, conditions=("correct_only",))
        measured = _behavioral_index(raw, source_ids, ("correct_only",), label, controlled_metrics, "controlled_decision")
        _behavioral_record(row, "clean_controlled_", _behavioral_means(measured, "correct_only", controlled_metrics), baseline["clean_controlled"])
        row["evaluated_stages"].append("clean_controlled")
        _behavioral_preserve(row, "clean_controlled_", ("success", "binding_correct"), ("format_failure", "binding_distractor"))
        if row["clean_controlled_success"] < cfg.get("min_controlled_success", 0.8):
            row["reasons"].append("clean_controlled_capability_below_threshold")
        if row["reasons"]:
            row["reasons"].append("later_stages_skipped_after_clean_damage")
            if not phenomenon_gate.get("eligible", False):
                row["reasons"].append("read_phenomenon_gate_failed")
            continue
        if not phenomenon_gate.get("eligible", False):
            row["reasons"].extend(["read_phenomenon_gate_failed", "not_evaluated_no_natural_binding_errors"])
            continue
        raw = evaluate_controlled_read(backend, cases, {label: row["heads"]}, cfg, conditions=("distractor",))
        measured = _behavioral_index(raw, source_ids, ("distractor",), label, controlled_metrics, "controlled_decision")
        _behavioral_record(row, "distractor_controlled_", _behavioral_means(measured, "distractor", controlled_metrics), baseline["distractor_controlled"])
        row["evaluated_stages"].append("distractor_controlled")
        _behavioral_preserve(row, "distractor_controlled_", ("success",), ("format_failure", "binding_distractor"))
        if row["distractor_controlled_binding_correct_change"] <= 1e-12:
            row["reasons"].append("no_distractor_binding_improvement")
        if baseline["clean_end_to_end"] is None:
            raw = evaluate_read(backend, cases, {"none": []}, cfg, conditions=conditions)
            measured = _behavioral_index(raw, source_ids, conditions, "none", end_metrics, "end_to_end")
            for condition, short in zip(conditions, ("clean", "distractor")):
                baseline[short + "_end_to_end"] = _behavioral_means(measured, condition, end_metrics)
        raw = evaluate_read(backend, cases, {label: row["heads"]}, cfg, conditions=conditions)
        measured = _behavioral_index(raw, source_ids, conditions, label, end_metrics, "end_to_end")
        for condition, short in zip(conditions, ("clean", "distractor")):
            prefix = short + "_end_to_end_"
            _behavioral_record(row, prefix, _behavioral_means(measured, condition, end_metrics), baseline[short + "_end_to_end"])
            _behavioral_preserve(row, prefix, ("success", "strict_success"), ("format_failure",))
        row["evaluated_stages"].append("end_to_end")
        if not phenomenon_gate.get("eligible", False):
            row["reasons"].append("read_phenomenon_gate_failed")
        row["eligible"] = not row["reasons"]
    return _behavioral_finish("read", ranking, candidates, baseline, phenomenon_gate,
                              "distractor_controlled_binding_correct_change")


def select_writer_heads_behavioral(backend, ranking, cases, histories, cfg, writer_gate):
    """Future Writer selection on frozen histories, with validity and semantic retention.

    Wrong values, omissions, duplicates and wrong sources cannot be traded for
    target correction. Failed outputs stay in every all-source denominator.
    """
    source_ids = _behavioral_cases(cases, cfg)
    conditions = ("correct_only", "distractor")
    metrics = {key: key for key in ("target_correct", "format_failure", "omission",
                                    "duplicate_count", "source_mismatch_count", "unsupported_count")}
    names = [condition + "_" + metric for condition in ("clean", "distractor") for metric in metrics]
    candidates = _behavioral_candidates(backend, ranking, names)
    for row in candidates:
        row["n_sources"] = len(source_ids)
    raw = evaluate_writer(backend, cases, histories, {"none": []}, cfg, conditions=conditions)
    indexed = _behavioral_index(raw, source_ids, conditions, "none", metrics)
    baseline = {short: _behavioral_means(indexed, condition, metrics)
                for condition, short in zip(conditions, ("clean", "distractor"))}
    lower = ("format_failure", "omission", "duplicate_count", "source_mismatch_count", "unsupported_count")
    for row in candidates:
        if row["reasons"]:
            continue
        label = f"behavioral_write_seed{ranking['seed']}_k{row['k']}"
        raw = evaluate_writer(backend, cases, histories, {label: row["heads"]}, cfg, conditions=("correct_only",))
        measured = _behavioral_index(raw, source_ids, ("correct_only",), label, metrics)
        _behavioral_record(row, "clean_", _behavioral_means(measured, "correct_only", metrics), baseline["clean"])
        row["evaluated_stages"].append("clean_writer")
        _behavioral_preserve(row, "clean_", ("target_correct",), lower)
        if 1 - row["clean_format_failure"] < cfg.get("min_writer_valid", 0.8):
            row["reasons"].append("clean_writer_validity_below_threshold")
        if row["clean_target_correct"] < cfg.get("min_writer_clean_correct", 0.8):
            row["reasons"].append("clean_writer_target_retention_below_threshold")
        if row["reasons"]:
            row["reasons"].append("distractor_writer_skipped_after_clean_damage")
            if not writer_gate.get("eligible", False):
                row["reasons"].append("writer_gate_failed")
            continue
        raw = evaluate_writer(backend, cases, histories, {label: row["heads"]}, cfg, conditions=("distractor",))
        measured = _behavioral_index(raw, source_ids, ("distractor",), label, metrics)
        _behavioral_record(row, "distractor_", _behavioral_means(measured, "distractor", metrics), baseline["distractor"])
        row["evaluated_stages"].append("distractor_writer")
        _behavioral_preserve(row, "distractor_", (), lower)
        if row["distractor_target_correct_change"] <= 1e-12:
            row["reasons"].append("no_distractor_target_correction_improvement")
        if not writer_gate.get("eligible", False):
            row["reasons"].append("writer_gate_failed")
        row["eligible"] = not row["reasons"]
    return _behavioral_finish("write", ranking, candidates, baseline, writer_gate,
                              "distractor_target_correct_change")

In [ ]:
IMPLEMENTATION_NAMES = ['canonical_json', 'stable_hash', 'atomic_json', 'checkpoint_item', 'seed_everything', 'normalize_heads', 'ablation_hooks', 'TLBackend', 'eligible_snapshots', 'snapshot_pairs', 'gumbel_drop', '_save_search', 'train_mask', 'select_heads_dev', 'build_cases', '_preference_record', 'expected_city', 'distractor_city', 'memory_for_condition', '_json_no_duplicates', 'parse_json_output', 'OrderEnvironment', '_task_goal', '_actor_protocol_examples', 'actor_prompt', 'run_actor', 'frozen_history', 'prepare_legal_decision_state', 'actor_snapshot', 'first_order_binding', 'score_actor_layers', 'run_controlled_actor', 'run_downstream_actor', '_latest_preference_records', '_memory_schema_error', '_writer_presented_history', '_writer_protocol_examples', 'writer_prompt', 'score_memory', 'run_writer', 'writer_snapshot', 'build_memory_pair', '_followup_case_and_environment', 'evaluate_factorial', 'local_entry_variants', 'evaluate_local_variants', 'run_online', 'collect_fixed_histories', 'evaluate_writer', 'evaluate_memory_pipeline', '_scalar_float', '_evaluate_read_fresh', 'same_layer_random_heads', '_evaluate_controlled_read_fresh', 'layered_read_effects', 'actor_diagnostics', '_passes_filters', '_get_metric', '_source_bootstrap', 'paired_effect', 'bootstrap_summary', '_implementation_source', 'live_implementation_hash', 'verify_live_run', 'capability_gate', 'natural_writer_gate', 'controlled_capability_gate', '_atomic_csv', 'export_rows', 'export_layered_read', 'factorial_contrasts', 'freeze_protocol', 'verify_protocol', 'runtime_smoke', '_read_hashed_json', '_baseline_comparable_config', '_previous_model_mismatch_reasons', '_validate_previous_data', '_verified_previous_cases', 'verified_previous_rankings', 'prepare_previous_run', '_verified_previous_baseline', '_evaluate_with_reuse', 'evaluate_read', 'evaluate_controlled_read', 'load_or_collect_histories', '_behavioral_number', '_behavioral_index', '_behavioral_means', 'read_phenomenon_gate', '_behavioral_cases', '_behavioral_candidates', '_behavioral_record', '_behavioral_preserve', '_behavioral_finish', 'select_read_heads_behavioral', 'select_writer_heads_behavioral']

# IPython keeps executable cell sources. Hash actual live definitions, not just a version label.
IMPLEMENTATION_HASH = live_implementation_hash()
print("Implementation:", IMPLEMENTATION_HASH[:16])

Implementation: 08baa85181b65bad


###  Read-only import old run and check source 

 Read old configurations, data and sort files; authenticating by article  frozen history  And Hara  Writer  Use  history_hash。 The old results will not be replicated into new experiments.  checkpoint。 Record file hashes, ranking provenance, and experience hashes in the new configuration manifest; explicitly show version differences. 

In [ ]:
PREVIOUS_INPUTS = prepare_previous_run(PREVIOUS_RUN_DIR, RUN_DIR, CFG, PACKAGE_VERSIONS, sys.version.split()[0])
old_revision = PREVIOUS_INPUTS['metadata']['previous_model']['revision']
if MODEL_REVISION is not None and MODEL_REVISION != old_revision:
    raise RuntimeError(' The same model must be maintained when importing the old sorting revision； Please set up a separate model migration experiment. ')
MODEL_REVISION = old_revision
CFG['previous_inputs'] = PREVIOUS_INPUTS['metadata']
print(' Old  Actor baseline  Reusable: ', PREVIOUS_INPUTS['metadata']['reuse_actor_baselines'])
print(' Reasons for not being reused: ', PREVIOUS_INPUTS['metadata']['baseline_reuse_disabled_reasons'])
print(' Sorting found. Reading side will not be retrained.  mask。')

 Old  Actor baseline  Reusable:  True
 Reasons for not being reused:  []
 Sorting found. Reading side will not be retrained.  mask。


## 03 ·  Login, model loading and running list 

 Keep the old one.  `fold_ln=False`、`center_writing_weights=False`、`center_unembed=False`。 Do not automatically quantify and do not replace  Instruct。 To accommodate the full experience, display settings  `n_ctx=4096`（ The old version of the library will only be assigned to this model by default  2048）； Expand only the context buffer when running, with weights equal to  RoPE  parameters are retained. Reject overlong inputs rather than truncating evidence. The next step separately checks controlled decisions and full tool-protocol capability of the base model. This cell reports missing steps if core definitions or hash cells have not run. 

In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [ ]:
if not callable(globals().get("live_implementation_hash")) or "IMPLEMENTATION_NAMES" not in globals():
    raise RuntimeError(" Please run in sequence. 02 Section all defined blocks and their last  IMPLEMENTATION_NAMES/hash  block, rerun model loaded. ")
missing_definitions = [name for name in IMPLEMENTATION_NAMES if name not in globals()]
if missing_definitions:
    raise RuntimeError("02 Section is defined as not running: " + ", ".join(missing_definitions))
IMPLEMENTATION_HASH = live_implementation_hash()
if not get_token():
    notebook_login()
if not torch.cuda.is_available():
    raise RuntimeError(" This.  notebook   8B  Experimental needs.  CUDA GPU； First.  Colab  Change running time. ")
if REQUESTED_DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError(" current  GPU  Not supported  bfloat16。 Replaces running or is specified as  float16  and use new  RUN_NAME； Do not mix the results with precision. ")
DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16}[REQUESTED_DTYPE]
DEVICE = "cuda"
revision_path = RUN_DIR / "model_revision.json"
if revision_path.exists():
    saved_revision = json.loads(revision_path.read_text())
    if saved_revision["model_name"] != MODEL_NAME or (MODEL_REVISION and MODEL_REVISION != saved_revision["sha"]):
        raise RuntimeError("Model revision differs; use a new RUN_NAME.")
    RESOLVED_REVISION = saved_revision["sha"]
else:
    RESOLVED_REVISION = model_info(MODEL_NAME, revision=MODEL_REVISION).sha
    atomic_json(revision_path, {"model_name": MODEL_NAME, "sha": RESOLVED_REVISION})
MODEL_IDENTITY = {"name": MODEL_NAME, "revision": RESOLVED_REVISION, "dtype": REQUESTED_DTYPE,
                  "packages": PACKAGE_VERSIONS, "python": sys.version.split()[0]}
RUN_FINGERPRINT = stable_hash({"config": CFG, "model": MODEL_IDENTITY, "implementation": IMPLEMENTATION_HASH})
manifest = {"fingerprint": RUN_FINGERPRINT, "config": CFG, "model": MODEL_IDENTITY,
            "implementation": IMPLEMENTATION_HASH}
manifest_path = RUN_DIR / "manifest.json"
if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
    raise RuntimeError("Run directory contains another configuration or package stack. Choose a new RUN_NAME.")
atomic_json(manifest_path, manifest)
atomic_json(RUN_DIR / "previous_inputs.json", PREVIOUS_INPUTS["metadata"])
atomic_json(RUN_DIR / "imported_frozen_histories.json", PREVIOUS_INPUTS["histories"])
seed_everything(CFG["data_seed"])
print("GPU:", torch.cuda.get_device_name(0), "GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
model = HookedTransformer.from_pretrained(
    MODEL_NAME, revision=RESOLVED_REVISION, device=DEVICE, dtype=DTYPE, n_ctx=CFG["context_window"],
    fold_ln=False, center_writing_weights=False, center_unembed=False,
)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
N_LAYERS, N_HEADS = model.cfg.n_layers, model.cfg.n_heads
backend = TLBackend(model, CFG)
print("Model:", MODEL_NAME, "revision:", RESOLVED_REVISION, "layers/query-heads:", N_LAYERS, N_HEADS)

GPU: NVIDIA A100-SXM4-80GB GB: 79.3


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Loaded pretrained model meta-llama/Llama-3.1-8B into HookedTransformer
Model: meta-llama/Llama-3.1-8B revision: d04e592bb4f6aa9cfee91e2e20afa771667e1d4b layers/query-heads: 32 32


## 04 ·  Model Interface  smoke  and  token  Audit 

 Here, check it out.  GPU  Go on.  hook、mask  Gradient and  cache； Failed to fix interface first, not enter the thesis experiment. `hook_z`  and  `hook_result`  Low precision may introduce small numerical differences; record the maximum deviation against preset tolerances. Short cached and uncached  greedy  Generating must be consistent. 

In [ ]:
SMOKE = runtime_smoke(backend)
atomic_json(RUN_DIR / "software_smoke.json", SMOKE)
display(SMOKE)

# Initial vocabulary screen is role-specific; exact task prefixes are checked again below.
TOKEN_AUDIT = []
for city in CFG["city_candidates"]:
    actor_id = backend.one_token_id('Next tool call:\n{"city":"', city)
    writer_id = backend.one_token_id('MEMORY_JSON:\n[{"value":"', city)
    TOKEN_AUDIT.append({"city": city, "actor_token": actor_id, "writer_token": writer_id,
                        "eligible": actor_id is not None and writer_id is not None})
CITY_VALUES = [row["city"] for row in TOKEN_AUDIT if row["eligible"]]
if len(CITY_VALUES) < 4:
    raise RuntimeError("Fewer than four single-token cities; inspect TOKEN_AUDIT before proceeding.")
atomic_json(RUN_DIR / "token_audit.json", TOKEN_AUDIT)
display(pd.DataFrame(TOKEN_AUDIT))

{'passed': True,
 'z_result_max_abs_difference': 0.15625,
 'mask_gradient_norm': 8.011000633239746,
 'cache_text_agreement': True,
 'cache_test_skipped': False,
 'cuda_peak_gb': 17.048995971679688}

,city,actor_token,writer_token,eligible
0,Paris,60704.0,60704.0,True
1,Tokyo,NaN,NaN,False
2,Rome,NaN,NaN,False
3,London,40672.0,40672.0,True
4,Berlin,95509.0,95509.0,True
5,Boston,65432.0,65432.0,True
6,Chicago,47275.0,47275.0,True
7,Madrid,NaN,NaN,False
8,Moscow,NaN,NaN,False
9,Seattle,72772.0,72772.0,True


## 05 ·  Create and Seal  source  Data 

 Pregenerated and saved  test  inputs, but do not use their model outputs during development. Diagnostic prefixes represent manually constructed, valid reserved states rather than actual model trajectories. Controlled decisions reuse this valid state without an answer prefix. Autonomous end-to-end tests start from initial stock, and the model generates every action. 

In [ ]:
DATA = {split: build_cases(split, CFG[f"n_{split}"], CFG["data_seed"], CITY_VALUES)
        for split in ("pilot", "train", "dev", "test")}
for left, right in (("train", "dev"), ("train", "test"), ("dev", "test"), ("pilot", "train"), ("pilot", "dev"), ("pilot", "test")):
    for key in ("source_id", "target_name", "other_name", "template_id"):
        assert not ({case[key] for case in DATA[left]} & {case[key] for case in DATA[right]})
DATA_HASHES = {split: stable_hash(cases) for split, cases in DATA.items()}
if DATA_HASHES != PREVIOUS_INPUTS["metadata"]["data_hashes"]:
    raise RuntimeError(" current token selection or data generation differs from the historical run; the old ranking cannot be reused / Experience. ")
atomic_json(RUN_DIR / "data_manifest.json", {"hashes": DATA_HASHES, "data": DATA})
# Development token lengths quantify the approximately matched neutral control.
PROMPT_LENGTHS = [{"source_id": case["source_id"], "condition": condition,
                   "actor_prompt_tokens": len(backend.encode(actor_prompt(case, memory_for_condition(case, condition), [])))}
                  for case in DATA["dev"] for condition in CONDITIONS]
_atomic_csv(RUN_DIR / "results" / "dev-prompt-lengths.csv", PROMPT_LENGTHS)
DESIGN_ROWS = [{"split": split, "template_id": case["template_id"],
                "preferred_available": case["initial_env"]["stock"][case["preferred_item"]] > 0,
                "position_order": case["position_order"]}
               for split, cases in DATA.items() for case in cases]
_atomic_csv(RUN_DIR / "results" / "design-factors.csv", DESIGN_ROWS)
display(pd.DataFrame(DESIGN_ROWS).groupby(["split", "template_id", "preferred_available", "position_order"]).size().rename("n_sources").reset_index())
print("Source counts:", {split:len(cases) for split,cases in DATA.items()})
print("Test outputs have not been evaluated.")
display(pd.DataFrame(PROMPT_LENGTHS).groupby("condition")["actor_prompt_tokens"].agg(["mean", "min", "max"]))

,split,template_id,preferred_available,position_order,n_sources
0,dev,dev_request,False,other_first,5
1,dev,dev_request,False,target_first,5
2,dev,dev_request,True,other_first,5
3,dev,dev_request,True,target_first,5
4,dev,dev_shipment,False,other_first,5
5,dev,dev_shipment,False,target_first,5
6,dev,dev_shipment,True,other_first,5
7,dev,dev_shipment,True,target_first,5
8,pilot,pilot_dispatch,False,other_first,3
9,pilot,pilot_dispatch,False,target_first,3


Source counts: {'pilot': 24, 'train': 120, 'dev': 40, 'test': 120}
Test outputs have not been evaluated.


,mean,min,max
condition,,,
correct_only,757.5,754,761
distractor,798.5,795,802
distractor_value_swap,798.5,795,802
legitimate_update,798.5,795,802
matched_neutral,798.5,795,802
useful_memory,798.5,795,802


## 06 · FULL  Internal inspection: capability, natural error, Writer  Separate. 

 Read / Implementation 24  pilot  Completeness and Control  Actor； Old compatible  baseline  Reuse directly. Add  READ_PHENOMENON_GATE： Homogeneity  clean  Correct. distractor  Theres a real client. The citys at least bound it wrongly. 2 cases; format failures do not count as this error. This is a feasibility threshold, not a significance test. 

Writer  Regenerated using a new version of the reminder on the same freezing experience. strict schema  Keeping a score with the facts. Old records still.  A  In the Audit of Sections. No new entries are made without natural errors  heads  Training. 

In [ ]:
verify_live_run()
PILOT_READ = evaluate_read(backend, DATA["pilot"], {"none": []}, CFG,
                           conditions=("correct_only", "distractor"))
PILOT_DECISION = evaluate_controlled_read(backend, DATA["pilot"], {"none": []}, CFG,
                                         conditions=("correct_only", "distractor"))
END_TO_END_CAPABILITY = capability_gate(PILOT_READ, CFG)
CAPABILITY = controlled_capability_gate(PILOT_DECISION, CFG)
READ_PHENOMENON_GATE = read_phenomenon_gate(PILOT_DECISION, CFG)
atomic_json(RUN_DIR / "pilot_read_phenomenon_gate.json", READ_PHENOMENON_GATE)
display(READ_PHENOMENON_GATE)
atomic_json(RUN_DIR / "pilot_capability.json", {"controlled_decision": CAPABILITY,
                                               "end_to_end": END_TO_END_CAPABILITY})
display(pd.DataFrame(export_layered_read("pilot-read", PILOT_READ, PILOT_DECISION)))
display({"controlled_decision": CAPABILITY, "end_to_end": END_TO_END_CAPABILITY})
PILOT_DIAGNOSTICS = actor_diagnostics(PILOT_READ)
_atomic_csv(RUN_DIR / "results" / "pilot-actor-diagnostics.csv", PILOT_DIAGNOSTICS)
atomic_json(RUN_DIR / "results" / "pilot-failures.json", [r for r in PILOT_READ if not r["task_success"]])
display(pd.DataFrame(PILOT_DIAGNOSTICS))
PILOT_HISTORIES = load_or_collect_histories(backend, DATA["pilot"], ("correct_only", "distractor"), CFG)
PILOT_WRITER = evaluate_writer(backend, DATA["pilot"], PILOT_HISTORIES, {"none": []}, CFG,
                               conditions=("correct_only", "distractor"))
WRITER_GATE = natural_writer_gate(PILOT_WRITER, CFG)
display(pd.DataFrame(export_rows("pilot-writer", PILOT_WRITER, ["condition", "intervention"],
                                 ["target_distractor_error", "target_correct", "omission", "format_failure"])))
display(WRITER_GATE)
if not CAPABILITY["eligible"]:
    print(" Controlled complete action capability not reached threshold: Skipped  H_R  Search, keep  FULL baseline  and independence  Writer  Branch. ")
if not END_TO_END_CAPABILITY["eligible"]:
    print(" Lack of self-executing capacity: step-checking / Typing diagnostics; controlled tests cannot substitute for end-to-end evidence. ")
atomic_json(RUN_DIR / "pilot_writer_gate.json", WRITER_GATE)

end_to_end: imported=48, cached=0, pending=0


end_to_end ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

end_to_end ·  New calculation :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision: imported=48, cached=0, pending=0


controlled_decision ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision ·  New calculation :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

... (3 outputs left out)

,source_id,condition,intervention,template_id,preferred_available,position_order,task_success,format_failure,terminal,action_count,...,gold_item,binding_category,binding_correct,binding_distractor,order_attempted,subgoal_progress,prerequisite_path_valid,strict_task_success,first_tool_error_step,first_tool_error
0,pilot-1729-00000,correct_only,none,pilot_order,True,target_first,True,False,order_created,3,...,item-00-primary,correct_binding,True,False,True,1.00,True,True,NaN,None
1,pilot-1729-00000,distractor,none,pilot_order,True,target_first,True,False,order_created,3,...,item-00-primary,correct_binding,True,False,True,1.00,True,True,NaN,None
2,pilot-1729-00001,correct_only,none,pilot_order,False,target_first,False,False,action_budget,6,...,item-01-backup,correct_binding,True,False,True,0.50,False,False,1.0,out_of_stock
3,pilot-1729-00001,distractor,none,pilot_order,False,target_first,True,False,order_created,3,...,item-01-backup,correct_binding,True,False,True,0.75,False,False,NaN,None
4,pilot-1729-00002,correct_only,none,pilot_dispatch,True,target_first,True,False,order_created,3,...,item-02-primary,correct_binding,True,False,True,1.00,True,True,NaN,None
5,pilot-1729-00002,distractor,none,pilot_dispatch,True,target_first,True,False,order_created,3,...,item-02-primary,correct_binding,True,False,True,1.00,True,True,NaN,None
6,pilot-1729-00003,correct_only,none,pilot_dispatch,False,target_first,True,False,order_created,3,...,item-03-backup,correct_binding,True,False,True,0.75,False,False,NaN,None
7,pilot-1729-00003,distractor,none,pilot_dispatch,False,target_first,True,False,order_created,3,...,item-03-backup,correct_binding,True,False,True,0.75,False,False,NaN,None
8,pilot-1729-00004,correct_only,none,pilot_order,True,other_first,True,False,order_created,3,...,item-04-primary,correct_binding,True,False,True,1.00,True,True,NaN,None
9,pilot-1729-00004,distractor,none,pilot_order,True,other_first,True,False,order_created,3,...,item-04-primary,correct_binding,True,False,True,1.00,True,True,NaN,None


,condition,intervention,metric,n_rows,n_missing_rows,n_valid_rows,n_all_sources,estimate,ci_low,ci_high,n_sources,n_boot,ci_method,small_sample_warning
0,correct_only,none,target_distractor_error,24,0,24,24,0.000000,0.000000,0.000000,24,2000,percentile_source_bootstrap_95pct,False
1,correct_only,none,target_correct,24,0,24,24,1.000000,1.000000,1.000000,24,2000,percentile_source_bootstrap_95pct,False
2,correct_only,none,omission,24,0,24,24,0.000000,0.000000,0.000000,24,2000,percentile_source_bootstrap_95pct,False
3,correct_only,none,format_failure,24,0,24,24,0.000000,0.000000,0.000000,24,2000,percentile_source_bootstrap_95pct,False
4,distractor,none,target_distractor_error,24,0,24,24,0.000000,0.000000,0.000000,24,2000,percentile_source_bootstrap_95pct,False
5,distractor,none,target_correct,24,0,24,24,0.541667,0.333333,0.750000,24,2000,percentile_source_bootstrap_95pct,False
6,distractor,none,omission,24,0,24,24,0.500000,0.291667,0.708333,24,2000,percentile_source_bootstrap_95pct,False
7,distractor,none,format_failure,24,0,24,24,0.000000,0.000000,0.000000,24,2000,percentile_source_bootstrap_95pct,False


{'eligible': False,
 'natural_distractor_errors': 0,
 'n_sources': 24,
 'clean_valid_fraction': 1.0,
 'clean_target_correct': 1.0,
 'min_errors': 2,
 'interpretation': 'feasibility_gate_only_not_a_significance_test'}

 Lack of self-executing capacity: step-checking / Typing diagnostics; controlled tests cannot substitute for end-to-end evidence. 


## 07 ·  Reuse old sorting: small  K  Behavioural diagnosis and selection 

** No re-training.  H_R。**  Read Old  read-seed11/23/37.json， Keep Fixed Master seed=11。 each K Check all of them first. 40 dev  clean  controlled actions; stop that candidate on failure. If natural misbinding exists, inspect distractor conditions and full trajectories. Otherwise output only clean Diagnosis of injury, no choice.  H_R。

 Selection requires complete action / preserve binding, clean and distractor end-to-end performance, and format validity, while improving distractor-condition binding. Break ties in favor of the smaller K。margin is excluded from selection. If no candidate qualifies, return an empty list rather than an error. 

In [ ]:
verify_live_run()
READ_SEARCHES, READ_SELECTIONS = [], []
READ_SELECTION = None
H_R, READ_CONTROLS, DEV_READ = [], {}, []
if RUN_DEVELOPMENT and CAPABILITY["eligible"]:
    READ_SEARCHES = verified_previous_rankings(PREVIOUS_INPUTS)
    for ranking in READ_SEARCHES:
        print(f" Staple Old Sorting  seed={ranking['seed']}； Start Small K Complete action diagnosis. ", flush=True)
        selection = select_read_heads_behavioral(backend, ranking, DATA["dev"], CFG, READ_PHENOMENON_GATE)
        READ_SELECTIONS.append(selection)
        display(pd.DataFrame(selection["candidates"]).drop(columns=["heads"]))
    READ_SELECTION = READ_SELECTIONS[0]
    H_R = READ_SELECTION["heads"]
    if H_R:
        READ_CONTROLS = {f"random_{seed}": same_layer_random_heads(H_R,N_LAYERS,N_HEADS,seed)
                         for seed in CFG["random_seeds"]}
    print("Primary read heads:", H_R)
    if not H_R:
        print(" No valid election H_R。 The diagnosis is retained and the empty intervention is not packaged into repair. ")
else:
    print(" Controlled base failed, skipping small K Intervention; first 08 Sections remain baseline。")

 Staple Old Sorting  seed=11； Start Small K Complete action diagnosis. 
controlled_decision: imported=80, cached=0, pending=0


controlled_decision ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision ·  New calculation :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision: imported=0, cached=0, pending=40


controlled_decision ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision ·  New calculation :   0%|          | 0/40 [ Used  00:00， Projected surplus  ?]

... (48 outputs left out)

controlled_decision ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision ·  New calculation :   0%|          | 0/40 [ Used  00:00， Projected surplus  ?]

,k,eligible,reasons,disjoint_random_control_possible,evaluated_stages,clean_controlled_success,clean_controlled_binding_correct,clean_controlled_binding_distractor,clean_controlled_format_failure,clean_end_to_end_success,...,clean_end_to_end_strict_success_change,clean_end_to_end_format_failure_change,distractor_controlled_success_change,distractor_controlled_binding_correct_change,distractor_controlled_binding_distractor_change,distractor_controlled_format_failure_change,distractor_end_to_end_success_change,distractor_end_to_end_strict_success_change,distractor_end_to_end_format_failure_change,n_sources
0,1,False,"[read_phenomenon_gate_failed, not_evaluated_no...",True,[clean_controlled],1.0,1.0,0.0,0.0,None,...,None,None,None,None,None,None,None,None,None,40
1,2,False,"[read_phenomenon_gate_failed, not_evaluated_no...",True,[clean_controlled],1.0,1.0,0.0,0.0,None,...,None,None,None,None,None,None,None,None,None,40
2,4,False,"[read_phenomenon_gate_failed, not_evaluated_no...",True,[clean_controlled],1.0,1.0,0.0,0.0,None,...,None,None,None,None,None,None,None,None,None,40
3,8,False,"[read_phenomenon_gate_failed, not_evaluated_no...",True,[clean_controlled],1.0,1.0,0.0,0.0,None,...,None,None,None,None,None,None,None,None,None,40
4,16,False,"[read_phenomenon_gate_failed, not_evaluated_no...",True,[clean_controlled],1.0,1.0,0.0,0.0,None,...,None,None,None,None,None,None,None,None,None,40


Primary read heads: []
 No valid election H_R。 The diagnosis is retained and the empty intervention is not packaged into repair. 


## 08 · FULL Actor  Development evaluation and breakpoint progress 

 All 40 source、 Six pairing conditions. baseline Directly reused and marked data_origin； Its only through the first. 07 The number of candidates for the festival has increased. H_R and random controls. Time cache reads separately from new inference and show elapsed and estimated remaining time. There is no H_R , and then the section is exported to complete baseline。

In [ ]:
DEV_READ_EFFECTS, DEV_DECISION = [], []
if RUN_DEVELOPMENT:
    read_head_sets = {"none": []}
    if H_R:
        read_head_sets.update({"H_R": H_R, **READ_CONTROLS})
    for selection in READ_SELECTIONS[1:]:
        if selection["eligible"]:
            read_head_sets[f"H_R_seed{selection['seed']}"] = selection["heads"]
    DEV_READ = evaluate_read(backend, DATA["dev"], read_head_sets, CFG)
    DEV_DECISION = evaluate_controlled_read(backend, DATA["dev"], read_head_sets, CFG)
    display(pd.DataFrame(export_layered_read("dev-read", DEV_READ, DEV_DECISION)))
    if H_R:
        DEV_READ_EFFECTS = layered_read_effects(DEV_READ, DEV_DECISION, n_boot=CFG["bootstrap_reps"])
        atomic_json(RUN_DIR / "results" / "dev-read-paired-effects.json", DEV_READ_EFFECTS)
        display(pd.DataFrame(DEV_READ_EFFECTS))

end_to_end: imported=240, cached=0, pending=0


end_to_end ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

end_to_end ·  New calculation :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision: imported=240, cached=0, pending=0


controlled_decision ·  Validate breakpoints :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

controlled_decision ·  New calculation :   0%|          | 0/0 [ Used  00:00， Projected surplus  ?]

,evaluation_layer,condition,intervention,metric,n_rows,n_missing_rows,n_valid_rows,n_all_sources,estimate,ci_low,ci_high,n_sources,n_boot,ci_method,small_sample_warning
0,end_to_end,correct_only,none,task_success,40,0,40,40,1.0,1.00,1.00,40,2000,percentile_source_bootstrap_95pct,False
1,end_to_end,correct_only,none,strict_task_success,40,0,40,40,0.5,0.35,0.65,40,2000,percentile_source_bootstrap_95pct,False
2,end_to_end,correct_only,none,binding_correct,40,0,40,40,1.0,1.00,1.00,40,2000,percentile_source_bootstrap_95pct,False
3,end_to_end,correct_only,none,binding_distractor,40,0,40,40,0.0,0.00,0.00,40,2000,percentile_source_bootstrap_95pct,False
4,end_to_end,correct_only,none,order_attempted,40,0,40,40,1.0,1.00,1.00,40,2000,percentile_source_bootstrap_95pct,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,controlled_decision,useful_memory,none,binding_correct,40,0,40,40,1.0,1.00,1.00,40,2000,percentile_source_bootstrap_95pct,False
86,controlled_decision,useful_memory,none,binding_distractor,40,0,40,40,0.0,0.00,0.00,40,2000,percentile_source_bootstrap_95pct,False
87,controlled_decision,useful_memory,none,order_attempted,40,0,40,40,1.0,1.00,1.00,40,2000,percentile_source_bootstrap_95pct,False
88,controlled_decision,useful_memory,none,tool_execution_valid,40,0,40,40,1.0,1.00,1.00,40,2000,percentile_source_bootstrap_95pct,False


## 09 ·  Freezing experience and profile candidates 

dev Retired experience hash verified historical records; train New only Writer gate New candidates are generated when adopted and training is required. H_W Still trained. mask Search to generate a candidate, but develop option to complete Writer Behavioural indicators generated, no longer used margin Selection K。clean Quality needs to be maintained, target correctness improved, missing / Repeat / The source error cannot be increased. If the candidate is not qualified, the side intervention is stopped. 

In [ ]:
WRITE_SEARCHES, WRITE_SELECTIONS = [], []
WRITE_SELECTION = None
H_W, WRITE_CONTROLS, DEV_WRITER = [], {}, []
TRAIN_HISTORIES, DEV_HISTORIES = {}, {}
if RUN_DEVELOPMENT and WRITER_GATE and WRITER_GATE["eligible"]:
    TRAIN_HISTORIES = load_or_collect_histories(backend, DATA["train"], ("distractor", "correct_only"), CFG)
    DEV_HISTORIES = load_or_collect_histories(backend, DATA["dev"], CONDITIONS, CFG)
    train_attack, train_clean, train_excluded = snapshot_pairs(backend, DATA["train"], "writer", TRAIN_HISTORIES)
    dev_attack, dev_clean, dev_excluded = snapshot_pairs(backend, DATA["dev"], "writer", DEV_HISTORIES)
    atomic_json(RUN_DIR / "search" / "write-token-exclusions.json", {"train":train_excluded,"dev":dev_excluded})
    for search_seed in CFG["search_seeds"]:
        ranking = train_mask(backend, train_attack, train_clean, CFG, "write", search_seed)
        selection = select_writer_heads_behavioral(backend, ranking, DATA["dev"], DEV_HISTORIES, CFG, WRITER_GATE)
        WRITE_SEARCHES.append(ranking); WRITE_SELECTIONS.append(selection)
    WRITE_SELECTION = WRITE_SELECTIONS[0]
    H_W = WRITE_SELECTION["heads"]
    if H_W:
        WRITE_CONTROLS = {f"random_{seed}": same_layer_random_heads(H_W,N_LAYERS,N_HEADS,seed)
                          for seed in CFG["random_seeds"]}
    display(pd.DataFrame(WRITE_SELECTION["candidates"]).drop(columns=["heads"]))
    print("Primary write heads:", H_W)
else:
    print(" Side search not enabled in this round: check Writer Independence clean/ Natural error gate and RUN_DEVELOPMENT。 The reading side results are still valid. ")

 Side search not enabled in this round: check Writer Independence clean/ Natural error gate and RUN_DEVELOPMENT。 The reading side results are still valid. 


## 10 ·  complete  Writer  Evaluation 

 Targets incorrectly tied, correctly retained, omitted, duplicated, source mismatched and  JSON  Failures are reported separately; “delete the facts of the target” cannot be considered as a correcting error. H_R  in  Writer  Go, go, go! H_W  in  Actor  The cross-application is just a role-specific diagnosis. Its not supposed to be.  shared。

In [ ]:
if RUN_DEVELOPMENT and not DEV_HISTORIES:
    DEV_HISTORIES = load_or_collect_histories(backend, DATA["dev"], CONDITIONS, CFG)
if DEV_HISTORIES:
    write_head_sets = {"none": []}
    if H_W:
        write_head_sets.update({"H_W": H_W, **WRITE_CONTROLS})
    if H_R:
        write_head_sets["H_R_cross_role"] = H_R
    for selection in WRITE_SELECTIONS[1:]:
        if selection["eligible"]:
            write_head_sets[f"H_W_seed{selection['seed']}"] = selection["heads"]
    DEV_WRITER = evaluate_writer(backend, DATA["dev"], DEV_HISTORIES, write_head_sets, CFG)
    display(pd.DataFrame(export_rows("dev-writer", DEV_WRITER, ["condition","intervention"],
                                     ["target_distractor_error", "target_correct", "omission", "duplicate_count", "source_mismatch_count", "format_failure"])))
    if H_W:
        writer_effects = [dict(condition=condition, **paired_effect(DEV_WRITER, "intervention", "none", "H_W",
                           metric="target_correct", filters={"condition":condition}, n_boot=CFG["bootstrap_reps"])) for condition in CONDITIONS]
        atomic_json(RUN_DIR / "results" / "dev-writer-paired-effects.json", writer_effects)
        CROSS_READ = evaluate_read(backend, DATA["dev"], {"H_W_cross_role":H_W}, CFG, conditions=("correct_only","distractor"))
        CROSS_DECISION = evaluate_controlled_read(backend, DATA["dev"], {"H_W_cross_role":H_W}, CFG, conditions=("correct_only","distractor"))
        export_layered_read("dev-cross-read", CROSS_READ, CROSS_DECISION)

,condition,intervention,metric,n_rows,n_missing_rows,n_valid_rows,n_all_sources,estimate,ci_low,ci_high,n_sources,n_boot,ci_method,small_sample_warning
0,correct_only,none,target_distractor_error,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False
1,correct_only,none,target_correct,40,0,40,40,1.000,1.000,1.000,40,2000,percentile_source_bootstrap_95pct,False
2,correct_only,none,omission,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False
3,correct_only,none,duplicate_count,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False
4,correct_only,none,source_mismatch_count,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False
5,correct_only,none,format_failure,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False
6,distractor,none,target_distractor_error,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False
7,distractor,none,target_correct,40,0,40,40,0.325,0.175,0.475,40,2000,percentile_source_bootstrap_95pct,False
8,distractor,none,omission,40,0,40,40,0.675,0.525,0.825,40,2000,percentile_source_bootstrap_95pct,False
9,distractor,none,duplicate_count,40,0,40,40,0.000,0.000,0.000,40,2000,percentile_source_bootstrap_95pct,False


## 11 ·  Naturally.  memory   2×2  Follow-up tasks 

 Read only. / Write two sets.  heads  Completely run by development selection  2×2， Avoid drawing empty intervention into an independent processing group. Main analysis uses all  source  Nature.  M0/M1， Includes failure of format; no case of  error fixed screened. 

Y00 =  Original  Writer /  Original  Actor；Y10 = H_W /  Original  Actor；Y01 =  Original  Writer / H_R；Y11 = H_W / H_R。 Overall  memory  propagation differences cannot alone identify the mechanism of a field. Run autonomous tasks and controlled full actions separately, generating independent  factorial  Tables and pairs cannot be mixed into a denominator. 

In [ ]:
DEV_PIPELINE = None
if H_R and H_W and DEV_HISTORIES:
    DEV_PIPELINE = evaluate_memory_pipeline(backend, DATA["dev"], DEV_HISTORIES, H_R, H_W, CFG)
    atomic_json(RUN_DIR / "results" / "dev-memory-pipeline.json", DEV_PIPELINE)
    display(pd.DataFrame(export_rows("dev-factorial", DEV_PIPELINE["factorial"], ["arm"],
                                     ["task_success","distractor_action","format_failure"])))
    display(pd.DataFrame(export_rows("dev-factorial-controlled", DEV_PIPELINE["controlled_factorial"], ["arm"],
                                     ["task_success", "binding_correct", "binding_distractor", "format_failure"])))
    atomic_json(RUN_DIR / "results" / "dev-factorial-controlled-contrasts.json",
                factorial_contrasts(DEV_PIPELINE["controlled_factorial"]))
    effects = factorial_contrasts(DEV_PIPELINE["factorial"])
    atomic_json(RUN_DIR / "results" / "dev-factorial-contrasts.json", effects)
    display(pd.DataFrame(effects))
else:
    print(" complete 2×2 Not running: Non-empty required H_R、H_W The study is based on the results already observed. ")

 complete 2×2 Not running: Non-empty required H_R、H_W The study is based on the results already observed. 


## 12 ·  Local  value  Replacement diagnostics 

 This section only runs autonomous end-to-end diagnostics.  M0  The natural target was wrongly tied. M1  Correctly and with a single target entry  source； Number of applications clearly demonstrated / total. Partially replaced is a synonym field, not an entire one.  memory  String just happens to change only one  tokenizer token； The main experiment.  single-token  The statement is only for certified prefixes. 

In [ ]:
if DEV_PIPELINE:
    local = DEV_PIPELINE["local"]
    eligibility = [{"source_id":r["source_id"],"eligible":r["eligible"],"reason":r["reason"]} for r in local]
    local_rows = [row for result in local for row in result["rows"]]
    _atomic_csv(RUN_DIR / "results" / "dev-local-eligibility.csv", eligibility)
    print("Local eligible:", sum(r["eligible"] for r in local), "/", len(local))
    if local_rows:
        display(pd.DataFrame(export_rows("dev-local-swaps",local_rows,["variant"],["task_success"])))
        local_effects = [paired_effect(local_rows,"variant",a,b,n_boot=CFG["bootstrap_reps"])
                         for a,b in (("M0","M0_corrected"),("M1","M1_corrupted"))]
        atomic_json(RUN_DIR / "results" / "dev-local-effects.json", local_effects)
        display(pd.DataFrame(local_effects))

## 13 ·  Optional online update 

 Thats one step that goes on during the follow-up mission.  memory。Writer  Received Only Existing  memory  ♪ And real new moves ♪ / Feedback does not re-infuse clean truth values; true values are used only for rating. horizon 1/3/5  Its the same.  source  The observations in question cannot be merged into a stand-alone sample. 

In [ ]:
ONLINE_RESULTS = []
if RUN_ONLINE and DEV_PIPELINE:
    for case in DATA["dev"]:
        pair = DEV_PIPELINE["pairs"][case["source_id"]]
        for arm, memory_label, read_heads, write_heads in (("none","M0",[],[]),("both","M1",H_R,H_W)):
            for horizon in CFG["online_horizons"]:
                payload = {"case":case,"memory":pair[memory_label]["memory"],"arm":arm,
                           "read_heads":read_heads,"write_heads":write_heads,"horizon":horizon,"cfg":CFG}
                result = checkpoint_item("online", f"{case['source_id']}-{arm}-{horizon}",payload,
                    lambda: run_online(backend,case,pair[memory_label]["memory"],read_heads,write_heads,horizon,CFG))
                ONLINE_RESULTS.append({"source_id":case["source_id"],"arm":arm,"horizon":horizon,**result})
    atomic_json(RUN_DIR / "results" / "dev-online-exploratory.json", ONLINE_RESULTS)
    print("Online trajectories saved:",len(ONLINE_RESULTS))

## 14 ·  Freezing of protocol and then one-off assessment  test

 Default close. Completed.  FULL  After development, execute separately  `FREEZE_AND_RUN_TEST = True`， And run down below.  test cell； Not here.  QUICK  Upstart formal  test。 No need to rerun configuration  cell  Or a search ahead. 

 Agreement fixed model  revision、 Dependence, core definition / Mission Constant  hash、 Data disaggregation, master  seed、heads、 The main endpoint is random. The new main endpoint is all under controlled conditions. source binding accuracy; executable full-action success and autonomous end-to-end success are reported separately. The primary propagation endpoint is the controlled full action Y10−Y00， Report autonomous propagation separately. Changing the endpoint is a development revision and cannot be described retrospectively as preregistered. Intervals for multiple endpoints are descriptive; do not claim uncorrected multiple significance. test  Recreate your own experiences and  M0/M1， No, I do not.  dev memory。 View  test  Were not going to change it.  K、prompt  or sifting samples; reusable  checkpoint  Complete the same test that was interrupted. 

In [ ]:
TEST_READ, TEST_DECISION, TEST_WRITER, TEST_PIPELINE = [], [], [], None
if FREEZE_AND_RUN_TEST:
    if READ_SELECTION is None and WRITE_SELECTION is None:
        raise RuntimeError("Finish the independent capability checks and development stages first.")
    if not H_R and not H_W:
        raise RuntimeError("No eligible intervention selected; report the feasibility result instead of creating a positive test arm.")
    FROZEN = freeze_protocol(READ_SELECTION,WRITE_SELECTION,READ_CONTROLS,WRITE_CONTROLS,DATA_HASHES,WRITER_GATE)
    verify_protocol(FROZEN, DATA)
    test_read_sets = {"none":[]}
    if FROZEN["read_heads"]:
        test_read_sets.update({"H_R":FROZEN["read_heads"],**FROZEN["read_controls"]})
    TEST_READ = evaluate_read(backend,DATA["test"],test_read_sets,CFG)
    TEST_DECISION = evaluate_controlled_read(backend,DATA["test"],test_read_sets,CFG)
    display(pd.DataFrame(export_layered_read("test-read", TEST_READ, TEST_DECISION)))
    if FROZEN["read_heads"]:
        test_effects = layered_read_effects(TEST_READ, TEST_DECISION, n_boot=CFG["bootstrap_reps"])
        atomic_json(RUN_DIR / "results" / "test-read-effects.json",test_effects)
    if FROZEN["writer_branch"]:
        TEST_HISTORIES = collect_fixed_histories(backend,DATA["test"],CONDITIONS,CFG)
        TEST_WRITER = evaluate_writer(backend,DATA["test"],TEST_HISTORIES,
                                      {"none":[],"H_W":FROZEN["write_heads"],**FROZEN["write_controls"]},CFG)
        export_rows("test-writer",TEST_WRITER,["condition","intervention"],
                    ["target_distractor_error","target_correct","omission","format_failure"])
        test_writer_effects = [dict(condition=c,**paired_effect(TEST_WRITER,"intervention","none","H_W",
                                   metric="target_correct",filters={"condition":c},n_boot=CFG["bootstrap_reps"])) for c in CONDITIONS]
        atomic_json(RUN_DIR / "results" / "test-writer-effects.json",test_writer_effects)
        if FROZEN["read_heads"]:
            TEST_PIPELINE = evaluate_memory_pipeline(backend,DATA["test"],TEST_HISTORIES,FROZEN["read_heads"],FROZEN["write_heads"],CFG)
            atomic_json(RUN_DIR / "results" / "test-memory-pipeline.json",TEST_PIPELINE)
            export_rows("test-factorial",TEST_PIPELINE["factorial"],["arm"],["task_success","format_failure"])
            atomic_json(RUN_DIR / "results" / "test-factorial-effects.json",factorial_contrasts(TEST_PIPELINE["factorial"]))
            export_rows("test-factorial-controlled",TEST_PIPELINE["controlled_factorial"],["arm"],
                        ["task_success", "binding_correct", "binding_distractor", "format_failure"])
            atomic_json(RUN_DIR / "results" / "test-factorial-controlled-effects.json",
                        factorial_contrasts(TEST_PIPELINE["controlled_factorial"]))
    atomic_json(RUN_DIR / "test_complete.json",{"protocol_hash":stable_hash(FROZEN),"test_sources":len(DATA["test"]),"completed":True})
    print("Frozen test complete. Results:",RUN_DIR / "results")
else:
    print(" confirmation test closing; current round output only pilot/dev。")

 confirmation test closing; current round output only pilot/dev。


## 15 ·  Save and read results 

- `manifest.json` / `model_revision.json` / `data_manifest.json`： This configuration, real model version and  source  Division. 
- `items/`： Case by case  raw generation、 Tool feedback, memory  and the cause of failure; resumes at the original configuration when the line is broken. 
- `search/`：mask logits、 Full Optimizer /RNG checkpoint、 each  seed  and  dev  Choose. 
- `results/*-summary.csv`： Press  source  aggregated tables; `*-effects.json`： Matching differences and compartments. 
- `frozen_protocol.json` / `test_complete.json`： Only officially.  test  Arising from implementation. 

 Read the controlled and the autonomous.  clean  Capabilities and Formatting, Read Again  distractor  Tie, execute, final act and randomly  heads  contrasts;rejudgment  Writer  Is there a natural fact that was wrongly tied?  memory  whether downstream tasks are affected. Fewer errors accompanied by more omissions or impaired authorized updates are not a cost-free repair. 

 Keep the master model as the old version  base 8B。 If it does not fit into the complete tool, /Writer  Agreement, where available, to locate this limitation;  Instruct  or the change hint should be run independently and cannot be mixed with the current comparison. 

 The interface is based on: [TransformerLens 2.17  Model Interface ](https://github.com/TransformerLensOrg/TransformerLens/blob/v2.17.0/transformer_lens/HookedTransformer.py)、[KV cache](https://github.com/TransformerLensOrg/TransformerLens/blob/v2.17.0/transformer_lens/past_key_value_caching.py)、[ Declaration of reliance ](https://github.com/TransformerLensOrg/TransformerLens/blob/v2.17.0/pyproject.toml)。

In [ ]:
RUN_STATUS = {"quick_mode":QUICK_MODE,"capability":CAPABILITY,"writer_gate":WRITER_GATE,
              "evaluation_version": CFG["evaluation_version"], "end_to_end_capability": END_TO_END_CAPABILITY,
              "read_phenomenon_gate": READ_PHENOMENON_GATE,
              "previous_inputs": PREVIOUS_INPUTS["metadata"],
              "read_mask_retrained": False,
              "dev_controlled_rows": len(DEV_DECISION),
              "read_heads":H_R,"write_heads":H_W,"dev_read_rows":len(DEV_READ),
              "dev_writer_rows":len(DEV_WRITER),"dev_factorial_rows":len(DEV_PIPELINE["factorial"]) if DEV_PIPELINE else 0,
              "test_executed":bool(TEST_READ),"output_dir":str(RUN_DIR)}
atomic_json(RUN_DIR / "run_status.json",RUN_STATUS)
display(RUN_STATUS)
print(" Give priority to sharing when continuing analysis  run_status.json、summary.csv  and related  raw  Cases of failure. ")

{'quick_mode': False,
 'capability': {'eligible': True,
  'n_sources': 24,
  'clean_success': 1.0,
  'format_failure': 0.0,
  'min_success': 0.8,
  'max_format_failure': 0.2,
  'evaluation_layer': 'controlled_decision',
  'metric': 'decision_action_success',
  'interpretation': 'project_feasibility_threshold_not_benchmark_standard'},
 'writer_gate': {'eligible': False,
  'natural_distractor_errors': 0,
  'n_sources': 24,
  'clean_valid_fraction': 1.0,
  'clean_target_correct': 1.0,
  'min_errors': 2,
  'interpretation': 'feasibility_gate_only_not_a_significance_test'},
 'evaluation_version': 'behavioral_v2_full',
 'end_to_end_capability': {'eligible': False,
  'n_sources': 24,
  'clean_success': 0.75,
  'format_failure': 0.0,
  'min_success': 0.8,
  'max_format_failure': 0.2},
 'read_phenomenon_gate': {'eligible': False,
  'reasons': ['insufficient_paired_natural_binding_errors'],
  'n_sources': 24,
  'natural_read_errors': 0,
  'paired_binding_regressions': 0,
  'min_errors': 2,
  'cl

 Give priority to sharing when continuing analysis  run_status.json、summary.csv  and related  raw  Cases of failure. 
